In [ ]:
# ============================================================
# NOTEBOOK 02 — ROOT CONTRACT + REFERENCE PREFLIGHT
# ============================================================

import importlib.util
import os
import platform
import sys
import tempfile
from pathlib import Path

import torch


def module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (AttributeError, ImportError, ModuleNotFoundError, ValueError):
        return False


def absolute_root(value, setting_name):
    root = Path(value)

    if not root.is_absolute():
        raise ValueError(
            f"{setting_name} deve indicare un path assoluto: {root}"
        )

    return root


def existing_disk_probe(path):
    probe = path

    while not probe.exists():
        parent = probe.parent

        if parent == probe:
            raise RuntimeError(
                f"Nessun percorso esistente per inventario disco: {path}"
            )

        probe = parent

    return probe


IN_COLAB = module_available("google.colab")
PROJECT_ROOT_OVERRIDE = os.environ.get("GRAVITYSPY_PROJECT_ROOT")
LOCAL_RUNTIME_ROOT_OVERRIDE = os.environ.get(
    "GRAVITYSPY_LOCAL_RUNTIME_ROOT"
)
PROJECT_ROOT_IS_EXPLICIT = PROJECT_ROOT_OVERRIDE is not None
LOCAL_RUNTIME_ROOT_IS_EXPLICIT = LOCAL_RUNTIME_ROOT_OVERRIDE is not None

COLAB_DRIVE_MOUNT_POINT = Path("/content/drive")
COLAB_MYDRIVE_ROOT = COLAB_DRIVE_MOUNT_POINT / "MyDrive"
DEFAULT_COLAB_PROJECT_ROOT = COLAB_MYDRIVE_ROOT / "TESI_SAKAI2022"

if PROJECT_ROOT_IS_EXPLICIT:
    PROJECT_ROOT = absolute_root(
        PROJECT_ROOT_OVERRIDE,
        "GRAVITYSPY_PROJECT_ROOT",
    )
elif IN_COLAB:
    PROJECT_ROOT = DEFAULT_COLAB_PROJECT_ROOT
else:
    raise RuntimeError(
        "Fuori da Colab impostare GRAVITYSPY_PROJECT_ROOT "
        "su una root persistente assoluta."
    )

if LOCAL_RUNTIME_ROOT_IS_EXPLICIT:
    LOCAL_RUNTIME_ROOT = absolute_root(
        LOCAL_RUNTIME_ROOT_OVERRIDE,
        "GRAVITYSPY_LOCAL_RUNTIME_ROOT",
    )
elif IN_COLAB:
    LOCAL_RUNTIME_ROOT = Path("/content")
else:
    LOCAL_RUNTIME_ROOT = absolute_root(
        tempfile.gettempdir(),
        "tempfile.gettempdir()",
    ) / "gravityspy_runtime"


# ------------------------------------------------------------
# PERSISTENT PROJECT PATHS
# ------------------------------------------------------------

REPOSITORY_REFERENCE = (
    PROJECT_ROOT
    / "repository_reference"
    / "gravityspy_preprocessing_official"
)

C_REFERENCE = (
    REPOSITORY_REFERENCE
    / "sakai_multires_c"
)

TRANSFER_V1 = (
    PROJECT_ROOT
    / "sakai224_full"
    / "transfer_v1"
)


# ------------------------------------------------------------
# REQUIRED REFERENCE FILES
# ------------------------------------------------------------

REFERENCE_REQUIRED = {
    "repository_root":
        REPOSITORY_REFERENCE,

    "sakai_multires_c":
        C_REFERENCE,

    "cmake":
        C_REFERENCE / "CMakeLists.txt",

    "profile_v2":
        C_REFERENCE
        / "configs"
        / "vae_c_profiles_v2.csv",

    "architecture_contract":
        C_REFERENCE
        / "docs"
        / "vae_c_architecture_contract_v1.md",

    "config_source":
        C_REFERENCE
        / "src"
        / "vae_c_config.c",

    "model_source":
        C_REFERENCE
        / "src"
        / "vae_c_model.c",

    "encoder_source":
        C_REFERENCE
        / "src"
        / "vae_c_model_encoder.c",

    "decoder_source":
        C_REFERENCE
        / "src"
        / "vae_c_model_decoder.c",

    "workspace_source":
        C_REFERENCE
        / "src"
        / "vae_c_model_workspace.c",

    "objective_source":
        C_REFERENCE
        / "src"
        / "vae_c_objective.c",

    "rng_source":
        C_REFERENCE
        / "src"
        / "vae_c_rng.c",

    "adam_source":
        C_REFERENCE
        / "src"
        / "vae_c_adam.c",

    "checkpoint_source":
        C_REFERENCE
        / "src"
        / "vae_c_checkpoint.c",

    "profile_test":
        C_REFERENCE
        / "tests"
        / "test_vae_c_profiles_v2.c",

    "model_forward_test":
        C_REFERENCE
        / "tests"
        / "test_vae_c_model_vae_forward.c",

    "model_backward_test":
        C_REFERENCE
        / "tests"
        / "test_vae_c_model_vae_backward.c",
}


TRANSFER_REQUIRED = {
    "transfer_manifest":
        TRANSFER_V1
        / "transfer_manifest_sha256.txt",

    "event_manifest":
        TRANSFER_V1
        / "manifest_events_272.csv",

    "tensor_manifest":
        TRANSFER_V1
        / "manifest_tensors_224_central.csv",

    "train_split":
        TRANSFER_V1
        / "split_train_indices_seed123.csv",

    "validation_split":
        TRANSFER_V1
        / "split_val_indices_seed123.csv",

    "out272_tar":
        TRANSFER_V1
        / "sakai224_full_out272_gray.tar",

    "central224_tar":
        TRANSFER_V1
        / "sakai224_full_central224.tar",
}


def check_paths(paths):
    missing = []

    for name, path in paths.items():
        exists = path.exists()

        print(
            f"{name:28s} = {exists}"
        )

        if not exists:
            missing.append(
                str(path)
            )

    return missing


print("=== ROOT CONTRACT ===")
print("IN_COLAB =", IN_COLAB)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("LOCAL_RUNTIME_ROOT =", LOCAL_RUNTIME_ROOT)

print()
print("=== PYTHON / PYTORCH ENVIRONMENT ===")
print("PYTHON_VERSION =", sys.version.split()[0])
print("PLATFORM       =", platform.platform())
print("TORCH_VERSION  =", torch.__version__)
print("CUDA_VERSION   =", torch.version.cuda)
print()

cuda_available = torch.cuda.is_available()

print("CUDA_AVAILABLE =", cuda_available)

if cuda_available:
    print(
        "GPU_NAME       =",
        torch.cuda.get_device_name(0),
    )

    properties = torch.cuda.get_device_properties(0)

    print(
        "GPU_TOTAL_MEMORY_GIB =",
        round(
            properties.total_memory
            / (1024 ** 3),
            3,
        ),
    )
else:
    print("GPU_NAME       = NONE")


Mounted at /content/drive
=== PYTHON / PYTORCH ENVIRONMENT ===
PYTHON_VERSION = 3.13.15
PLATFORM       = Linux-6.6.122+-x86_64-with-glibc2.35
TORCH_VERSION  = 2.11.0+cu128
CUDA_VERSION   = 12.8

CUDA_AVAILABLE = True
GPU_NAME       = Tesla T4
GPU_TOTAL_MEMORY_GIB = 14.563

=== REPOSITORY REFERENCE ===
repository_root              = True
sakai_multires_c             = True
cmake                        = True
profile_v2                   = True
architecture_contract        = True
config_source                = True
model_source                 = True
encoder_source               = True
decoder_source               = True
workspace_source             = True
objective_source             = True
rng_source                   = True
adam_source                  = True
checkpoint_source            = True
profile_test                 = True
model_forward_test           = True
model_backward_test          = True

=== SAKAI224 FULL TRANSFER V1 ===
transfer_manifest            = True
event_manifest

In [1]:
import torch
import psutil
import shutil


# ------------------------------------------------------------
# CONDITIONAL COLAB DRIVE MOUNT
# ------------------------------------------------------------

if IN_COLAB and not PROJECT_ROOT_IS_EXPLICIT:
    from google.colab import drive

    if not COLAB_MYDRIVE_ROOT.is_dir():
        drive.mount(
            str(COLAB_DRIVE_MOUNT_POINT),
            force_remount=False,
        )
    else:
        print(
            "Drive già disponibile:",
            COLAB_DRIVE_MOUNT_POINT,
        )
else:
    print(
        "Mount Drive non richiesto; PROJECT_ROOT =",
        PROJECT_ROOT,
    )


# ------------------------------------------------------------
# REFERENCE AND TRANSFER PREFLIGHT
# ------------------------------------------------------------

print()
print("=== REPOSITORY REFERENCE ===")

reference_missing = check_paths(
    REFERENCE_REQUIRED
)

print()
print("=== SAKAI224 FULL TRANSFER V1 ===")

transfer_missing = check_paths(
    TRANSFER_REQUIRED
)

reference_ok = len(reference_missing) == 0
transfer_ok = len(transfer_missing) == 0

print()
print(
    "REFERENCE_FILES_OK =",
    reference_ok,
)

print(
    "TRANSFER_V1_OK =",
    transfer_ok,
)

print(
    "NOTEBOOK_02_PREFLIGHT_PASS =",
    (
        reference_ok
        and transfer_ok
        and cuda_available
    ),
)


assert torch.cuda.is_available()

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
runtime_disk_probe = existing_disk_probe(
    LOCAL_RUNTIME_ROOT
)

print()
print("=== CURRENT GPU CHECK ===")
print("GPU_NAME =", gpu_name)
print(
    "GPU_VRAM_GIB =",
    round(gpu_props.total_memory / (1024**3), 3),
)
print(
    "COMPUTE_CAPABILITY =",
    torch.cuda.get_device_capability(0),
)
print("PYTORCH =", torch.__version__)
print("TORCH_CUDA =", torch.version.cuda)
print(
    "RAM_GIB =",
    round(psutil.virtual_memory().total / (1024**3), 3),
)
print(
    "RUNTIME_FREE_GIB =",
    round(
        shutil.disk_usage(runtime_disk_probe).free
        / (1024**3),
        3,
    ),
)
print("RUNTIME_DISK_PROBE =", runtime_disk_probe)


=== CURRENT GPU CHECK ===
GPU_NAME = NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU_VRAM_GIB = 94.971
COMPUTE_CAPABILITY = (12, 0)
PYTORCH = 2.11.0+cu128
TORCH_CUDA = 12.8
RAM_GIB = 176.885
CONTENT_FREE_GIB = 188.493


In [ ]:
# ============================================================
# NOTEBOOK 02 — SAKAI224 FULL ARCHITECTURE CONTRACT
# STEP 02-B: SHAPES + PARAMETER COUNT + MAC COUNT
# ============================================================

from dataclasses import dataclass


# ------------------------------------------------------------
# SPECIFICA FULL
#
# PAPER:
# - input 4x224x224
# - channels 64/128/256/512
# - latent 512
# - topology / shapes from Sakai Figure 8
#
# PROJECT:
# - Conv3x3
# - encoder stride 2, padding 1
# - MaxPool2d 2x2 stride 2
# - decoder hidden Conv3x3 stride 1 padding 1
# - BatchNorm semantics PyTorch-like
# - logvar
# - final linear logits
# ------------------------------------------------------------

@dataclass(frozen=True)
class Sakai224FullSpec:
    input_channels: int = 4
    input_side: int = 224

    channels: tuple = (64, 128, 256, 512)
    latent_dim: int = 512
    bottleneck_side: int = 7

    conv_kernel: int = 3
    conv_padding: int = 1
    encoder_stride: int = 2
    decoder_stride: int = 1

    maxpool_kernel: int = 2
    maxpool_stride: int = 2

    bn_eps: float = 1.0e-5
    bn_momentum: float = 0.1

    beta_kl: float = 1.0


SPEC = Sakai224FullSpec()


# ------------------------------------------------------------
# NON RIUTILIZZARE mini224_bridge_v1
# ------------------------------------------------------------

profile_v2_text = (
    REFERENCE_REQUIRED["profile_v2"]
    .read_text(encoding="utf-8")
)

bridge_present = "mini224_bridge_v1" in profile_v2_text
full_profile_present = "sakai224_full" in profile_v2_text

print("=== PROFILE SEPARATION ===")
print("MINI224_BRIDGE_PRESENT =", bridge_present)
print("SAKAI224_FULL_IN_C_CATALOG =", full_profile_present)

assert bridge_present
assert not full_profile_present


# ------------------------------------------------------------
# GEOMETRIA ENCODER
# ------------------------------------------------------------

def conv_out_side(
    side,
    kernel,
    stride,
    padding,
):
    return (
        (side + 2 * padding - kernel)
        // stride
        + 1
    )


def pool_out_side(
    side,
    kernel,
    stride,
):
    return (
        (side - kernel)
        // stride
        + 1
    )


encoder_sides = []

side = SPEC.input_side

# Conv1
side = conv_out_side(
    side,
    SPEC.conv_kernel,
    SPEC.encoder_stride,
    SPEC.conv_padding,
)
encoder_sides.append(side)

# MaxPool after Conv1
pool_side = pool_out_side(
    side,
    SPEC.maxpool_kernel,
    SPEC.maxpool_stride,
)

# Conv2, Conv3, Conv4
side = pool_side

for _ in range(3):
    side = conv_out_side(
        side,
        SPEC.conv_kernel,
        SPEC.encoder_stride,
        SPEC.conv_padding,
    )
    encoder_sides.append(side)


EXPECTED_ENCODER_SIDES = (
    112,
    28,
    14,
    7,
)

EXPECTED_POOL_SIDE = 56

assert tuple(encoder_sides) == EXPECTED_ENCODER_SIDES
assert pool_side == EXPECTED_POOL_SIDE
assert encoder_sides[-1] == SPEC.bottleneck_side


# ------------------------------------------------------------
# GEOMETRIA DECODER
# ------------------------------------------------------------

decoder_upsample_targets = (
    14,
    28,
    56,
    112,
    224,
)

decoder_hidden_conv_sides = (
    28,
    56,
    112,
)

assert decoder_upsample_targets == (
    14,
    28,
    56,
    112,
    224,
)

assert decoder_hidden_conv_sides == (
    28,
    56,
    112,
)

assert decoder_upsample_targets[-1] == SPEC.input_side


# ------------------------------------------------------------
# SHAPE CONTRACT
# ------------------------------------------------------------

shape_contract = {
    "input":
        ("N", 4, 224, 224),

    "enc_conv1":
        ("N", 64, 112, 112),

    "enc_pool":
        ("N", 64, 56, 56),

    "enc_conv2":
        ("N", 128, 28, 28),

    "enc_conv3":
        ("N", 256, 14, 14),

    "enc_conv4":
        ("N", 512, 7, 7),

    "enc_gap":
        ("N", 512),

    "enc_packed":
        ("N", 1024),

    "mu":
        ("N", 512),

    "logvar":
        ("N", 512),

    "decoder_fc":
        ("N", 512 * 7 * 7),

    "decoder_reshape":
        ("N", 512, 7, 7),

    "dec_up14":
        ("N", 512, 14, 14),

    "dec_up28":
        ("N", 512, 28, 28),

    "dec_conv256":
        ("N", 256, 28, 28),

    "dec_up56":
        ("N", 256, 56, 56),

    "dec_conv128":
        ("N", 128, 56, 56),

    "dec_up112":
        ("N", 128, 112, 112),

    "dec_conv64":
        ("N", 64, 112, 112),

    "dec_up224":
        ("N", 64, 224, 224),

    "logits":
        ("N", 4, 224, 224),
}


# ------------------------------------------------------------
# PARAMETER COUNT
# ------------------------------------------------------------

K2 = SPEC.conv_kernel ** 2
C = SPEC.channels
L = SPEC.latent_dim

parameter_count = 0

# Encoder Conv + BN
in_channels = SPEC.input_channels

for out_channels in C:
    parameter_count += (
        out_channels
        * in_channels
        * K2
    )

    # BatchNorm gamma + beta
    parameter_count += (
        2 * out_channels
    )

    in_channels = out_channels


# Encoder FC:
# 512 -> 2*latent = 1024
# WITH bias
encoder_fc_out = 2 * L

parameter_count += (
    C[-1]
    * encoder_fc_out
)

parameter_count += encoder_fc_out


# Decoder FC:
# latent -> 512*7*7
# NO bias
decoder_fc_features = (
    C[-1]
    * SPEC.bottleneck_side
    * SPEC.bottleneck_side
)

parameter_count += (
    L
    * decoder_fc_features
)

# Decoder FC BatchNorm1d
parameter_count += (
    2 * decoder_fc_features
)


# Decoder hidden Conv + BN
decoder_pairs = (
    (512, 256),
    (256, 128),
    (128, 64),
)

for in_channels, out_channels in decoder_pairs:
    parameter_count += (
        out_channels
        * in_channels
        * K2
    )

    parameter_count += (
        2 * out_channels
    )


# Final Conv:
# 64 -> 4
# WITH bias
parameter_count += (
    4 * 64 * K2
)

parameter_count += 4


EXPECTED_PARAMETER_COUNT = 16_524_548

assert parameter_count == EXPECTED_PARAMETER_COUNT


# ------------------------------------------------------------
# FORWARD MAC COUNT
#
# Come nel configuratore C:
# SOLO Conv + Fully Connected.
#
# Non include:
# BN, ReLU, Pooling, Upsampling,
# BCE, KL, sigmoid o memory copies.
# ------------------------------------------------------------

mac_count = 0


def conv_mac(
    out_channels,
    out_side,
    in_channels,
    kernel,
):
    return (
        out_channels
        * out_side
        * out_side
        * in_channels
        * kernel
        * kernel
    )


# Encoder Conv1
mac_count += conv_mac(
    64,
    112,
    4,
    3,
)

# Encoder Conv2
mac_count += conv_mac(
    128,
    28,
    64,
    3,
)

# Encoder Conv3
mac_count += conv_mac(
    256,
    14,
    128,
    3,
)

# Encoder Conv4
mac_count += conv_mac(
    512,
    7,
    256,
    3,
)

# Encoder FC
mac_count += (
    512 * 1024
)

# Decoder FC
mac_count += (
    512
    * (512 * 7 * 7)
)

# Decoder hidden Conv1
mac_count += conv_mac(
    256,
    28,
    512,
    3,
)

# Decoder hidden Conv2
mac_count += conv_mac(
    128,
    56,
    256,
    3,
)

# Decoder hidden Conv3
mac_count += conv_mac(
    64,
    112,
    128,
    3,
)

# Final output Conv
mac_count += conv_mac(
    4,
    224,
    64,
    3,
)


EXPECTED_FORWARD_MAC = 3_105_816_576

assert mac_count == EXPECTED_FORWARD_MAC


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print()
print("=== SAKAI224 FULL SPEC ===")

print(
    "INPUT_SHAPE =",
    (4, 224, 224),
)

print(
    "CHANNELS =",
    SPEC.channels,
)

print(
    "LATENT_DIM =",
    SPEC.latent_dim,
)

print(
    "BOTTLENECK_SIDE =",
    SPEC.bottleneck_side,
)

print(
    "ENCODER_CONV_SIDES =",
    tuple(encoder_sides),
)

print(
    "POOL_SIDE =",
    pool_side,
)

print(
    "DECODER_UPSAMPLE_TARGETS =",
    decoder_upsample_targets,
)

print(
    "DECODER_HIDDEN_CONV_SIDES =",
    decoder_hidden_conv_sides,
)

print()
print(
    "TRAINABLE_PARAMETERS =",
    parameter_count,
)

print(
    "EXPECTED_TRAINABLE_PARAMETERS =",
    EXPECTED_PARAMETER_COUNT,
)

print()
print(
    "FORWARD_MAC_PER_SAMPLE =",
    mac_count,
)

print(
    "EXPECTED_FORWARD_MAC_PER_SAMPLE =",
    EXPECTED_FORWARD_MAC,
)

print()
print(
    "FULL_ARCHITECTURE_CONTRACT_PASS =",
    (
        parameter_count
        == EXPECTED_PARAMETER_COUNT
        and mac_count
        == EXPECTED_FORWARD_MAC
    ),
)

=== PROFILE SEPARATION ===
MINI224_BRIDGE_PRESENT = True
SAKAI224_FULL_IN_C_CATALOG = False

=== SAKAI224 FULL SPEC ===
INPUT_SHAPE = (4, 224, 224)
CHANNELS = (64, 128, 256, 512)
LATENT_DIM = 512
BOTTLENECK_SIDE = 7
ENCODER_CONV_SIDES = (112, 28, 14, 7)
POOL_SIDE = 56
DECODER_UPSAMPLE_TARGETS = (14, 28, 56, 112, 224)
DECODER_HIDDEN_CONV_SIDES = (28, 56, 112)

TRAINABLE_PARAMETERS = 16524548
EXPECTED_TRAINABLE_PARAMETERS = 16524548

FORWARD_MAC_PER_SAMPLE = 3105816576
EXPECTED_FORWARD_MAC_PER_SAMPLE = 3105816576

FULL_ARCHITECTURE_CONTRACT_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — SAKAI224 FULL PYTORCH MODEL
# STEP 02-C: ARCHITECTURE DEFINITION + STRUCTURAL GATES
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class Sakai224FullEncoder(nn.Module):
    def __init__(self, spec):
        super().__init__()

        eps = spec.bn_eps
        momentum = spec.bn_momentum

        self.latent_dim = spec.latent_dim

        # 224 -> 112
        self.conv1 = nn.Conv2d(
            4, 64,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(
            64,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )
        self.relu1 = nn.ReLU(inplace=False)

        # 112 -> 56
        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
            padding=0,
        )

        # 56 -> 28
        self.conv2 = nn.Conv2d(
            64, 128,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(
            128,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )
        self.relu2 = nn.ReLU(inplace=False)

        # 28 -> 14
        self.conv3 = nn.Conv2d(
            128, 256,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )
        self.bn3 = nn.BatchNorm2d(
            256,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )
        self.relu3 = nn.ReLU(inplace=False)

        # 14 -> 7
        self.conv4 = nn.Conv2d(
            256, 512,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False,
        )
        self.bn4 = nn.BatchNorm2d(
            512,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )
        self.relu4 = nn.ReLU(inplace=False)

        # GAP -> packed [mu | logvar]
        self.fc = nn.Linear(
            512,
            2 * spec.latent_dim,
            bias=True,
        )

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.relu3(self.bn3(self.conv3(x)))
        x = self.relu4(self.bn4(self.conv4(x)))

        # Global Average Pooling:
        # (N,512,7,7) -> (N,512)
        x = x.mean(dim=(2, 3))

        packed = self.fc(x)

        # C contract:
        # first half = mu
        # second half = logvar
        mu = packed[:, :self.latent_dim]
        logvar = packed[:, self.latent_dim:]

        return mu, logvar


class Sakai224FullDecoder(nn.Module):
    def __init__(self, spec):
        super().__init__()

        eps = spec.bn_eps
        momentum = spec.bn_momentum

        self.bottleneck_channels = 512
        self.bottleneck_side = 7
        self.decoder_fc_features = 512 * 7 * 7

        # latent 512 -> 512*7*7
        # No bias: immediately followed by BatchNorm1d.
        self.fc = nn.Linear(
            spec.latent_dim,
            self.decoder_fc_features,
            bias=False,
        )

        self.bn_fc = nn.BatchNorm1d(
            self.decoder_fc_features,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )
        self.relu_fc = nn.ReLU(inplace=False)

        # 28x28: 512 -> 256
        self.conv1 = nn.Conv2d(
            512, 256,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(
            256,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )
        self.relu1 = nn.ReLU(inplace=False)

        # 56x56: 256 -> 128
        self.conv2 = nn.Conv2d(
            256, 128,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(
            128,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )
        self.relu2 = nn.ReLU(inplace=False)

        # 112x112: 128 -> 64
        self.conv3 = nn.Conv2d(
            128, 64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn3 = nn.BatchNorm2d(
            64,
            eps=eps,
            momentum=momentum,
            affine=True,
            track_running_stats=True,
        )
        self.relu3 = nn.ReLU(inplace=False)

        # Final linear logits.
        # No BatchNorm, no ReLU, no sigmoid.
        self.out_conv = nn.Conv2d(
            64, 4,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=True,
        )

    def forward(self, z):
        x = self.fc(z)
        x = self.relu_fc(self.bn_fc(x))

        x = x.reshape(
            z.shape[0],
            self.bottleneck_channels,
            self.bottleneck_side,
            self.bottleneck_side,
        )

        # Two consecutive nearest upsamplings,
        # exactly as in the architecture contract.
        x = F.interpolate(
            x,
            size=(14, 14),
            mode="nearest",
        )

        x = F.interpolate(
            x,
            size=(28, 28),
            mode="nearest",
        )

        x = self.relu1(self.bn1(self.conv1(x)))

        x = F.interpolate(
            x,
            size=(56, 56),
            mode="nearest",
        )

        x = self.relu2(self.bn2(self.conv2(x)))

        x = F.interpolate(
            x,
            size=(112, 112),
            mode="nearest",
        )

        x = self.relu3(self.bn3(self.conv3(x)))

        x = F.interpolate(
            x,
            size=(224, 224),
            mode="nearest",
        )

        logits = self.out_conv(x)

        return logits


class Sakai224FullVAE(nn.Module):
    def __init__(self, spec):
        super().__init__()

        self.spec = spec

        self.encoder = Sakai224FullEncoder(spec)
        self.decoder = Sakai224FullDecoder(spec)

    def encode(self, x):
        return self.encoder(x)

    @staticmethod
    def reparameterize(
        mu,
        logvar,
        epsilon=None,
        mean_mode=False,
    ):
        # Deterministic official embedding/reconstruction path.
        if mean_mode:
            return mu

        # PROJECT semantics:
        # RNG is NOT generated inside the model.
        if epsilon is None:
            raise ValueError(
                "epsilon is required when mean_mode=False"
            )

        if epsilon.shape != mu.shape:
            raise ValueError(
                "epsilon shape must match mu/logvar"
            )

        sigma = torch.exp(
            0.5 * logvar
        )

        return mu + sigma * epsilon

    def decode(self, z):
        return self.decoder(z)

    def forward(
        self,
        x,
        epsilon=None,
        mean_mode=False,
    ):
        mu, logvar = self.encode(x)

        z = self.reparameterize(
            mu,
            logvar,
            epsilon=epsilon,
            mean_mode=mean_mode,
        )

        logits = self.decode(z)

        return logits, mu, logvar, z


# ------------------------------------------------------------
# INSTANTIATE
#
# ATTENZIONE:
# I pesi mantengono qui l'inizializzazione corrente.
# DEFAULT di PyTorch.
#
# NON verranno usati per training o confronti.
# La successiva inizializzazione esplicita è verificata separatamente.
# ------------------------------------------------------------

model = Sakai224FullVAE(SPEC)


# ------------------------------------------------------------
# STRUCTURAL HARD GATES
# ------------------------------------------------------------

trainable_parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

parameter_tensor_count = len(
    list(model.parameters())
)


assert (
    trainable_parameter_count
    == EXPECTED_PARAMETER_COUNT
)

assert parameter_tensor_count == 28


# Encoder contracts
assert model.encoder.conv1.bias is None
assert model.encoder.conv2.bias is None
assert model.encoder.conv3.bias is None
assert model.encoder.conv4.bias is None

assert model.encoder.fc.bias is not None

assert model.encoder.pool.kernel_size == 2
assert model.encoder.pool.stride == 2
assert model.encoder.pool.padding == 0


# Decoder contracts
assert model.decoder.fc.bias is None

assert model.decoder.conv1.bias is None
assert model.decoder.conv2.bias is None
assert model.decoder.conv3.bias is None

assert model.decoder.out_conv.bias is not None


# BatchNorm contracts
batchnorm_layers = [
    model.encoder.bn1,
    model.encoder.bn2,
    model.encoder.bn3,
    model.encoder.bn4,
    model.decoder.bn_fc,
    model.decoder.bn1,
    model.decoder.bn2,
    model.decoder.bn3,
]

for bn in batchnorm_layers:
    assert bn.eps == SPEC.bn_eps
    assert bn.momentum == SPEC.bn_momentum
    assert bn.affine is True
    assert bn.track_running_stats is True


# Final output must be linear logits.
assert isinstance(
    model.decoder.out_conv,
    nn.Conv2d,
)


print("=== PYTORCH MODEL STRUCTURAL AUDIT ===")

print(
    "MODEL_CLASS =",
    model.__class__.__name__,
)

print(
    "PARAMETER_TENSORS =",
    parameter_tensor_count,
)

print(
    "TRAINABLE_PARAMETERS =",
    trainable_parameter_count,
)

print(
    "EXPECTED_TRAINABLE_PARAMETERS =",
    EXPECTED_PARAMETER_COUNT,
)

print(
    "ENCODER_FC_BIAS =",
    model.encoder.fc.bias is not None,
)

print(
    "DECODER_FC_BIAS =",
    model.decoder.fc.bias is not None,
)

print(
    "FINAL_CONV_BIAS =",
    model.decoder.out_conv.bias is not None,
)

print(
    "RNG_INSIDE_MODEL =",
    False,
)

print(
    "MODEL_DEVICE =",
    next(model.parameters()).device,
)

print(
    "INITIALIZATION_STATUS =",
    "PYTORCH_DEFAULT_TEMPORARY",
)

print()
print(
    "PYTORCH_MODEL_STRUCTURE_PASS =",
    (
        parameter_tensor_count == 28
        and trainable_parameter_count
        == EXPECTED_PARAMETER_COUNT
    ),
)

=== PYTORCH MODEL STRUCTURAL AUDIT ===
MODEL_CLASS = Sakai224FullVAE
PARAMETER_TENSORS = 28
TRAINABLE_PARAMETERS = 16524548
EXPECTED_TRAINABLE_PARAMETERS = 16524548
ENCODER_FC_BIAS = True
DECODER_FC_BIAS = False
FINAL_CONV_BIAS = True
RNG_INSIDE_MODEL = False
MODEL_DEVICE = cpu
INITIALIZATION_STATUS = PYTORCH_DEFAULT_TEMPORARY

PYTORCH_MODEL_STRUCTURE_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — SAKAI224 FULL FIDELITY CONTRACT
# STEP 02-D
# ============================================================

SAKAI_FIDELITY = {
    # --------------------------------------------------------
    # PAPER — Sakai et al. 2022, Case 3 / Figure 8
    # --------------------------------------------------------

    "paper_input_shape": (4, 224, 224),

    "paper_encoder_channels": (
        64,
        128,
        256,
        512,
    ),

    "paper_latent_dim": 512,

    "paper_optimizer": "Adam",

    "paper_learning_rate": 5.0e-4,

    "paper_beta1": 0.9,

    "paper_beta2": 0.999,

    "paper_adam_epsilon": 1.0e-8,

    "paper_batch_size_target": 128,

    # The encoder used for the downstream analysis
    # is the VAE encoder after 100 epochs.
    "paper_official_epoch": 100,

    "paper_likelihood": "Bernoulli",

    "paper_decoder_upsampling": "nearest",

    # --------------------------------------------------------
    # PROJECT — choices required because the paper does not
    # fully specify these implementation details.
    # --------------------------------------------------------

    "project_variance_parameterization": "logvar",

    "project_conv_kernel": 3,

    "project_encoder_stride": 2,

    "project_conv_padding": 1,

    "project_maxpool_kernel": 2,

    "project_maxpool_stride": 2,

    "project_bn_eps": 1.0e-5,

    "project_bn_momentum": 0.1,

    "project_main_output": "linear_logits",

    "project_reconstruction_loss":
        "BCEWithLogits",

    "project_beta_kl": 1.0,

    "project_kl_warmup": False,

    "project_logvar_clamp": False,

    "project_rng_inside_model": False,

    "project_official_embedding": "mu",

    "project_official_reconstruction_latent": "mu",

    # --------------------------------------------------------
    # KNOWN PAPER AMBIGUITY
    # --------------------------------------------------------

    "paper_final_layer_figure":
        "Conv + BatchNorm + ReLU",

    "paper_final_layer_objective":
        "Bernoulli likelihood",

    "project_resolution_of_final_layer_ambiguity":
        "linear logits + BCE-with-logits",
}


# ============================================================
# HARD GATES
# ============================================================

assert (
    SAKAI_FIDELITY["paper_input_shape"]
    == (4, 224, 224)
)

assert (
    SAKAI_FIDELITY["paper_encoder_channels"]
    == SPEC.channels
)

assert (
    SAKAI_FIDELITY["paper_latent_dim"]
    == SPEC.latent_dim
)

assert (
    SAKAI_FIDELITY["paper_official_epoch"]
    == 100
)

assert (
    SAKAI_FIDELITY["paper_batch_size_target"]
    == 128
)

assert (
    SAKAI_FIDELITY["paper_learning_rate"]
    == 5.0e-4
)

assert (
    SAKAI_FIDELITY["project_rng_inside_model"]
    is False
)

assert (
    SAKAI_FIDELITY["project_main_output"]
    == "linear_logits"
)


print("=== SAKAI224 FULL FIDELITY CONTRACT ===")

print(
    "PAPER_INPUT =",
    SAKAI_FIDELITY["paper_input_shape"],
)

print(
    "PAPER_CHANNELS =",
    SAKAI_FIDELITY["paper_encoder_channels"],
)

print(
    "PAPER_LATENT_DIM =",
    SAKAI_FIDELITY["paper_latent_dim"],
)

print(
    "PAPER_BATCH_SIZE_TARGET =",
    SAKAI_FIDELITY["paper_batch_size_target"],
)

print(
    "PAPER_LEARNING_RATE =",
    SAKAI_FIDELITY["paper_learning_rate"],
)

print(
    "PAPER_OFFICIAL_EPOCH =",
    SAKAI_FIDELITY["paper_official_epoch"],
)

print(
    "OFFICIAL_MODEL_SELECTION =",
    "EPOCH_100",
)

print(
    "BEST_CHECKPOINT_ROLE =",
    "DIAGNOSTIC_ONLY",
)

print(
    "FINAL_LAYER_PAPER_FIGURE =",
    SAKAI_FIDELITY[
        "paper_final_layer_figure"
    ],
)

print(
    "FINAL_LAYER_PROJECT_MAIN =",
    SAKAI_FIDELITY[
        "project_resolution_of_final_layer_ambiguity"
    ],
)

print()
print(
    "SAKAI_FIDELITY_CONTRACT_PASS = True"
)

=== SAKAI224 FULL FIDELITY CONTRACT ===
PAPER_INPUT = (4, 224, 224)
PAPER_CHANNELS = (64, 128, 256, 512)
PAPER_LATENT_DIM = 512
PAPER_BATCH_SIZE_TARGET = 128
PAPER_LEARNING_RATE = 0.0005
PAPER_OFFICIAL_EPOCH = 100
OFFICIAL_MODEL_SELECTION = EPOCH_100
BEST_CHECKPOINT_ROLE = DIAGNOSTIC_ONLY
FINAL_LAYER_PAPER_FIGURE = Conv + BatchNorm + ReLU
FINAL_LAYER_PROJECT_MAIN = linear logits + BCE-with-logits

SAKAI_FIDELITY_CONTRACT_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — PROJECT C RNG PARITY
# STEP 02-E1
#
# PAPER:
#   initialization RNG is NOT specified by Sakai et al.
#
# PROJECT:
#   reproduce the verified C RNG exactly before using it
#   for model initialization.
# ============================================================

MASK64 = (1 << 64) - 1

SPLITMIX_GAMMA = 0x9E3779B97F4A7C15
SPLITMIX_MUL1 = 0xBF58476D1CE4E5B9
SPLITMIX_MUL2 = 0x94D049BB133111EB

DERIVE_XOR = 0xD1B54A32D192ED03

RNG_DOMAIN_INITIALIZATION = 1
RNG_DOMAIN_SHUFFLE = 2
RNG_DOMAIN_AUGMENTATION = 3
RNG_DOMAIN_LATENT_TRAIN = 4
RNG_DOMAIN_LATENT_VALIDATION = 5

PROJECT_INITIALIZATION_SEED = 123


def c_mix64(value):
    value = (
        value + SPLITMIX_GAMMA
    ) & MASK64

    value = (
        (value ^ (value >> 30))
        * SPLITMIX_MUL1
    ) & MASK64

    value = (
        (value ^ (value >> 27))
        * SPLITMIX_MUL2
    ) & MASK64

    return (
        value ^ (value >> 31)
    ) & MASK64


def c_next_u64(state):
    state = (
        state + SPLITMIX_GAMMA
    ) & MASK64

    value = state

    value = (
        (value ^ (value >> 30))
        * SPLITMIX_MUL1
    ) & MASK64

    value = (
        (value ^ (value >> 27))
        * SPLITMIX_MUL2
    ) & MASK64

    value = (
        value ^ (value >> 31)
    ) & MASK64

    return state, value


def c_derive_seed(
    base_seed,
    domain,
    epoch,
    manifest_index,
    stream_index,
):
    value = c_mix64(
        (base_seed ^ DERIVE_XOR)
        & MASK64
    )

    value = c_mix64(
        value ^ domain
    )

    value = c_mix64(
        value ^ epoch
    )

    value = c_mix64(
        value ^ manifest_index
    )

    value = c_mix64(
        value ^ stream_index
    )

    return value


# ------------------------------------------------------------
# GOLDEN VECTOR 1:
# SplitMix64 sequence from C test_vae_c_rng_init.c
# ------------------------------------------------------------

expected_splitmix = [
    0xE220A8397B1DCDAF,
    0x6E789E6AA1B965F4,
    0x06C45D188009454F,
    0xF88BB8A8724C81EC,
    0x1B39896A51A8749B,
]

state = 0
observed_splitmix = []

for _ in range(5):
    state, value = c_next_u64(state)
    observed_splitmix.append(value)

splitmix_pass = (
    observed_splitmix
    == expected_splitmix
)


# ------------------------------------------------------------
# GOLDEN VECTOR 2:
# seed derivation from the real C test
# ------------------------------------------------------------

seed_cases = [
    (
        123,
        RNG_DOMAIN_INITIALIZATION,
        0,
        0,
        7,
        0x1E073756348C0AD9,
    ),
    (
        456,
        RNG_DOMAIN_LATENT_TRAIN,
        3,
        42,
        0,
        0xBFF9489E190BCDD4,
    ),
    (
        456,
        RNG_DOMAIN_LATENT_VALIDATION,
        0,
        42,
        0,
        0xB849E85BFE7894E7,
    ),
    (
        0,
        RNG_DOMAIN_INITIALIZATION,
        0,
        0,
        0,
        0xAAFF81EB53AF65A4,
    ),
]

seed_derivation_pass = True

for (
    base_seed,
    domain,
    epoch,
    manifest_index,
    stream_index,
    expected,
) in seed_cases:

    observed = c_derive_seed(
        base_seed,
        domain,
        epoch,
        manifest_index,
        stream_index,
    )

    case_ok = (
        observed == expected
    )

    seed_derivation_pass &= case_ok

    print(
        "DERIVE_CASE",
        (
            base_seed,
            domain,
            epoch,
            manifest_index,
            stream_index,
        ),
        "=",
        hex(observed),
        "PASS =",
        case_ok,
    )


# ------------------------------------------------------------
# HARD GATES
# ------------------------------------------------------------

assert splitmix_pass
assert seed_derivation_pass


INITIALIZATION_POLICY = (
    "PROJECT_C_EXACT_V1"
)


print()
print("=== PROJECT C RNG AUDIT ===")

print(
    "SPLITMIX_GOLDEN_PASS =",
    splitmix_pass,
)

print(
    "SEED_DERIVATION_GOLDEN_PASS =",
    seed_derivation_pass,
)

print(
    "INITIALIZATION_SEED =",
    PROJECT_INITIALIZATION_SEED,
)

print(
    "INITIALIZATION_POLICY =",
    INITIALIZATION_POLICY,
)

print(
    "SAKAI_INITIALIZATION_PUBLISHED =",
    False,
)

print()
print(
    "PROJECT_C_RNG_PARITY_PASS =",
    (
        splitmix_pass
        and seed_derivation_pass
    ),
)

DERIVE_CASE (123, 1, 0, 0, 7) = 0x1e073756348c0ad9 PASS = True
DERIVE_CASE (456, 4, 3, 42, 0) = 0xbff9489e190bcdd4 PASS = True
DERIVE_CASE (456, 5, 0, 42, 0) = 0xb849e85bfe7894e7 PASS = True
DERIVE_CASE (0, 1, 0, 0, 0) = 0xaaff81eb53af65a4 PASS = True

=== PROJECT C RNG AUDIT ===
SPLITMIX_GOLDEN_PASS = True
SEED_DERIVATION_GOLDEN_PASS = True
INITIALIZATION_SEED = 123
INITIALIZATION_POLICY = PROJECT_C_EXACT_V1
SAKAI_INITIALIZATION_PUBLISHED = False

PROJECT_C_RNG_PARITY_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — PROJECT INITIALIZATION PARITY
# STEP 02-E2
#
# Verify the Python implementation BITWISE against
# the actual C initialization primitives compiled in the active runtime.
#
# This still DOES NOT modify `model`.
# ============================================================

import ctypes
import math
import subprocess
from pathlib import Path

import numpy as np


# ------------------------------------------------------------
# BUILD THE REAL C REFERENCE LIBRARY
# ------------------------------------------------------------

C_INIT_LIB = (
    LOCAL_RUNTIME_ROOT
    / "libvae_c_init_reference.so"
)

compile_command = [
    "gcc",
    "-std=c11",
    "-O2",
    "-shared",
    "-fPIC",

    "-I",
    str(C_REFERENCE / "include"),

    str(
        C_REFERENCE
        / "src"
        / "vae_c_rng.c"
    ),

    str(
        C_REFERENCE
        / "src"
        / "vae_c_init.c"
    ),

    "-lm",

    "-o",
    str(C_INIT_LIB),
]


compile_result = subprocess.run(
    compile_command,
    check=True,
    capture_output=True,
    text=True,
)


assert C_INIT_LIB.exists()


# ------------------------------------------------------------
# LOAD C FUNCTIONS
# ------------------------------------------------------------

c_init = ctypes.CDLL(
    str(C_INIT_LIB)
)

FLOAT_PTR = ctypes.POINTER(
    ctypes.c_float
)


c_init.vae_c_init_he_uniform_f32.argtypes = [
    FLOAT_PTR,
    ctypes.c_size_t,
    ctypes.c_uint64,
    ctypes.c_uint64,
    ctypes.c_size_t,
]

c_init.vae_c_init_he_uniform_f32.restype = (
    ctypes.c_int
)


c_init.vae_c_init_xavier_uniform_f32.argtypes = [
    FLOAT_PTR,
    ctypes.c_size_t,
    ctypes.c_uint64,
    ctypes.c_uint64,
    ctypes.c_size_t,
    ctypes.c_size_t,
]

c_init.vae_c_init_xavier_uniform_f32.restype = (
    ctypes.c_int
)


# ------------------------------------------------------------
# VECTORIZED PYTHON IMPLEMENTATION
#
# Same C algorithm:
#
# derived seed
# -> sequential SplitMix64
# -> top 24 bits
# -> float32 uniform [0,1)
# -> double calculation of bound transform
# -> cast to float32
# ------------------------------------------------------------

def project_c_uniform_values(
    count,
    base_seed,
    parameter_key,
    bound,
    chunk_size=262_144,
):
    output = np.empty(
        count,
        dtype=np.float32,
    )

    derived_seed = c_derive_seed(
        base_seed,
        RNG_DOMAIN_INITIALIZATION,
        0,
        0,
        parameter_key,
    )

    seed64 = np.uint64(
        derived_seed
    )

    gamma64 = np.uint64(
        SPLITMIX_GAMMA
    )

    mul1_64 = np.uint64(
        SPLITMIX_MUL1
    )

    mul2_64 = np.uint64(
        SPLITMIX_MUL2
    )

    uniform_scale = np.float32(
        1.0 / 16_777_216.0
    )

    position = 0

    while position < count:
        block_count = min(
            chunk_size,
            count - position,
        )

        sequence = np.arange(
            position + 1,
            position + block_count + 1,
            dtype=np.uint64,
        )

        # uint64 overflow is intentional:
        # it reproduces C unsigned arithmetic.
        value = (
            seed64
            + gamma64 * sequence
        )

        value = (
            (
                value
                ^ (value >> np.uint64(30))
            )
            * mul1_64
        )

        value = (
            (
                value
                ^ (value >> np.uint64(27))
            )
            * mul2_64
        )

        value = (
            value
            ^ (value >> np.uint64(31))
        )

        bits24 = (
            value >> np.uint64(40)
        )

        uniform = (
            bits24.astype(np.float32)
            * uniform_scale
        )

        # C performs this expression in double
        # and then stores float32.
        candidate = (
            -bound
            + 2.0
            * bound
            * uniform.astype(np.float64)
        )

        output[
            position:
            position + block_count
        ] = candidate.astype(
            np.float32
        )

        position += block_count

    return output


def python_he_values(
    count,
    seed,
    key,
    fan_in,
):
    bound = math.sqrt(
        6.0 / float(fan_in)
    )

    return project_c_uniform_values(
        count,
        seed,
        key,
        bound,
    )


def python_xavier_values(
    count,
    seed,
    key,
    fan_in,
    fan_out,
):
    bound = math.sqrt(
        6.0
        / float(
            fan_in + fan_out
        )
    )

    return project_c_uniform_values(
        count,
        seed,
        key,
        bound,
    )


def c_he_values(
    count,
    seed,
    key,
    fan_in,
):
    output = np.empty(
        count,
        dtype=np.float32,
    )

    status = (
        c_init
        .vae_c_init_he_uniform_f32(
            output.ctypes.data_as(
                FLOAT_PTR
            ),
            count,
            seed,
            key,
            fan_in,
        )
    )

    assert status == 0

    return output


def c_xavier_values(
    count,
    seed,
    key,
    fan_in,
    fan_out,
):
    output = np.empty(
        count,
        dtype=np.float32,
    )

    status = (
        c_init
        .vae_c_init_xavier_uniform_f32(
            output.ctypes.data_as(
                FLOAT_PTR
            ),
            count,
            seed,
            key,
            fan_in,
            fan_out,
        )
    )

    assert status == 0

    return output


# ------------------------------------------------------------
# REAL PARAMETER-KEY CASES FROM THE C MODEL
# ------------------------------------------------------------

PARITY_COUNT = 4096

parity_cases = [
    # name, kind, key, fan_in, fan_out

    (
        "encoder_conv1",
        "he",
        1,
        4 * 3 * 3,
        None,
    ),

    (
        "encoder_conv4",
        "he",
        10,
        256 * 3 * 3,
        None,
    ),

    (
        "encoder_fc",
        "xavier",
        13,
        512,
        1024,
    ),

    (
        "decoder_fc",
        "he",
        15,
        512,
        None,
    ),

    (
        "decoder_output",
        "xavier",
        27,
        64 * 3 * 3,
        4 * 3 * 3,
    ),
]


all_cases_pass = True


print(
    "=== PYTHON vs REAL C INITIALIZATION ==="
)


for (
    name,
    kind,
    key,
    fan_in,
    fan_out,
) in parity_cases:

    if kind == "he":
        python_values = python_he_values(
            PARITY_COUNT,
            PROJECT_INITIALIZATION_SEED,
            key,
            fan_in,
        )

        c_values = c_he_values(
            PARITY_COUNT,
            PROJECT_INITIALIZATION_SEED,
            key,
            fan_in,
        )

    else:
        python_values = python_xavier_values(
            PARITY_COUNT,
            PROJECT_INITIALIZATION_SEED,
            key,
            fan_in,
            fan_out,
        )

        c_values = c_xavier_values(
            PARITY_COUNT,
            PROJECT_INITIALIZATION_SEED,
            key,
            fan_in,
            fan_out,
        )


    # Compare raw IEEE-754 float32 bits.
    case_pass = np.array_equal(
        python_values.view(np.uint32),
        c_values.view(np.uint32),
    )

    all_cases_pass &= case_pass

    print(
        f"{name:20s}",
        "KEY =",
        key,
        "BITWISE_PASS =",
        case_pass,
    )


# ------------------------------------------------------------
# HARD GATE
# ------------------------------------------------------------

assert all_cases_pass


print()
print(
    "C_REFERENCE_LIBRARY =",
    C_INIT_LIB,
)

print(
    "PARITY_VALUES_PER_CASE =",
    PARITY_COUNT,
)

print(
    "MODEL_MODIFIED =",
    False,
)

print()
print(
    "PROJECT_C_INIT_PRIMITIVE_PARITY_PASS =",
    all_cases_pass,
)

=== PYTHON vs REAL C INITIALIZATION ===
encoder_conv1        KEY = 1 BITWISE_PASS = True
encoder_conv4        KEY = 10 BITWISE_PASS = True
encoder_fc           KEY = 13 BITWISE_PASS = True
decoder_fc           KEY = 15 BITWISE_PASS = True
decoder_output       KEY = 27 BITWISE_PASS = True

C_REFERENCE_LIBRARY = /content/libvae_c_init_reference.so
PARITY_VALUES_PER_CASE = 4096
MODEL_MODIFIED = False

PROJECT_C_INIT_PRIMITIVE_PARITY_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — EXACT PROJECT C MODEL INITIALIZATION
# STEP 02-E3
#
# PAPER:
#   Sakai initialization is not published.
#
# PROJECT:
#   reproduce the verified C initialization policy exactly.
# ============================================================

import hashlib
import numpy as np
import torch


# ------------------------------------------------------------
# PARAMETER PLAN
#
# Keys are the exact stable keys from vae_c_model.h
# and follow the C registry order 1..28.
# ------------------------------------------------------------

def build_project_parameter_plan(model):
    return [
        # key, tensor, kind, fan_in, fan_out

        (1,  model.encoder.conv1.weight, "he", 4 * 3 * 3, None),
        (2,  model.encoder.bn1.weight,    "one", None, None),
        (3,  model.encoder.bn1.bias,      "zero", None, None),

        (4,  model.encoder.conv2.weight, "he", 64 * 3 * 3, None),
        (5,  model.encoder.bn2.weight,    "one", None, None),
        (6,  model.encoder.bn2.bias,      "zero", None, None),

        (7,  model.encoder.conv3.weight, "he", 128 * 3 * 3, None),
        (8,  model.encoder.bn3.weight,    "one", None, None),
        (9,  model.encoder.bn3.bias,      "zero", None, None),

        (10, model.encoder.conv4.weight, "he", 256 * 3 * 3, None),
        (11, model.encoder.bn4.weight,    "one", None, None),
        (12, model.encoder.bn4.bias,      "zero", None, None),

        (13, model.encoder.fc.weight, "xavier", 512, 1024),
        (14, model.encoder.fc.bias,   "zero", None, None),

        (15, model.decoder.fc.weight, "he", 512, None),

        (16, model.decoder.bn_fc.weight, "one", None, None),
        (17, model.decoder.bn_fc.bias,   "zero", None, None),

        (18, model.decoder.conv1.weight, "he", 512 * 3 * 3, None),
        (19, model.decoder.bn1.weight,    "one", None, None),
        (20, model.decoder.bn1.bias,      "zero", None, None),

        (21, model.decoder.conv2.weight, "he", 256 * 3 * 3, None),
        (22, model.decoder.bn2.weight,    "one", None, None),
        (23, model.decoder.bn2.bias,      "zero", None, None),

        (24, model.decoder.conv3.weight, "he", 128 * 3 * 3, None),
        (25, model.decoder.bn3.weight,    "one", None, None),
        (26, model.decoder.bn3.bias,      "zero", None, None),

        (27, model.decoder.out_conv.weight, "xavier", 64 * 3 * 3, 4 * 3 * 3),
        (28, model.decoder.out_conv.bias,   "zero", None, None),
    ]


def get_project_batchnorm_layers(model):
    return [
        model.encoder.bn1,
        model.encoder.bn2,
        model.encoder.bn3,
        model.encoder.bn4,
        model.decoder.bn_fc,
        model.decoder.bn1,
        model.decoder.bn2,
        model.decoder.bn3,
    ]


# ------------------------------------------------------------
# EXACT INITIALIZATION
# ------------------------------------------------------------

def initialize_project_c_exact_v1(
    model,
    seed=PROJECT_INITIALIZATION_SEED,
):
    assert next(model.parameters()).device.type == "cpu"

    plan = build_project_parameter_plan(model)

    keys = [entry[0] for entry in plan]

    assert keys == list(range(1, 29))
    assert len(plan) == 28

    with torch.no_grad():

        for (
            key,
            parameter,
            kind,
            fan_in,
            fan_out,
        ) in plan:

            assert parameter.dtype == torch.float32

            count = parameter.numel()

            if kind == "zero":
                parameter.zero_()

            elif kind == "one":
                parameter.fill_(1.0)

            elif kind == "he":
                values = python_he_values(
                    count,
                    seed,
                    key,
                    fan_in,
                )

                source = torch.from_numpy(
                    values.reshape(
                        tuple(parameter.shape)
                    )
                )

                parameter.copy_(source)

            elif kind == "xavier":
                values = python_xavier_values(
                    count,
                    seed,
                    key,
                    fan_in,
                    fan_out,
                )

                source = torch.from_numpy(
                    values.reshape(
                        tuple(parameter.shape)
                    )
                )

                parameter.copy_(source)

            else:
                raise RuntimeError(
                    f"Unknown initialization kind: {kind}"
                )

        # Match PROJECT BatchNorm initial state.
        for bn in get_project_batchnorm_layers(model):
            bn.running_mean.zero_()
            bn.running_var.fill_(1.0)

            if bn.num_batches_tracked is not None:
                bn.num_batches_tracked.zero_()


# ------------------------------------------------------------
# CANONICAL PARAMETER HASH
#
# Raw float32 bytes concatenated in C parameter-key order.
# This does NOT include PyTorch-only BN num_batches_tracked.
# ------------------------------------------------------------

def project_parameter_sha256(model):
    digest = hashlib.sha256()

    plan = build_project_parameter_plan(model)

    for key, parameter, kind, fan_in, fan_out in plan:
        array = (
            parameter
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        assert array.dtype == np.float32

        digest.update(
            array.tobytes(order="C")
        )

    return digest.hexdigest()


# ------------------------------------------------------------
# FIRST INITIALIZATION
# ------------------------------------------------------------

initialize_project_c_exact_v1(
    model,
    PROJECT_INITIALIZATION_SEED,
)

hash_first = project_parameter_sha256(
    model
)


# ------------------------------------------------------------
# REPEATABILITY CHECK
#
# Reinitialize the same model with the same seed.
# Hash must remain identical.
# ------------------------------------------------------------

initialize_project_c_exact_v1(
    model,
    PROJECT_INITIALIZATION_SEED,
)

hash_second = project_parameter_sha256(
    model
)

repeatable = (
    hash_first == hash_second
)


# ------------------------------------------------------------
# STATE AUDIT
# ------------------------------------------------------------

all_parameters_finite = all(
    torch.isfinite(parameter).all().item()
    for parameter in model.parameters()
)


bn_state_ok = True

for bn in get_project_batchnorm_layers(model):

    bn_state_ok &= torch.equal(
        bn.weight,
        torch.ones_like(bn.weight),
    )

    bn_state_ok &= torch.equal(
        bn.bias,
        torch.zeros_like(bn.bias),
    )

    bn_state_ok &= torch.equal(
        bn.running_mean,
        torch.zeros_like(bn.running_mean),
    )

    bn_state_ok &= torch.equal(
        bn.running_var,
        torch.ones_like(bn.running_var),
    )

    bn_state_ok &= (
        int(bn.num_batches_tracked.item())
        == 0
    )


bias_state_ok = (
    torch.equal(
        model.encoder.fc.bias,
        torch.zeros_like(
            model.encoder.fc.bias
        ),
    )
    and
    torch.equal(
        model.decoder.out_conv.bias,
        torch.zeros_like(
            model.decoder.out_conv.bias
        ),
    )
)


parameter_count_after_init = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


# ------------------------------------------------------------
# HARD GATES
# ------------------------------------------------------------

assert repeatable
assert all_parameters_finite
assert bn_state_ok
assert bias_state_ok

assert (
    parameter_count_after_init
    == EXPECTED_PARAMETER_COUNT
)


INITIALIZATION_STATUS = (
    "PROJECT_C_EXACT_V1"
)


print(
    "=== EXACT MODEL INITIALIZATION AUDIT ==="
)

print(
    "INITIALIZATION_SEED =",
    PROJECT_INITIALIZATION_SEED,
)

print(
    "INITIALIZATION_STATUS =",
    INITIALIZATION_STATUS,
)

print(
    "SAKAI_INITIALIZATION_PUBLISHED =",
    False,
)

print(
    "PARAMETER_KEY_RANGE =",
    (1, 28),
)

print(
    "TRAINABLE_PARAMETERS =",
    parameter_count_after_init,
)

print(
    "ALL_PARAMETERS_FINITE =",
    all_parameters_finite,
)

print(
    "BN_INITIAL_STATE_OK =",
    bn_state_ok,
)

print(
    "BIAS_INITIAL_STATE_OK =",
    bias_state_ok,
)

print(
    "REINITIALIZATION_BITWISE_REPEATABLE =",
    repeatable,
)

print()
print(
    "PROJECT_PARAMETER_SHA256 =",
    hash_first,
)

print()
print(
    "MODEL_DEVICE =",
    next(model.parameters()).device,
)

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print()
print(
    "PROJECT_C_EXACT_MODEL_INIT_PASS =",
    (
        repeatable
        and all_parameters_finite
        and bn_state_ok
        and bias_state_ok
        and parameter_count_after_init
        == EXPECTED_PARAMETER_COUNT
    ),
)

=== EXACT MODEL INITIALIZATION AUDIT ===
INITIALIZATION_SEED = 123
INITIALIZATION_STATUS = PROJECT_C_EXACT_V1
SAKAI_INITIALIZATION_PUBLISHED = False
PARAMETER_KEY_RANGE = (1, 28)
TRAINABLE_PARAMETERS = 16524548
ALL_PARAMETERS_FINITE = True
BN_INITIAL_STATE_OK = True
BIAS_INITIAL_STATE_OK = True
REINITIALIZATION_BITWISE_REPEATABLE = True

PROJECT_PARAMETER_SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b

MODEL_DEVICE = cpu
MODEL_FORWARD_EXECUTED = False

PROJECT_C_EXACT_MODEL_INIT_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — FIRST CONTROLLED FULL FORWARD
# STEP 02-F
#
# Mode:
#   EVAL + MEAN
#
# Therefore:
#   z = mu
#   no latent RNG
#   no BatchNorm running-stat updates
#   no optimizer
#   no backward
# ============================================================

import torch


# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device("cuda")

assert torch.cuda.is_available()

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

model = model.to(
    device=device,
    dtype=torch.float32,
)

model.eval()

assert next(model.parameters()).device.type == "cuda"
assert next(model.parameters()).dtype == torch.float32


# ------------------------------------------------------------
# PARAMETER HASH MUST SURVIVE CPU -> GPU TRANSFER
# ------------------------------------------------------------

FROZEN_INIT_SHA256 = (
    "1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b"
)

hash_after_device_transfer = (
    project_parameter_sha256(model)
)

assert (
    hash_after_device_transfer
    == FROZEN_INIT_SHA256
)


# ------------------------------------------------------------
# SYNTHETIC INPUT
#
# No RNG is used.
# Two deterministic samples in [0,1].
# ------------------------------------------------------------

values_per_sample = (
    SPEC.input_channels
    * SPEC.input_side
    * SPEC.input_side
)

sample_a = torch.linspace(
    0.0,
    1.0,
    steps=values_per_sample,
    dtype=torch.float32,
    device=device,
).reshape(
    1,
    4,
    224,
    224,
)

sample_b = (
    1.0 - sample_a
)

x_smoke = torch.cat(
    [sample_a, sample_b],
    dim=0,
)

assert x_smoke.shape == (
    2,
    4,
    224,
    224,
)

assert x_smoke.dtype == torch.float32
assert x_smoke.device.type == "cuda"

assert float(x_smoke.min()) >= 0.0
assert float(x_smoke.max()) <= 1.0


# ------------------------------------------------------------
# INTERNAL SHAPE TRACE
#
# Hooks store only shape tuples, never activations.
# ------------------------------------------------------------

trace = {}


def make_shape_hook(name):
    def hook(module, inputs, output):

        input_tensor = inputs[0]

        trace[name] = {
            "input":
                tuple(input_tensor.shape),

            "output":
                tuple(output.shape),
        }

    return hook


hook_targets = {
    "enc_conv1":
        model.encoder.conv1,

    "enc_pool":
        model.encoder.pool,

    "enc_conv2":
        model.encoder.conv2,

    "enc_conv3":
        model.encoder.conv3,

    "enc_conv4":
        model.encoder.conv4,

    "enc_fc":
        model.encoder.fc,

    "dec_fc":
        model.decoder.fc,

    "dec_conv1":
        model.decoder.conv1,

    "dec_conv2":
        model.decoder.conv2,

    "dec_conv3":
        model.decoder.conv3,

    "dec_out":
        model.decoder.out_conv,
}


handles = [
    module.register_forward_hook(
        make_shape_hook(name)
    )
    for name, module
    in hook_targets.items()
]


# ------------------------------------------------------------
# SNAPSHOT BATCHNORM STATE BEFORE EVAL FORWARD
# ------------------------------------------------------------

bn_layers = get_project_batchnorm_layers(
    model
)


def snapshot_bn_state(layers):
    return [
        (
            bn.running_mean.detach().clone(),
            bn.running_var.detach().clone(),
            bn.num_batches_tracked.detach().clone(),
        )
        for bn in layers
    ]


bn_before = snapshot_bn_state(
    bn_layers
)


# ------------------------------------------------------------
# CONTROLLED FORWARD
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

with torch.no_grad():

    logits, mu, logvar, z = model(
        x_smoke,
        epsilon=None,
        mean_mode=True,
    )

torch.cuda.synchronize()


# Remove hooks immediately.
for handle in handles:
    handle.remove()


bn_after = snapshot_bn_state(
    bn_layers
)


# ------------------------------------------------------------
# OUTPUT HARD GATES
# ------------------------------------------------------------

assert logits.shape == (
    2,
    4,
    224,
    224,
)

assert mu.shape == (
    2,
    512,
)

assert logvar.shape == (
    2,
    512,
)

assert z.shape == (
    2,
    512,
)


# MEAN mode must be exactly z = mu.
mean_mode_exact = torch.equal(
    z,
    mu,
)

assert mean_mode_exact


all_outputs_finite = (
    torch.isfinite(logits).all().item()
    and torch.isfinite(mu).all().item()
    and torch.isfinite(logvar).all().item()
    and torch.isfinite(z).all().item()
)

assert all_outputs_finite


# ------------------------------------------------------------
# INTERNAL GEOMETRY HARD GATES
# ------------------------------------------------------------

expected_trace = {
    "enc_conv1": {
        "input":  (2, 4, 224, 224),
        "output": (2, 64, 112, 112),
    },

    "enc_pool": {
        "input":  (2, 64, 112, 112),
        "output": (2, 64, 56, 56),
    },

    "enc_conv2": {
        "input":  (2, 64, 56, 56),
        "output": (2, 128, 28, 28),
    },

    "enc_conv3": {
        "input":  (2, 128, 28, 28),
        "output": (2, 256, 14, 14),
    },

    "enc_conv4": {
        "input":  (2, 256, 14, 14),
        "output": (2, 512, 7, 7),
    },

    "enc_fc": {
        "input":  (2, 512),
        "output": (2, 1024),
    },

    "dec_fc": {
        "input":  (2, 512),
        "output": (2, 512 * 7 * 7),
    },

    # Its input being 28x28 proves the two consecutive
    # nearest upsamplings 7 -> 14 -> 28 occurred.
    "dec_conv1": {
        "input":  (2, 512, 28, 28),
        "output": (2, 256, 28, 28),
    },

    "dec_conv2": {
        "input":  (2, 256, 56, 56),
        "output": (2, 128, 56, 56),
    },

    "dec_conv3": {
        "input":  (2, 128, 112, 112),
        "output": (2, 64, 112, 112),
    },

    "dec_out": {
        "input":  (2, 64, 224, 224),
        "output": (2, 4, 224, 224),
    },
}


geometry_pass = (
    trace == expected_trace
)

assert geometry_pass


# ------------------------------------------------------------
# EVAL MODE MUST NOT MODIFY BATCHNORM RUNNING STATE
# ------------------------------------------------------------

bn_eval_unchanged = True

for before, after in zip(
    bn_before,
    bn_after,
):
    for before_tensor, after_tensor in zip(
        before,
        after,
    ):
        if not torch.equal(
            before_tensor,
            after_tensor,
        ):
            bn_eval_unchanged = False


assert bn_eval_unchanged


# ------------------------------------------------------------
# PARAMETERS MUST ALSO REMAIN UNCHANGED
# ------------------------------------------------------------

hash_after_forward = (
    project_parameter_sha256(model)
)

parameters_unchanged = (
    hash_after_forward
    == FROZEN_INIT_SHA256
)

assert parameters_unchanged


# ------------------------------------------------------------
# MEMORY DIAGNOSTIC
# ------------------------------------------------------------

peak_allocated_mib = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 2)
)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print(
    "=== FIRST SAKAI224 FULL FORWARD AUDIT ==="
)

print(
    "DEVICE =",
    next(model.parameters()).device,
)

print(
    "INPUT_SHAPE =",
    tuple(x_smoke.shape),
)

print(
    "LOGITS_SHAPE =",
    tuple(logits.shape),
)

print(
    "MU_SHAPE =",
    tuple(mu.shape),
)

print(
    "LOGVAR_SHAPE =",
    tuple(logvar.shape),
)

print(
    "Z_SHAPE =",
    tuple(z.shape),
)

print()

print(
    "MEAN_MODE_Z_EQUALS_MU =",
    mean_mode_exact,
)

print(
    "ALL_OUTPUTS_FINITE =",
    all_outputs_finite,
)

print(
    "INTERNAL_GEOMETRY_PASS =",
    geometry_pass,
)

print(
    "BN_EVAL_STATE_UNCHANGED =",
    bn_eval_unchanged,
)

print(
    "PARAMETERS_UNCHANGED =",
    parameters_unchanged,
)

print()

print(
    "PARAMETER_SHA256 =",
    hash_after_forward,
)

print(
    "PEAK_CUDA_ALLOCATED_MIB =",
    round(
        peak_allocated_mib,
        3,
    ),
)

print()

print(
    "FORWARD_MODE =",
    "MEAN+EVAL",
)

print(
    "LATENT_RNG_USED =",
    False,
)

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print()

print(
    "SAKAI224_FULL_MEAN_EVAL_FORWARD_PASS =",
    (
        mean_mode_exact
        and all_outputs_finite
        and geometry_pass
        and bn_eval_unchanged
        and parameters_unchanged
    ),
)

=== FIRST SAKAI224 FULL FORWARD AUDIT ===
DEVICE = cuda:0
INPUT_SHAPE = (2, 4, 224, 224)
LOGITS_SHAPE = (2, 4, 224, 224)
MU_SHAPE = (2, 512)
LOGVAR_SHAPE = (2, 512)
Z_SHAPE = (2, 512)

MEAN_MODE_Z_EQUALS_MU = True
ALL_OUTPUTS_FINITE = True
INTERNAL_GEOMETRY_PASS = True
BN_EVAL_STATE_UNCHANGED = True
PARAMETERS_UNCHANGED = True

PARAMETER_SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b
PEAK_CUDA_ALLOCATED_MIB = 107.272

FORWARD_MODE = MEAN+EVAL
LATENT_RNG_USED = False
BACKWARD_EXECUTED = False
OPTIMIZER_EXECUTED = False

SAKAI224_FULL_MEAN_EVAL_FORWARD_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — VAE LOSS CONTRACT
# STEP 02-G
#
# PAPER:
#   Bernoulli reconstruction likelihood + Gaussian KL.
#
# PROJECT:
#   logvar parameterization
#   linear logits + BCE-with-logits
#   beta_KL = 1
#   sum over reconstruction/latent dimensions
#   mean ONLY over batch
#
# No model backward and no optimizer in this step.
# ============================================================

import math
import numpy as np
import torch
import torch.nn.functional as F


# ------------------------------------------------------------
# DIFFERENTIABLE PYTORCH LOSS
# ------------------------------------------------------------

def sakai_vae_loss(
    logits,
    targets,
    mu,
    logvar,
    beta_kl=1.0,
):
    if logits.shape != targets.shape:
        raise ValueError(
            "logits and targets must have identical shapes"
        )

    if mu.shape != logvar.shape:
        raise ValueError(
            "mu and logvar must have identical shapes"
        )

    if logits.shape[0] != mu.shape[0]:
        raise ValueError(
            "reconstruction and latent batch sizes must match"
        )

    if beta_kl != 1.0:
        raise ValueError(
            "sakai224_full main contract requires beta_kl=1"
        )

    batch_size = logits.shape[0]

    # Reconstruction:
    # SUM over every channel/pixel in the whole batch.
    reconstruction_sum = (
        F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="sum",
        )
    )

    # KL:
    #
    # 0.5 * (
    #     mu^2
    #     + exp(logvar)
    #     - 1
    #     - logvar
    # )
    #
    # expm1(logvar) = exp(logvar) - 1
    kl_sum = 0.5 * torch.sum(
        mu.square()
        + torch.expm1(logvar)
        - logvar
    )

    total_sum = (
        reconstruction_sum
        + beta_kl * kl_sum
    )

    # IMPORTANT:
    # mean over BATCH ONLY.
    loss = total_sum / batch_size

    stats = {
        "loss":
            loss,

        "total_sum":
            total_sum,

        "reconstruction_sum":
            reconstruction_sum,

        "reconstruction_mean_per_sample":
            reconstruction_sum / batch_size,

        "reconstruction_mean_per_value":
            reconstruction_sum / logits.numel(),

        "kl_sum":
            kl_sum,

        "kl_mean_per_sample":
            kl_sum / batch_size,

        "kl_mean_per_dimension":
            kl_sum / mu.numel(),

        "beta_kl":
            float(beta_kl),

        "batch_size":
            int(batch_size),
    }

    return stats


# ============================================================
# PART 1 — TINY ANALYTIC FIXTURE
#
# Values come from the real C VAE-loss test.
# We use a tiny CPU fixture so gradients can be compared
# directly against the closed-form C equations.
# ============================================================

fixture_logits = torch.tensor(
    [
        [-1.2, -0.7, -0.2,  0.3],
        [ 0.6,  1.1,  1.7, -1.8],
    ],
    dtype=torch.float32,
    requires_grad=True,
)

fixture_targets = torch.tensor(
    [
        [0.0, 0.25, 1.0, 0.4],
        [0.75, 0.0, 1.0, 0.6],
    ],
    dtype=torch.float32,
)

fixture_mu = torch.tensor(
    [
        [-1.1, -0.4, 0.2],
        [ 0.7,  1.3, -0.8],
    ],
    dtype=torch.float32,
    requires_grad=True,
)

fixture_logvar = torch.tensor(
    [
        [-0.9, -0.3, 0.1],
        [ 0.4,  0.8, -0.6],
    ],
    dtype=torch.float32,
    requires_grad=True,
)


fixture_stats = sakai_vae_loss(
    fixture_logits,
    fixture_targets,
    fixture_mu,
    fixture_logvar,
    beta_kl=SPEC.beta_kl,
)

fixture_loss = fixture_stats["loss"]

fixture_loss.backward()


# ------------------------------------------------------------
# CLOSED-FORM EXPECTED GRADIENTS
#
# Exact mathematical semantics used by the C backward:
#
# d BCE / d logit =
#     (sigmoid(logit) - target) / batch
#
# d KL / d mu =
#     mu / batch
#
# d KL / d logvar =
#     0.5 * (exp(logvar) - 1) / batch
# ------------------------------------------------------------

fixture_batch = fixture_logits.shape[0]

expected_logits_grad = (
    torch.sigmoid(
        fixture_logits.detach()
    )
    - fixture_targets
) / fixture_batch

expected_mu_grad = (
    fixture_mu.detach()
    / fixture_batch
)

expected_logvar_grad = (
    0.5
    * torch.expm1(
        fixture_logvar.detach()
    )
    / fixture_batch
)


logits_grad_pass = torch.allclose(
    fixture_logits.grad,
    expected_logits_grad,
    rtol=1e-6,
    atol=1e-7,
)

mu_grad_pass = torch.allclose(
    fixture_mu.grad,
    expected_mu_grad,
    rtol=1e-6,
    atol=1e-7,
)

logvar_grad_pass = torch.allclose(
    fixture_logvar.grad,
    expected_logvar_grad,
    rtol=1e-6,
    atol=1e-7,
)


assert logits_grad_pass
assert mu_grad_pass
assert logvar_grad_pass


# ============================================================
# PART 2 — DOUBLE-PRECISION C-LIKE FORWARD REFERENCE
#
# C accumulates diagnostic forward loss in double.
# We reproduce that independently here.
# ============================================================

def c_like_bce_reference(
    logits_array,
    targets_array,
):
    total = 0.0

    logits_flat = (
        np.asarray(
            logits_array,
            dtype=np.float32,
        )
        .reshape(-1)
    )

    targets_flat = (
        np.asarray(
            targets_array,
            dtype=np.float32,
        )
        .reshape(-1)
    )

    for logit_f32, target_f32 in zip(
        logits_flat,
        targets_flat,
    ):
        logit = float(logit_f32)
        target = float(target_f32)

        element = (
            max(logit, 0.0)
            - logit * target
            + math.log1p(
                math.exp(
                    -abs(logit)
                )
            )
        )

        total += element

    return total


def c_like_kl_reference(
    mu_array,
    logvar_array,
):
    total = 0.0

    mu_flat = (
        np.asarray(
            mu_array,
            dtype=np.float32,
        )
        .reshape(-1)
    )

    logvar_flat = (
        np.asarray(
            logvar_array,
            dtype=np.float32,
        )
        .reshape(-1)
    )

    for mu_f32, logvar_f32 in zip(
        mu_flat,
        logvar_flat,
    ):
        mu_value = float(mu_f32)
        logvar_value = float(logvar_f32)

        element = 0.5 * (
            mu_value * mu_value
            + math.expm1(logvar_value)
            - logvar_value
        )

        total += element

    return total


reference_reconstruction_sum = (
    c_like_bce_reference(
        fixture_logits.detach().numpy(),
        fixture_targets.numpy(),
    )
)

reference_kl_sum = (
    c_like_kl_reference(
        fixture_mu.detach().numpy(),
        fixture_logvar.detach().numpy(),
    )
)

reference_total_sum = (
    reference_reconstruction_sum
    + reference_kl_sum
)

reference_mean_per_sample = (
    reference_total_sum
    / fixture_batch
)


# PyTorch uses backend reductions rather than the literal
# scalar C loop, so semantic equivalence is checked with
# numerical tolerance rather than bitwise equality.

reconstruction_forward_pass = math.isclose(
    float(
        fixture_stats[
            "reconstruction_sum"
        ].detach()
    ),
    reference_reconstruction_sum,
    rel_tol=2e-6,
    abs_tol=2e-6,
)

kl_forward_pass = math.isclose(
    float(
        fixture_stats[
            "kl_sum"
        ].detach()
    ),
    reference_kl_sum,
    rel_tol=2e-6,
    abs_tol=2e-6,
)

total_forward_pass = math.isclose(
    float(
        fixture_loss.detach()
    ),
    reference_mean_per_sample,
    rel_tol=2e-6,
    abs_tol=2e-6,
)


assert reconstruction_forward_pass
assert kl_forward_pass
assert total_forward_pass


# ============================================================
# PART 3 — FULL 224 LOSS SMOKE
#
# Use outputs already produced by the successful
# MEAN+EVAL full forward.
#
# NO backward through model.
# ============================================================

with torch.no_grad():

    full_loss_stats = sakai_vae_loss(
        logits,
        x_smoke,
        mu,
        logvar,
        beta_kl=SPEC.beta_kl,
    )


full_loss_finite = all(
    math.isfinite(
        float(value)
    )
    for key, value
    in full_loss_stats.items()
    if isinstance(value, torch.Tensor)
)


assert full_loss_finite

assert (
    full_loss_stats["batch_size"]
    == 2
)

assert (
    full_loss_stats["beta_kl"]
    == 1.0
)


# ------------------------------------------------------------
# MODEL MUST REMAIN COMPLETELY UNCHANGED
# ------------------------------------------------------------

hash_after_loss_audit = (
    project_parameter_sha256(model)
)

model_parameters_unchanged = (
    hash_after_loss_audit
    == FROZEN_INIT_SHA256
)

assert model_parameters_unchanged


# ============================================================
# REPORT
# ============================================================

print(
    "=== SAKAI224 FULL LOSS AUDIT ==="
)

print(
    "BETA_KL =",
    full_loss_stats["beta_kl"],
)

print(
    "REDUCTION =",
    "SUM_DIMENSIONS_THEN_MEAN_BATCH",
)

print()

print(
    "FIXTURE_RECON_FORWARD_PASS =",
    reconstruction_forward_pass,
)

print(
    "FIXTURE_KL_FORWARD_PASS =",
    kl_forward_pass,
)

print(
    "FIXTURE_TOTAL_FORWARD_PASS =",
    total_forward_pass,
)

print()

print(
    "FIXTURE_LOGITS_GRAD_PASS =",
    logits_grad_pass,
)

print(
    "FIXTURE_MU_GRAD_PASS =",
    mu_grad_pass,
)

print(
    "FIXTURE_LOGVAR_GRAD_PASS =",
    logvar_grad_pass,
)

print()

print(
    "FULL_RECON_MEAN_PER_SAMPLE =",
    float(
        full_loss_stats[
            "reconstruction_mean_per_sample"
        ]
    ),
)

print(
    "FULL_RECON_MEAN_PER_VALUE =",
    float(
        full_loss_stats[
            "reconstruction_mean_per_value"
        ]
    ),
)

print(
    "FULL_KL_MEAN_PER_SAMPLE =",
    float(
        full_loss_stats[
            "kl_mean_per_sample"
        ]
    ),
)

print(
    "FULL_KL_MEAN_PER_DIMENSION =",
    float(
        full_loss_stats[
            "kl_mean_per_dimension"
        ]
    ),
)

print(
    "FULL_TOTAL_LOSS =",
    float(
        full_loss_stats["loss"]
    ),
)

print()

print(
    "FULL_LOSS_FINITE =",
    full_loss_finite,
)

print(
    "MODEL_PARAMETERS_UNCHANGED =",
    model_parameters_unchanged,
)

print(
    "MODEL_BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print()

print(
    "SAKAI224_FULL_LOSS_CONTRACT_PASS =",
    (
        reconstruction_forward_pass
        and kl_forward_pass
        and total_forward_pass
        and logits_grad_pass
        and mu_grad_pass
        and logvar_grad_pass
        and full_loss_finite
        and model_parameters_unchanged
    ),
)

=== SAKAI224 FULL LOSS AUDIT ===
BETA_KL = 1.0
REDUCTION = SUM_DIMENSIONS_THEN_MEAN_BATCH

FIXTURE_RECON_FORWARD_PASS = True
FIXTURE_KL_FORWARD_PASS = True
FIXTURE_TOTAL_FORWARD_PASS = True

FIXTURE_LOGITS_GRAD_PASS = True
FIXTURE_MU_GRAD_PASS = True
FIXTURE_LOGVAR_GRAD_PASS = True

FULL_RECON_MEAN_PER_SAMPLE = 141846.234375
FULL_RECON_MEAN_PER_VALUE = 0.7067434191703796
FULL_KL_MEAN_PER_SAMPLE = 44.04901123046875
FULL_KL_MEAN_PER_DIMENSION = 0.08603322505950928
FULL_TOTAL_LOSS = 141890.28125

FULL_LOSS_FINITE = True
MODEL_PARAMETERS_UNCHANGED = True
MODEL_BACKWARD_EXECUTED = False
OPTIMIZER_EXECUTED = False

SAKAI224_FULL_LOSS_CONTRACT_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — STATELESS LATENT EPSILON
# STEP 02-H1: PYTHON <-> REAL C PARITY
#
# PROJECT:
# TRAIN:
#   epsilon = f(seed=456, epoch, manifest_index)
#
# VALIDATION:
#   epsilon = f(seed=456, manifest_index)
#   epoch-independent
#
# No model forward in this step.
# ============================================================

import ctypes
import math
import subprocess
from pathlib import Path

import numpy as np
import torch


PROJECT_LATENT_SEED = 456

TWO_PI = 6.283185307179586476925286766559


# ------------------------------------------------------------
# BUILD CURRENT C RNG REFERENCE
# ------------------------------------------------------------

C_RNG_LIB = (
    LOCAL_RUNTIME_ROOT
    / "libvae_c_rng_reference.so"
)

compile_command = [
    "gcc",
    "-std=c11",
    "-O2",
    "-shared",
    "-fPIC",

    "-I",
    str(C_REFERENCE / "include"),

    str(
        C_REFERENCE
        / "src"
        / "vae_c_rng.c"
    ),

    "-lm",

    "-o",
    str(C_RNG_LIB),
]


subprocess.run(
    compile_command,
    check=True,
    capture_output=True,
    text=True,
)

assert C_RNG_LIB.exists()


# ------------------------------------------------------------
# LOAD REAL C FUNCTIONS
# ------------------------------------------------------------

c_rng = ctypes.CDLL(
    str(C_RNG_LIB)
)

FLOAT_PTR = ctypes.POINTER(
    ctypes.c_float
)


c_rng.vae_c_rng_fill_latent_train_epsilon_f32.argtypes = [
    ctypes.c_uint64,   # base_seed
    ctypes.c_uint64,   # epoch
    ctypes.c_uint64,   # manifest_index
    FLOAT_PTR,
    ctypes.c_size_t,
]

c_rng.vae_c_rng_fill_latent_train_epsilon_f32.restype = (
    ctypes.c_int
)


c_rng.vae_c_rng_fill_latent_validation_epsilon_f32.argtypes = [
    ctypes.c_uint64,   # base_seed
    ctypes.c_uint64,   # manifest_index
    FLOAT_PTR,
    ctypes.c_size_t,
]

c_rng.vae_c_rng_fill_latent_validation_epsilon_f32.restype = (
    ctypes.c_int
)


# ------------------------------------------------------------
# INDEPENDENT PYTHON REFERENCE
#
# Same C algorithm:
#
# derived seed
# -> SplitMix64
# -> open-uniform double using top 52 bits
# -> Box-Muller
# -> float32
# ------------------------------------------------------------

def python_latent_epsilon(
    base_seed,
    domain,
    epoch,
    manifest_index,
    count,
):
    state = c_derive_seed(
        base_seed,
        domain,
        epoch,
        manifest_index,
        0,
    )

    output = np.empty(
        count,
        dtype=np.float32,
    )

    for index in range(count):

        state, random1 = c_next_u64(
            state
        )

        state, random2 = c_next_u64(
            state
        )

        bits52_1 = random1 >> 12
        bits52_2 = random2 >> 12

        uniform1 = (
            float(bits52_1) + 0.5
        ) * (
            1.0
            / 4503599627370496.0
        )

        uniform2 = (
            float(bits52_2) + 0.5
        ) * (
            1.0
            / 4503599627370496.0
        )

        radius = math.sqrt(
            -2.0 * math.log(
                uniform1
            )
        )

        angle = (
            TWO_PI
            * uniform2
        )

        normal = (
            radius
            * math.cos(angle)
        )

        output[index] = np.float32(
            normal
        )

    return output


# ------------------------------------------------------------
# C HELPERS
# ------------------------------------------------------------

def c_train_epsilon(
    manifest_index,
    epoch,
    count=SPEC.latent_dim,
):
    output = np.empty(
        count,
        dtype=np.float32,
    )

    status = (
        c_rng
        .vae_c_rng_fill_latent_train_epsilon_f32(
            PROJECT_LATENT_SEED,
            epoch,
            manifest_index,
            output.ctypes.data_as(
                FLOAT_PTR
            ),
            count,
        )
    )

    assert status == 0

    return output


def c_validation_epsilon(
    manifest_index,
    count=SPEC.latent_dim,
):
    output = np.empty(
        count,
        dtype=np.float32,
    )

    status = (
        c_rng
        .vae_c_rng_fill_latent_validation_epsilon_f32(
            PROJECT_LATENT_SEED,
            manifest_index,
            output.ctypes.data_as(
                FLOAT_PTR
            ),
            count,
        )
    )

    assert status == 0

    return output


# ------------------------------------------------------------
# PYTHON <-> C BITWISE PARITY
# ------------------------------------------------------------

cases = [
    (42,   0),
    (42,   3),
    (100,  8),
    (7663, 0),
]


all_train_parity = True


print(
    "=== TRAIN EPSILON PYTHON vs C ==="
)


for manifest_index, epoch in cases:

    python_values = python_latent_epsilon(
        PROJECT_LATENT_SEED,
        RNG_DOMAIN_LATENT_TRAIN,
        epoch,
        manifest_index,
        SPEC.latent_dim,
    )

    c_values = c_train_epsilon(
        manifest_index,
        epoch,
    )

    case_pass = np.array_equal(
        python_values.view(np.uint32),
        c_values.view(np.uint32),
    )

    all_train_parity &= case_pass

    print(
        "manifest_index =",
        manifest_index,
        "epoch =",
        epoch,
        "BITWISE_PASS =",
        case_pass,
    )


# ------------------------------------------------------------
# VALIDATION PARITY
# ------------------------------------------------------------

validation_manifest = 42

python_validation = python_latent_epsilon(
    PROJECT_LATENT_SEED,
    RNG_DOMAIN_LATENT_VALIDATION,
    0,
    validation_manifest,
    SPEC.latent_dim,
)

c_validation = c_validation_epsilon(
    validation_manifest
)

validation_parity = np.array_equal(
    python_validation.view(np.uint32),
    c_validation.view(np.uint32),
)

assert validation_parity


# ------------------------------------------------------------
# STATELESS SEMANTICS
# ------------------------------------------------------------

train_42_epoch0_a = c_train_epsilon(
    42,
    0,
)

train_42_epoch0_b = c_train_epsilon(
    42,
    0,
)

train_42_epoch1 = c_train_epsilon(
    42,
    1,
)

train_100_epoch0 = c_train_epsilon(
    100,
    0,
)


same_event_same_epoch_repeatable = (
    np.array_equal(
        train_42_epoch0_a.view(np.uint32),
        train_42_epoch0_b.view(np.uint32),
    )
)

epoch_changes_epsilon = (
    not np.array_equal(
        train_42_epoch0_a.view(np.uint32),
        train_42_epoch1.view(np.uint32),
    )
)

manifest_changes_epsilon = (
    not np.array_equal(
        train_42_epoch0_a.view(np.uint32),
        train_100_epoch0.view(np.uint32),
    )
)


# ------------------------------------------------------------
# ORDER INDEPENDENCE
# ------------------------------------------------------------

order_a = {
    42: c_train_epsilon(42, 5),
    100: c_train_epsilon(100, 5),
}

order_b = {
    100: c_train_epsilon(100, 5),
    42: c_train_epsilon(42, 5),
}


order_independent = (
    np.array_equal(
        order_a[42].view(np.uint32),
        order_b[42].view(np.uint32),
    )
    and
    np.array_equal(
        order_a[100].view(np.uint32),
        order_b[100].view(np.uint32),
    )
)


# ------------------------------------------------------------
# VALIDATION IS FIXED BY MANIFEST INDEX
# ------------------------------------------------------------

validation_a = c_validation_epsilon(
    42
)

validation_b = c_validation_epsilon(
    42
)

validation_repeatable = (
    np.array_equal(
        validation_a.view(np.uint32),
        validation_b.view(np.uint32),
    )
)


# Separate RNG domains must differ.
train_validation_domain_separated = (
    not np.array_equal(
        train_42_epoch0_a.view(np.uint32),
        validation_a.view(np.uint32),
    )
)


# ------------------------------------------------------------
# NUMERICAL SANITY
# ------------------------------------------------------------

epsilon_finite = (
    np.isfinite(
        train_42_epoch0_a
    ).all()
    and
    np.isfinite(
        validation_a
    ).all()
)


# ------------------------------------------------------------
# MODEL MUST REMAIN UNCHANGED
# ------------------------------------------------------------

hash_after_epsilon_audit = (
    project_parameter_sha256(
        model
    )
)

model_unchanged = (
    hash_after_epsilon_audit
    == FROZEN_INIT_SHA256
)


# ------------------------------------------------------------
# HARD GATES
# ------------------------------------------------------------

assert all_train_parity
assert validation_parity
assert same_event_same_epoch_repeatable
assert epoch_changes_epsilon
assert manifest_changes_epsilon
assert order_independent
assert validation_repeatable
assert train_validation_domain_separated
assert epsilon_finite
assert model_unchanged


print()
print(
    "=== STATELESS LATENT RNG AUDIT ==="
)

print(
    "TRAIN_PYTHON_C_BITWISE_PASS =",
    all_train_parity,
)

print(
    "VALIDATION_PYTHON_C_BITWISE_PASS =",
    validation_parity,
)

print(
    "SAME_EVENT_SAME_EPOCH_REPEATABLE =",
    same_event_same_epoch_repeatable,
)

print(
    "EPOCH_CHANGES_TRAIN_EPSILON =",
    epoch_changes_epsilon,
)

print(
    "MANIFEST_CHANGES_EPSILON =",
    manifest_changes_epsilon,
)

print(
    "ORDER_INDEPENDENT =",
    order_independent,
)

print(
    "VALIDATION_REPEATABLE =",
    validation_repeatable,
)

print(
    "TRAIN_VALIDATION_DOMAIN_SEPARATED =",
    train_validation_domain_separated,
)

print(
    "EPSILON_FINITE =",
    epsilon_finite,
)

print(
    "MODEL_PARAMETERS_UNCHANGED =",
    model_unchanged,
)

print()

print(
    "LATENT_SEED =",
    PROJECT_LATENT_SEED,
)

print(
    "BITWISE_SCOPE =",
    "CURRENT_COLAB_LIBM",
)

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print()

print(
    "PROJECT_STATELESS_LATENT_RNG_PASS =",
    (
        all_train_parity
        and validation_parity
        and same_event_same_epoch_repeatable
        and epoch_changes_epsilon
        and manifest_changes_epsilon
        and order_independent
        and validation_repeatable
        and train_validation_domain_separated
        and epsilon_finite
        and model_unchanged
    ),
)

=== TRAIN EPSILON PYTHON vs C ===
manifest_index = 42 epoch = 0 BITWISE_PASS = True
manifest_index = 42 epoch = 3 BITWISE_PASS = True
manifest_index = 100 epoch = 8 BITWISE_PASS = True
manifest_index = 7663 epoch = 0 BITWISE_PASS = True

=== STATELESS LATENT RNG AUDIT ===
TRAIN_PYTHON_C_BITWISE_PASS = True
VALIDATION_PYTHON_C_BITWISE_PASS = True
SAME_EVENT_SAME_EPOCH_REPEATABLE = True
EPOCH_CHANGES_TRAIN_EPSILON = True
MANIFEST_CHANGES_EPSILON = True
ORDER_INDEPENDENT = True
VALIDATION_REPEATABLE = True
TRAIN_VALIDATION_DOMAIN_SEPARATED = True
EPSILON_FINITE = True
MODEL_PARAMETERS_UNCHANGED = True

LATENT_SEED = 456
BITWISE_SCOPE = CURRENT_COLAB_LIBM
MODEL_FORWARD_EXECUTED = False

PROJECT_STATELESS_LATENT_RNG_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — FIRST STOCHASTIC FULL FORWARD
# STEP 02-H2
#
# Mode:
#   STOCHASTIC + EVAL
#
# PROJECT:
#   epsilon supplied externally
#   validation epsilon is fixed by manifest_index
#
# No backward.
# No optimizer.
# ============================================================

import math
import numpy as np
import torch


model.eval()

assert next(model.parameters()).device.type == "cuda"

hash_before_stochastic = (
    project_parameter_sha256(model)
)

assert (
    hash_before_stochastic
    == FROZEN_INIT_SHA256
)


# ------------------------------------------------------------
# SMOKE EVENT IDENTITIES
#
# These IDs are used only to exercise the stateless
# validation-epsilon contract in this synthetic model smoke.
# ------------------------------------------------------------

SMOKE_MANIFEST_INDICES = [
    6864,
    1488,
]

assert len(SMOKE_MANIFEST_INDICES) == x_smoke.shape[0]


# ------------------------------------------------------------
# BUILD VALIDATION EPSILON WITH THE VERIFIED PYTHON
# IMPLEMENTATION
# ------------------------------------------------------------

epsilon_rows = []

for manifest_index in SMOKE_MANIFEST_INDICES:

    epsilon_np = python_latent_epsilon(
        PROJECT_LATENT_SEED,
        RNG_DOMAIN_LATENT_VALIDATION,
        0,  # validation is epoch-independent
        manifest_index,
        SPEC.latent_dim,
    )

    epsilon_rows.append(
        epsilon_np
    )


epsilon_validation_np = np.stack(
    epsilon_rows,
    axis=0,
)

epsilon_validation = torch.from_numpy(
    epsilon_validation_np
).to(
    device=device,
    dtype=torch.float32,
)


assert epsilon_validation.shape == (
    2,
    512,
)

assert torch.isfinite(
    epsilon_validation
).all()


# ------------------------------------------------------------
# SNAPSHOT BATCHNORM STATE
# ------------------------------------------------------------

bn_before_stochastic = snapshot_bn_state(
    bn_layers
)


# ------------------------------------------------------------
# STOCHASTIC + EVAL FORWARD
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


with torch.no_grad():

    (
        logits_stochastic,
        mu_stochastic,
        logvar_stochastic,
        z_stochastic,
    ) = model(
        x_smoke,
        epsilon=epsilon_validation,
        mean_mode=False,
    )


torch.cuda.synchronize()


# ------------------------------------------------------------
# ANALYTIC REPARAMETERIZATION CHECK
# ------------------------------------------------------------

with torch.no_grad():

    z_expected = (
        mu_stochastic
        + torch.exp(
            0.5 * logvar_stochastic
        )
        * epsilon_validation
    )


z_formula_pass = torch.allclose(
    z_stochastic,
    z_expected,
    rtol=1e-6,
    atol=1e-7,
)

assert z_formula_pass


# ------------------------------------------------------------
# ENCODER MUST BE IDENTICAL TO PREVIOUS MEAN+EVAL FORWARD
# ------------------------------------------------------------

mu_same_as_mean = torch.equal(
    mu_stochastic,
    mu,
)

logvar_same_as_mean = torch.equal(
    logvar_stochastic,
    logvar,
)

assert mu_same_as_mean
assert logvar_same_as_mean


# ------------------------------------------------------------
# STOCHASTIC LATENT MUST ACTUALLY DIFFER FROM MU
# ------------------------------------------------------------

stochastic_latent_differs_from_mu = (
    not torch.equal(
        z_stochastic,
        mu_stochastic,
    )
)

assert stochastic_latent_differs_from_mu


# ------------------------------------------------------------
# STOCHASTIC DECODER OUTPUT SHOULD DIFFER FROM MEAN PATH
# ------------------------------------------------------------

stochastic_logits_differ_from_mean = (
    not torch.equal(
        logits_stochastic,
        logits,
    )
)

assert stochastic_logits_differ_from_mean


# ------------------------------------------------------------
# SAME EPSILON + SAME INPUT + EVAL MODEL:
# REPEAT THE FORWARD
# ------------------------------------------------------------

with torch.no_grad():

    (
        logits_repeat,
        mu_repeat,
        logvar_repeat,
        z_repeat,
    ) = model(
        x_smoke,
        epsilon=epsilon_validation,
        mean_mode=False,
    )


torch.cuda.synchronize()


same_epsilon_repeatable = (
    torch.equal(
        mu_stochastic,
        mu_repeat,
    )
    and
    torch.equal(
        logvar_stochastic,
        logvar_repeat,
    )
    and
    torch.equal(
        z_stochastic,
        z_repeat,
    )
    and
    torch.equal(
        logits_stochastic,
        logits_repeat,
    )
)

assert same_epsilon_repeatable


# ------------------------------------------------------------
# FINITE OUTPUTS
# ------------------------------------------------------------

all_stochastic_outputs_finite = (
    torch.isfinite(
        logits_stochastic
    ).all().item()
    and
    torch.isfinite(
        mu_stochastic
    ).all().item()
    and
    torch.isfinite(
        logvar_stochastic
    ).all().item()
    and
    torch.isfinite(
        z_stochastic
    ).all().item()
)

assert all_stochastic_outputs_finite


# ------------------------------------------------------------
# STOCHASTIC LOSS SMOKE
#
# Still no backward.
# ------------------------------------------------------------

with torch.no_grad():

    stochastic_loss_stats = sakai_vae_loss(
        logits_stochastic,
        x_smoke,
        mu_stochastic,
        logvar_stochastic,
        beta_kl=SPEC.beta_kl,
    )


stochastic_loss_finite = math.isfinite(
    float(
        stochastic_loss_stats[
            "loss"
        ]
    )
)

assert stochastic_loss_finite


# ------------------------------------------------------------
# BATCHNORM MUST REMAIN UNCHANGED IN EVAL
# ------------------------------------------------------------

bn_after_stochastic = snapshot_bn_state(
    bn_layers
)


bn_stochastic_eval_unchanged = True

for before, after in zip(
    bn_before_stochastic,
    bn_after_stochastic,
):

    for before_tensor, after_tensor in zip(
        before,
        after,
    ):

        if not torch.equal(
            before_tensor,
            after_tensor,
        ):
            bn_stochastic_eval_unchanged = False


assert bn_stochastic_eval_unchanged


# ------------------------------------------------------------
# PARAMETERS MUST REMAIN UNCHANGED
# ------------------------------------------------------------

hash_after_stochastic = (
    project_parameter_sha256(model)
)

parameters_still_unchanged = (
    hash_after_stochastic
    == FROZEN_INIT_SHA256
)

assert parameters_still_unchanged


# ------------------------------------------------------------
# MEMORY DIAGNOSTIC
# ------------------------------------------------------------

peak_cuda_mib = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 2)
)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print(
    "=== STOCHASTIC+EVAL FULL FORWARD AUDIT ==="
)

print(
    "MANIFEST_INDICES =",
    SMOKE_MANIFEST_INDICES,
)

print(
    "EPSILON_SHAPE =",
    tuple(
        epsilon_validation.shape
    ),
)

print(
    "EPSILON_FINITE =",
    bool(
        torch.isfinite(
            epsilon_validation
        ).all()
    ),
)

print()

print(
    "MU_SAME_AS_MEAN_FORWARD =",
    mu_same_as_mean,
)

print(
    "LOGVAR_SAME_AS_MEAN_FORWARD =",
    logvar_same_as_mean,
)

print(
    "Z_FORMULA_PASS =",
    z_formula_pass,
)

print(
    "STOCHASTIC_Z_DIFFERS_FROM_MU =",
    stochastic_latent_differs_from_mu,
)

print(
    "STOCHASTIC_LOGITS_DIFFER_FROM_MEAN =",
    stochastic_logits_differ_from_mean,
)

print(
    "SAME_EPSILON_FORWARD_REPEATABLE =",
    same_epsilon_repeatable,
)

print()

print(
    "ALL_STOCHASTIC_OUTPUTS_FINITE =",
    all_stochastic_outputs_finite,
)

print(
    "STOCHASTIC_LOSS_FINITE =",
    stochastic_loss_finite,
)

print(
    "STOCHASTIC_TOTAL_LOSS =",
    float(
        stochastic_loss_stats[
            "loss"
        ]
    ),
)

print()

print(
    "BN_EVAL_STATE_UNCHANGED =",
    bn_stochastic_eval_unchanged,
)

print(
    "PARAMETERS_UNCHANGED =",
    parameters_still_unchanged,
)

print(
    "PARAMETER_SHA256 =",
    hash_after_stochastic,
)

print()

print(
    "PEAK_CUDA_ALLOCATED_MIB =",
    round(
        peak_cuda_mib,
        3,
    ),
)

print()

print(
    "FORWARD_MODE =",
    "STOCHASTIC+EVAL",
)

print(
    "EPSILON_SOURCE =",
    "PROJECT_STATELESS_VALIDATION",
)

print(
    "RNG_INSIDE_MODEL =",
    False,
)

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print()

print(
    "SAKAI224_FULL_STOCHASTIC_EVAL_PASS =",
    (
        mu_same_as_mean
        and logvar_same_as_mean
        and z_formula_pass
        and stochastic_latent_differs_from_mu
        and stochastic_logits_differ_from_mean
        and same_epsilon_repeatable
        and all_stochastic_outputs_finite
        and stochastic_loss_finite
        and bn_stochastic_eval_unchanged
        and parameters_still_unchanged
    ),
)

=== STOCHASTIC+EVAL FULL FORWARD AUDIT ===
MANIFEST_INDICES = [6864, 1488]
EPSILON_SHAPE = (2, 512)
EPSILON_FINITE = True

MU_SAME_AS_MEAN_FORWARD = True
LOGVAR_SAME_AS_MEAN_FORWARD = True
Z_FORMULA_PASS = True
STOCHASTIC_Z_DIFFERS_FROM_MU = True
STOCHASTIC_LOGITS_DIFFER_FROM_MEAN = True
SAME_EPSILON_FORWARD_REPEATABLE = True

ALL_STOCHASTIC_OUTPUTS_FINITE = True
STOCHASTIC_LOSS_FINITE = True
STOCHASTIC_TOTAL_LOSS = 166818.0625

BN_EVAL_STATE_UNCHANGED = True
PARAMETERS_UNCHANGED = True
PARAMETER_SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b

PEAK_CUDA_ALLOCATED_MIB = 110.784

FORWARD_MODE = STOCHASTIC+EVAL
EPSILON_SOURCE = PROJECT_STATELESS_VALIDATION
RNG_INSIDE_MODEL = False
BACKWARD_EXECUTED = False
OPTIMIZER_EXECUTED = False

SAKAI224_FULL_STOCHASTIC_EVAL_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — END-TO-END BACKWARD AUDIT
# STEP 02-I
#
# Engineering gate:
#   STOCHASTIC + TRAIN
#   full forward
#   VAE loss
#   backward
#
# NO optimizer.
# NO parameter update.
#
# A separate audit model is used so the frozen main model
# remains untouched, including BatchNorm running statistics.
# ============================================================

import math
import numpy as np
import torch


# ------------------------------------------------------------
# FRESH, INDEPENDENT MODEL
# ------------------------------------------------------------

backward_model = Sakai224FullVAE(
    SPEC
)

initialize_project_c_exact_v1(
    backward_model,
    PROJECT_INITIALIZATION_SEED,
)

audit_hash_initial = (
    project_parameter_sha256(
        backward_model
    )
)

assert (
    audit_hash_initial
    == FROZEN_INIT_SHA256
)

backward_model = backward_model.to(
    device=device,
    dtype=torch.float32,
)

backward_model.train()


# ------------------------------------------------------------
# SYNTHETIC TRAIN INPUT
#
# Separate tensor so we can also inspect input gradients.
# ------------------------------------------------------------

x_backward = (
    x_smoke
    .detach()
    .clone()
    .requires_grad_(True)
)

assert x_backward.shape == (
    2,
    4,
    224,
    224,
)


# ------------------------------------------------------------
# TRAINING EPSILON
#
# Use actual train manifest indices from the official split.
# Epoch 0.
# ------------------------------------------------------------

BACKWARD_TRAIN_MANIFEST_INDICES = [
    7663,
    2606,
]

BACKWARD_EPOCH = 0


epsilon_train_np = np.stack(
    [
        python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_TRAIN,
            BACKWARD_EPOCH,
            manifest_index,
            SPEC.latent_dim,
        )
        for manifest_index
        in BACKWARD_TRAIN_MANIFEST_INDICES
    ],
    axis=0,
)


epsilon_train = torch.from_numpy(
    epsilon_train_np
).to(
    device=device,
    dtype=torch.float32,
)


assert epsilon_train.shape == (
    2,
    512,
)

assert torch.isfinite(
    epsilon_train
).all()


# ------------------------------------------------------------
# BATCHNORM STATE BEFORE TRAIN FORWARD
# ------------------------------------------------------------

backward_bn_layers = (
    get_project_batchnorm_layers(
        backward_model
    )
)

bn_before_backward = (
    snapshot_bn_state(
        backward_bn_layers
    )
)


# ------------------------------------------------------------
# CLEAR GRADIENTS
# ------------------------------------------------------------

backward_model.zero_grad(
    set_to_none=True
)

assert all(
    parameter.grad is None
    for parameter
    in backward_model.parameters()
)


# ------------------------------------------------------------
# FULL STOCHASTIC + TRAIN FORWARD
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


(
    logits_train,
    mu_train,
    logvar_train,
    z_train,
) = backward_model(
    x_backward,
    epsilon=epsilon_train,
    mean_mode=False,
)


train_loss_stats = sakai_vae_loss(
    logits_train,
    x_backward,
    mu_train,
    logvar_train,
    beta_kl=SPEC.beta_kl,
)


train_loss = (
    train_loss_stats["loss"]
)


assert torch.isfinite(
    train_loss
).item()


# ------------------------------------------------------------
# PARAMETERS MUST STILL BE UNCHANGED BEFORE BACKWARD
# ------------------------------------------------------------

hash_before_backward = (
    project_parameter_sha256(
        backward_model
    )
)

assert (
    hash_before_backward
    == FROZEN_INIT_SHA256
)


# ------------------------------------------------------------
# BACKWARD
# ------------------------------------------------------------

train_loss.backward()

torch.cuda.synchronize()


# ------------------------------------------------------------
# GRADIENT AUDIT
# ------------------------------------------------------------

gradient_tensor_count = 0
nonzero_gradient_tensor_count = 0

all_gradients_present = True
all_gradients_finite = True

total_gradient_abs_sum = 0.0


for name, parameter in (
    backward_model.named_parameters()
):

    gradient = parameter.grad

    if gradient is None:

        all_gradients_present = False
        continue

    gradient_tensor_count += 1

    finite = torch.isfinite(
        gradient
    ).all().item()

    all_gradients_finite &= finite

    abs_sum = float(
        gradient
        .detach()
        .abs()
        .sum()
    )

    total_gradient_abs_sum += (
        abs_sum
    )

    if abs_sum > 0.0:
        nonzero_gradient_tensor_count += 1


assert all_gradients_present
assert all_gradients_finite

assert gradient_tensor_count == 28

assert total_gradient_abs_sum > 0.0


# ------------------------------------------------------------
# KEY PATHS MUST RECEIVE NONZERO GRADIENTS
# ------------------------------------------------------------

key_gradient_parameters = {
    "encoder_conv1":
        backward_model.encoder.conv1.weight,

    "encoder_fc":
        backward_model.encoder.fc.weight,

    "decoder_fc":
        backward_model.decoder.fc.weight,

    "decoder_output":
        backward_model.decoder.out_conv.weight,
}


key_gradients_nonzero = True


for name, parameter in (
    key_gradient_parameters.items()
):

    gradient = parameter.grad

    case_ok = (
        gradient is not None
        and torch.isfinite(
            gradient
        ).all().item()
        and float(
            gradient
            .detach()
            .abs()
            .sum()
        ) > 0.0
    )

    key_gradients_nonzero &= case_ok

    print(
        "KEY_GRAD",
        name,
        "PASS =",
        case_ok,
    )


assert key_gradients_nonzero


# ------------------------------------------------------------
# INPUT GRADIENT
# ------------------------------------------------------------

input_gradient_present = (
    x_backward.grad is not None
)

input_gradient_finite = (
    input_gradient_present
    and torch.isfinite(
        x_backward.grad
    ).all().item()
)

input_gradient_nonzero = (
    input_gradient_present
    and float(
        x_backward.grad
        .detach()
        .abs()
        .sum()
    ) > 0.0
)


assert input_gradient_present
assert input_gradient_finite
assert input_gradient_nonzero


# ------------------------------------------------------------
# TRAINING BATCHNORM MUST HAVE ADVANCED
# ------------------------------------------------------------

bn_after_backward = (
    snapshot_bn_state(
        backward_bn_layers
    )
)


bn_counters_incremented = True
bn_any_running_state_changed = False


for before, after in zip(
    bn_before_backward,
    bn_after_backward,
):

    before_mean, before_var, before_count = before
    after_mean, after_var, after_count = after

    if int(after_count.item()) != (
        int(before_count.item()) + 1
    ):
        bn_counters_incremented = False

    if (
        not torch.equal(
            before_mean,
            after_mean,
        )
        or
        not torch.equal(
            before_var,
            after_var,
        )
    ):
        bn_any_running_state_changed = True


assert bn_counters_incremented
assert bn_any_running_state_changed


# ------------------------------------------------------------
# BACKWARD MUST NOT UPDATE PARAMETERS
# ------------------------------------------------------------

hash_after_backward = (
    project_parameter_sha256(
        backward_model
    )
)


parameters_unchanged_by_backward = (
    hash_after_backward
    == FROZEN_INIT_SHA256
)

assert parameters_unchanged_by_backward


# ------------------------------------------------------------
# MAIN FROZEN MODEL MUST ALSO BE UNTOUCHED
# ------------------------------------------------------------

main_model_unchanged = (
    project_parameter_sha256(
        model
    )
    == FROZEN_INIT_SHA256
)

assert main_model_unchanged


# ------------------------------------------------------------
# MEMORY
# ------------------------------------------------------------

peak_backward_cuda_mib = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 2)
)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print()
print(
    "=== END-TO-END BACKWARD AUDIT ==="
)

print(
    "FORWARD_MODE =",
    "STOCHASTIC+TRAIN",
)

print(
    "TRAIN_MANIFEST_INDICES =",
    BACKWARD_TRAIN_MANIFEST_INDICES,
)

print(
    "TRAIN_EPOCH =",
    BACKWARD_EPOCH,
)

print()

print(
    "TRAIN_TOTAL_LOSS =",
    float(
        train_loss.detach()
    ),
)

print(
    "TRAIN_RECON_MEAN_PER_SAMPLE =",
    float(
        train_loss_stats[
            "reconstruction_mean_per_sample"
        ].detach()
    ),
)

print(
    "TRAIN_KL_MEAN_PER_SAMPLE =",
    float(
        train_loss_stats[
            "kl_mean_per_sample"
        ].detach()
    ),
)

print()

print(
    "GRADIENT_TENSORS =",
    gradient_tensor_count,
)

print(
    "NONZERO_GRADIENT_TENSORS =",
    nonzero_gradient_tensor_count,
)

print(
    "ALL_GRADIENTS_PRESENT =",
    all_gradients_present,
)

print(
    "ALL_GRADIENTS_FINITE =",
    all_gradients_finite,
)

print(
    "KEY_GRADIENTS_NONZERO =",
    key_gradients_nonzero,
)

print(
    "INPUT_GRADIENT_PRESENT =",
    input_gradient_present,
)

print(
    "INPUT_GRADIENT_FINITE =",
    input_gradient_finite,
)

print(
    "INPUT_GRADIENT_NONZERO =",
    input_gradient_nonzero,
)

print()

print(
    "BN_COUNTERS_INCREMENTED =",
    bn_counters_incremented,
)

print(
    "BN_RUNNING_STATE_CHANGED =",
    bn_any_running_state_changed,
)

print()

print(
    "PARAMETERS_UNCHANGED_BY_BACKWARD =",
    parameters_unchanged_by_backward,
)

print(
    "MAIN_FROZEN_MODEL_UNCHANGED =",
    main_model_unchanged,
)

print(
    "PARAMETER_SHA256_AFTER_BACKWARD =",
    hash_after_backward,
)

print()

print(
    "PEAK_CUDA_ALLOCATED_MIB =",
    round(
        peak_backward_cuda_mib,
        3,
    ),
)

print()

print(
    "BACKWARD_EXECUTED =",
    True,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print()

print(
    "SAKAI224_FULL_BACKWARD_PASS =",
    (
        all_gradients_present
        and all_gradients_finite
        and gradient_tensor_count == 28
        and total_gradient_abs_sum > 0.0
        and key_gradients_nonzero
        and input_gradient_present
        and input_gradient_finite
        and input_gradient_nonzero
        and bn_counters_incremented
        and bn_any_running_state_changed
        and parameters_unchanged_by_backward
        and main_model_unchanged
    ),
)

KEY_GRAD encoder_conv1 PASS = True
KEY_GRAD encoder_fc PASS = True
KEY_GRAD decoder_fc PASS = True
KEY_GRAD decoder_output PASS = True

=== END-TO-END BACKWARD AUDIT ===
FORWARD_MODE = STOCHASTIC+TRAIN
TRAIN_MANIFEST_INDICES = [7663, 2606]
TRAIN_EPOCH = 0

TRAIN_TOTAL_LOSS = 165374.0625
TRAIN_RECON_MEAN_PER_SAMPLE = 165314.0625
TRAIN_KL_MEAN_PER_SAMPLE = 59.99898910522461

GRADIENT_TENSORS = 28
NONZERO_GRADIENT_TENSORS = 28
ALL_GRADIENTS_PRESENT = True
ALL_GRADIENTS_FINITE = True
KEY_GRADIENTS_NONZERO = True
INPUT_GRADIENT_PRESENT = True
INPUT_GRADIENT_FINITE = True
INPUT_GRADIENT_NONZERO = True

BN_COUNTERS_INCREMENTED = True
BN_RUNNING_STATE_CHANGED = True

PARAMETERS_UNCHANGED_BY_BACKWARD = True
MAIN_FROZEN_MODEL_UNCHANGED = True
PARAMETER_SHA256_AFTER_BACKWARD = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b

PEAK_CUDA_ALLOCATED_MIB = 311.41

BACKWARD_EXECUTED = True
OPTIMIZER_EXECUTED = False

SAKAI224_FULL_BACKWARD_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — ADAM OPTIMIZER CONTRACT
# STEP 02-J1
#
# PAPER:
#   Adam
#   learning rate = 5e-4
#   beta1 = 0.9
#   beta2 = 0.999
#   epsilon = 1e-8
#
# PROJECT:
#   standard torch.optim.Adam
#   no weight decay
#   no AMSGrad
#
# This step DOES NOT touch the VAE.
# ============================================================

import math
import numpy as np
import torch


PAPER_LEARNING_RATE = 5.0e-4
PAPER_ADAM_BETA1 = 0.9
PAPER_ADAM_BETA2 = 0.999
PAPER_ADAM_EPSILON = 1.0e-8


# ------------------------------------------------------------
# TINY INDEPENDENT PARAMETER
# ------------------------------------------------------------

theta_initial = np.array(
    [
        0.50,
        -1.25,
        2.00,
        -0.75,
    ],
    dtype=np.float32,
)

gradient_sequence = [
    np.array(
        [0.20, -0.40, 0.10, 0.30],
        dtype=np.float32,
    ),

    np.array(
        [-0.10, 0.25, 0.50, -0.20],
        dtype=np.float32,
    ),

    np.array(
        [0.05, -0.15, -0.30, 0.40],
        dtype=np.float32,
    ),
]


theta_torch = torch.tensor(
    theta_initial.copy(),
    dtype=torch.float32,
    requires_grad=True,
)


optimizer_test = torch.optim.Adam(
    [theta_torch],

    lr=PAPER_LEARNING_RATE,

    betas=(
        PAPER_ADAM_BETA1,
        PAPER_ADAM_BETA2,
    ),

    eps=PAPER_ADAM_EPSILON,

    weight_decay=0.0,

    amsgrad=False,

    foreach=False,

    fused=False,
)


# ------------------------------------------------------------
# MANUAL C-LIKE REFERENCE
#
# Same Adam equations as current C implementation.
# Moments are rounded/stored as float32 after every step.
# ------------------------------------------------------------

theta_ref = theta_initial.copy()

m_ref = np.zeros_like(
    theta_ref,
    dtype=np.float32,
)

v_ref = np.zeros_like(
    theta_ref,
    dtype=np.float32,
)

beta1_power = 1.0
beta2_power = 1.0


step_results = []


for step_index, gradient_np in enumerate(
    gradient_sequence,
    start=1,
):

    # ----------------------------------------
    # C-like manual reference
    # ----------------------------------------

    beta1_power *= PAPER_ADAM_BETA1
    beta2_power *= PAPER_ADAM_BETA2

    bias_correction1 = (
        1.0 - beta1_power
    )

    bias_correction2 = (
        1.0 - beta2_power
    )


    for index in range(
        theta_ref.size
    ):

        gradient = float(
            gradient_np[index]
        )

        m_candidate = (
            PAPER_ADAM_BETA1
            * float(m_ref[index])
            +
            (1.0 - PAPER_ADAM_BETA1)
            * gradient
        )

        v_candidate = (
            PAPER_ADAM_BETA2
            * float(v_ref[index])
            +
            (1.0 - PAPER_ADAM_BETA2)
            * gradient
            * gradient
        )

        # C stores moments as float32.
        m_ref[index] = np.float32(
            m_candidate
        )

        v_ref[index] = np.float32(
            v_candidate
        )

        m_hat = (
            float(m_ref[index])
            / bias_correction1
        )

        v_hat = (
            float(v_ref[index])
            / bias_correction2
        )

        update = (
            PAPER_LEARNING_RATE
            * m_hat
            /
            (
                math.sqrt(v_hat)
                + PAPER_ADAM_EPSILON
            )
        )

        theta_ref[index] = np.float32(
            float(theta_ref[index])
            - update
        )


    # ----------------------------------------
    # PyTorch Adam
    # ----------------------------------------

    optimizer_test.zero_grad(
        set_to_none=True
    )

    theta_torch.grad = torch.tensor(
        gradient_np,
        dtype=torch.float32,
    )

    optimizer_test.step()


    # ----------------------------------------
    # COMPARE
    # ----------------------------------------

    state = optimizer_test.state[
        theta_torch
    ]

    theta_np = (
        theta_torch
        .detach()
        .cpu()
        .numpy()
    )

    m_torch = (
        state["exp_avg"]
        .detach()
        .cpu()
        .numpy()
    )

    v_torch = (
        state["exp_avg_sq"]
        .detach()
        .cpu()
        .numpy()
    )


    theta_pass = np.allclose(
        theta_np,
        theta_ref,
        rtol=2e-6,
        atol=2e-7,
    )

    m_pass = np.allclose(
        m_torch,
        m_ref,
        rtol=2e-6,
        atol=2e-7,
    )

    v_pass = np.allclose(
        v_torch,
        v_ref,
        rtol=2e-6,
        atol=2e-7,
    )


    step_pass = (
        theta_pass
        and m_pass
        and v_pass
    )


    step_results.append(
        step_pass
    )


    print(
        "STEP",
        step_index,
        "THETA_PASS =",
        theta_pass,
        "M_PASS =",
        m_pass,
        "V_PASS =",
        v_pass,
        "TOTAL_PASS =",
        step_pass,
    )


# ------------------------------------------------------------
# OPTIMIZER CONFIG AUDIT
# ------------------------------------------------------------

group = optimizer_test.param_groups[0]


config_pass = (
    group["lr"]
        == PAPER_LEARNING_RATE

    and group["betas"]
        == (
            PAPER_ADAM_BETA1,
            PAPER_ADAM_BETA2,
        )

    and group["eps"]
        == PAPER_ADAM_EPSILON

    and group["weight_decay"]
        == 0.0

    and group["amsgrad"]
        is False
)


assert config_pass
assert all(step_results)


# ------------------------------------------------------------
# MAIN VAE MUST STILL BE UNTOUCHED
# ------------------------------------------------------------

main_model_unchanged = (
    project_parameter_sha256(
        model
    )
    == FROZEN_INIT_SHA256
)

assert main_model_unchanged


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print()
print(
    "=== ADAM CONTRACT AUDIT ==="
)

print(
    "OPTIMIZER =",
    "torch.optim.Adam",
)

print(
    "LEARNING_RATE =",
    group["lr"],
)

print(
    "BETAS =",
    group["betas"],
)

print(
    "EPSILON =",
    group["eps"],
)

print(
    "WEIGHT_DECAY =",
    group["weight_decay"],
)

print(
    "AMSGRAD =",
    group["amsgrad"],
)

print()

print(
    "THREE_STEP_C_SEMANTIC_PARITY =",
    all(step_results),
)

print(
    "BITWISE_PARITY_REQUIRED =",
    False,
)

print(
    "MAIN_MODEL_PARAMETERS_UNCHANGED =",
    main_model_unchanged,
)

print()

print(
    "VAE_OPTIMIZER_STEP_EXECUTED =",
    False,
)

print()

print(
    "SAKAI224_FULL_ADAM_CONTRACT_PASS =",
    (
        config_pass
        and all(step_results)
        and main_model_unchanged
    ),
)

STEP 1 THETA_PASS = True M_PASS = True V_PASS = True TOTAL_PASS = True
STEP 2 THETA_PASS = True M_PASS = True V_PASS = True TOTAL_PASS = True
STEP 3 THETA_PASS = True M_PASS = True V_PASS = True TOTAL_PASS = True

=== ADAM CONTRACT AUDIT ===
OPTIMIZER = torch.optim.Adam
LEARNING_RATE = 0.0005
BETAS = (0.9, 0.999)
EPSILON = 1e-08
WEIGHT_DECAY = 0.0
AMSGRAD = False

THREE_STEP_C_SEMANTIC_PARITY = True
BITWISE_PARITY_REQUIRED = False
MAIN_MODEL_PARAMETERS_UNCHANGED = True

VAE_OPTIMIZER_STEP_EXECUTED = False

SAKAI224_FULL_ADAM_CONTRACT_PASS = True


In [ ]:
# ============================================================
# NOTEBOOK 02 — ONE COMPLETE TRAINING STEP
# STEP 02-J2
#
# Full chain:
#   STOCHASTIC+TRAIN
#   -> VAE loss
#   -> backward
#   -> Adam step
#
# Separate model.
# Main frozen model must remain untouched.
# ============================================================

import math
import numpy as np
import torch


# ------------------------------------------------------------
# FRESH TRAINING MODEL
# ------------------------------------------------------------

training_model = Sakai224FullVAE(
    SPEC
)

initialize_project_c_exact_v1(
    training_model,
    PROJECT_INITIALIZATION_SEED,
)

hash_training_initial = (
    project_parameter_sha256(
        training_model
    )
)

assert (
    hash_training_initial
    == FROZEN_INIT_SHA256
)

training_model = training_model.to(
    device=device,
    dtype=torch.float32,
)

training_model.train()


# ------------------------------------------------------------
# PAPER ADAM CONFIGURATION
# ------------------------------------------------------------

training_optimizer = torch.optim.Adam(
    training_model.parameters(),

    lr=PAPER_LEARNING_RATE,

    betas=(
        PAPER_ADAM_BETA1,
        PAPER_ADAM_BETA2,
    ),

    eps=PAPER_ADAM_EPSILON,

    weight_decay=0.0,

    amsgrad=False,

    foreach=False,

    fused=False,
)


# ------------------------------------------------------------
# DETERMINISTIC SYNTHETIC TRAIN BATCH
# ------------------------------------------------------------

x_train_step = (
    x_smoke
    .detach()
    .clone()
)

TRAIN_STEP_MANIFEST_INDICES = [
    7663,
    2606,
]

TRAIN_STEP_EPOCH = 0


epsilon_step_np = np.stack(
    [
        python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_TRAIN,
            TRAIN_STEP_EPOCH,
            manifest_index,
            SPEC.latent_dim,
        )
        for manifest_index
        in TRAIN_STEP_MANIFEST_INDICES
    ],
    axis=0,
)

epsilon_step = torch.from_numpy(
    epsilon_step_np
).to(
    device=device,
    dtype=torch.float32,
)


assert epsilon_step.shape == (
    2,
    512,
)


# ------------------------------------------------------------
# SNAPSHOTS BEFORE STEP
# ------------------------------------------------------------

parameters_before = {
    name:
        parameter
        .detach()
        .clone()

    for name, parameter
    in training_model.named_parameters()
}


training_bn_layers = (
    get_project_batchnorm_layers(
        training_model
    )
)

bn_before_step = snapshot_bn_state(
    training_bn_layers
)


# ------------------------------------------------------------
# ZERO GRAD
# ------------------------------------------------------------

training_optimizer.zero_grad(
    set_to_none=True
)


# ------------------------------------------------------------
# STOCHASTIC + TRAIN FORWARD
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


(
    logits_step,
    mu_step,
    logvar_step,
    z_step,
) = training_model(
    x_train_step,
    epsilon=epsilon_step,
    mean_mode=False,
)


loss_step_stats = sakai_vae_loss(
    logits_step,
    x_train_step,
    mu_step,
    logvar_step,
    beta_kl=SPEC.beta_kl,
)

loss_before_update = (
    loss_step_stats["loss"]
)

assert torch.isfinite(
    loss_before_update
).item()


# ------------------------------------------------------------
# BACKWARD
# ------------------------------------------------------------

loss_before_update.backward()


all_gradients_finite = all(
    parameter.grad is not None
    and torch.isfinite(
        parameter.grad
    ).all().item()

    for parameter
    in training_model.parameters()
)

assert all_gradients_finite


# ------------------------------------------------------------
# ADAM UPDATE
# ------------------------------------------------------------

training_optimizer.step()

torch.cuda.synchronize()


# ------------------------------------------------------------
# PARAMETERS MUST HAVE CHANGED
# ------------------------------------------------------------

changed_parameter_names = []

for name, parameter in (
    training_model.named_parameters()
):

    if not torch.equal(
        parameter.detach(),
        parameters_before[name],
    ):
        changed_parameter_names.append(
            name
        )


changed_parameter_tensor_count = len(
    changed_parameter_names
)


key_parameters_changed = all(
    name in changed_parameter_names

    for name in [
        "encoder.conv1.weight",
        "encoder.fc.weight",
        "decoder.fc.weight",
        "decoder.out_conv.weight",
    ]
)


assert changed_parameter_tensor_count > 0
assert key_parameters_changed


# ------------------------------------------------------------
# NEW PARAMETER HASH
# ------------------------------------------------------------

hash_after_optimizer = (
    project_parameter_sha256(
        training_model
    )
)

optimizer_changed_parameters = (
    hash_after_optimizer
    != FROZEN_INIT_SHA256
)

assert optimizer_changed_parameters


# ------------------------------------------------------------
# ADAM STATE AUDIT
# ------------------------------------------------------------

optimizer_state_count = len(
    training_optimizer.state
)

optimizer_states_finite = True
optimizer_steps_are_one = True


for parameter in (
    training_model.parameters()
):

    state = training_optimizer.state[
        parameter
    ]

    if not torch.isfinite(
        state["exp_avg"]
    ).all().item():

        optimizer_states_finite = False

    if not torch.isfinite(
        state["exp_avg_sq"]
    ).all().item():

        optimizer_states_finite = False

    step_value = int(
        state["step"].item()
    )

    if step_value != 1:
        optimizer_steps_are_one = False


assert optimizer_state_count == 28
assert optimizer_states_finite
assert optimizer_steps_are_one


# ------------------------------------------------------------
# BATCHNORM TRAIN STATE
# ------------------------------------------------------------

bn_after_step = snapshot_bn_state(
    training_bn_layers
)

bn_counters_are_one = True
bn_running_state_changed = False


for before, after in zip(
    bn_before_step,
    bn_after_step,
):

    before_mean, before_var, before_count = before
    after_mean, after_var, after_count = after

    if int(after_count.item()) != 1:
        bn_counters_are_one = False

    if (
        not torch.equal(
            before_mean,
            after_mean,
        )
        or
        not torch.equal(
            before_var,
            after_var,
        )
    ):
        bn_running_state_changed = True


assert bn_counters_are_one
assert bn_running_state_changed


# ------------------------------------------------------------
# ALL UPDATED PARAMETERS MUST BE FINITE
# ------------------------------------------------------------

all_updated_parameters_finite = all(
    torch.isfinite(
        parameter
    ).all().item()

    for parameter
    in training_model.parameters()
)

assert all_updated_parameters_finite


# ------------------------------------------------------------
# MAIN FROZEN MODEL MUST REMAIN IDENTICAL
# ------------------------------------------------------------

main_model_hash_after_training_audit = (
    project_parameter_sha256(
        model
    )
)

main_model_unchanged = (
    main_model_hash_after_training_audit
    == FROZEN_INIT_SHA256
)

assert main_model_unchanged


# ------------------------------------------------------------
# MEMORY DIAGNOSTIC
# ------------------------------------------------------------

peak_training_step_cuda_mib = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 2)
)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print(
    "=== ONE COMPLETE TRAINING STEP AUDIT ==="
)

print(
    "FORWARD_MODE =",
    "STOCHASTIC+TRAIN",
)

print(
    "TRAIN_MANIFEST_INDICES =",
    TRAIN_STEP_MANIFEST_INDICES,
)

print(
    "TRAIN_EPOCH =",
    TRAIN_STEP_EPOCH,
)

print()

print(
    "LOSS_BEFORE_UPDATE =",
    float(
        loss_before_update.detach()
    ),
)

print(
    "RECON_MEAN_PER_SAMPLE =",
    float(
        loss_step_stats[
            "reconstruction_mean_per_sample"
        ].detach()
    ),
)

print(
    "KL_MEAN_PER_SAMPLE =",
    float(
        loss_step_stats[
            "kl_mean_per_sample"
        ].detach()
    ),
)

print()

print(
    "ALL_GRADIENTS_FINITE =",
    all_gradients_finite,
)

print(
    "CHANGED_PARAMETER_TENSORS =",
    changed_parameter_tensor_count,
)

print(
    "KEY_PARAMETERS_CHANGED =",
    key_parameters_changed,
)

print()

print(
    "INITIAL_PARAMETER_SHA256 =",
    FROZEN_INIT_SHA256,
)

print(
    "POST_ADAM_PARAMETER_SHA256 =",
    hash_after_optimizer,
)

print(
    "OPTIMIZER_CHANGED_PARAMETERS =",
    optimizer_changed_parameters,
)

print()

print(
    "ADAM_STATE_TENSORS =",
    optimizer_state_count,
)

print(
    "ADAM_STATES_FINITE =",
    optimizer_states_finite,
)

print(
    "ADAM_STEPS_ARE_ONE =",
    optimizer_steps_are_one,
)

print()

print(
    "BN_COUNTERS_ARE_ONE =",
    bn_counters_are_one,
)

print(
    "BN_RUNNING_STATE_CHANGED =",
    bn_running_state_changed,
)

print(
    "ALL_UPDATED_PARAMETERS_FINITE =",
    all_updated_parameters_finite,
)

print()

print(
    "MAIN_FROZEN_MODEL_UNCHANGED =",
    main_model_unchanged,
)

print(
    "MAIN_FROZEN_MODEL_SHA256 =",
    main_model_hash_after_training_audit,
)

print()

print(
    "PEAK_CUDA_ALLOCATED_MIB =",
    round(
        peak_training_step_cuda_mib,
        3,
    ),
)

print()

print(
    "BACKWARD_EXECUTED =",
    True,
)

print(
    "OPTIMIZER_EXECUTED =",
    True,
)

print(
    "OPTIMIZER_STEP =",
    1,
)

print()

print(
    "SAKAI224_FULL_ONE_TRAIN_STEP_PASS =",
    (
        all_gradients_finite
        and changed_parameter_tensor_count > 0
        and key_parameters_changed
        and optimizer_changed_parameters
        and optimizer_state_count == 28
        and optimizer_states_finite
        and optimizer_steps_are_one
        and bn_counters_are_one
        and bn_running_state_changed
        and all_updated_parameters_finite
        and main_model_unchanged
    ),
)

=== ONE COMPLETE TRAINING STEP AUDIT ===
FORWARD_MODE = STOCHASTIC+TRAIN
TRAIN_MANIFEST_INDICES = [7663, 2606]
TRAIN_EPOCH = 0

LOSS_BEFORE_UPDATE = 165374.0625
RECON_MEAN_PER_SAMPLE = 165314.0625
KL_MEAN_PER_SAMPLE = 59.99898910522461

ALL_GRADIENTS_FINITE = True
CHANGED_PARAMETER_TENSORS = 28
KEY_PARAMETERS_CHANGED = True

INITIAL_PARAMETER_SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b
POST_ADAM_PARAMETER_SHA256 = 082283d7796e3a1641e21525ffd21be14ad8ec0e5d92a110bb9e9312415f216a
OPTIMIZER_CHANGED_PARAMETERS = True

ADAM_STATE_TENSORS = 28
ADAM_STATES_FINITE = True
ADAM_STEPS_ARE_ONE = True

BN_COUNTERS_ARE_ONE = True
BN_RUNNING_STATE_CHANGED = True
ALL_UPDATED_PARAMETERS_FINITE = True

MAIN_FROZEN_MODEL_UNCHANGED = True
MAIN_FROZEN_MODEL_SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b

PEAK_CUDA_ALLOCATED_MIB = 652.148

BACKWARD_EXECUTED = True
OPTIMIZER_EXECUTED = True
OPTIMIZER_STEP = 1

SAKAI224_FULL_ONE_TRAIN_STEP_PASS = True


## Stato di validazione del modello `sakai224_full`

Questa sezione conclude i test isolati del motore neurale prima del collegamento ai dati Gravity Spy reali.

### Contratti già verificati

- **Architettura**: input `4×224×224`, encoder `64→128→256→512`, latent dimension `512`, decoder simmetrico Sakai-like.
- **Inizializzazione**: policy PROJECT C deterministica con seed `123`, verificata contro le primitive C reali.
- **Forward deterministico**: modalità `MEAN+EVAL`, con `z = μ`.
- **Loss VAE**: Bernoulli tramite BCE-with-logits + KL gaussiana, somma sulle dimensioni e media soltanto sul batch.
- **RNG latente**: epsilon stateless PROJECT, verificato Python ↔ C.
- **Forward stocastico**: epsilon esplicito esterno al modello.
- **Backward**: gradienti finiti e non nulli su tutti i 28 tensori parametro.
- **Optimizer**: Adam con i parametri pubblicati da Sakai (`lr=5e-4`, `β1=0.9`, `β2=0.999`, `eps=1e-8`).
- **Training step**: catena completa `forward → loss → backward → Adam` verificata su un modello di audit separato.

### Fingerprint iniziale PROJECT

`1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b`

Il modello principale rimane nello stato iniziale congelato. I test di backward e optimizer sono stati eseguiti su modelli separati.

### Integrazione con i dati Gravity Spy

L'integrazione del modello con i dati Gravity Spy reali usa **esattamente il contratto dati già validato nel Notebook 01**, con smoke test piccoli che non avviano il training ufficiale.


In [ ]:
# ============================================================
# NOTEBOOK 02 — REAL DATA RUNTIME PREFLIGHT
# STEP 02-K0
#
# Purpose:
#   verify whether the already validated Gravity Spy data
#   extracted under LOCAL_RUNTIME_ROOT survived the current runtime.
#
# READ-ONLY:
#   no extraction
#   no modification
#   no model execution
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# LOCAL RUNTIME PATHS
#
# These paths use temporary runtime storage.
# They disappear when the runtime is reset/disconnected.
# ------------------------------------------------------------

LOCAL_DATA_ROOT = (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_data"
)

LOCAL_OUT272_DIR = (
    LOCAL_DATA_ROOT
    / "out272_gray"
)

LOCAL_CENTRAL224_DIR = (
    LOCAL_DATA_ROOT
    / "tensors_224_central_uint8"
)


# ------------------------------------------------------------
# PERSISTENT PROJECT SOURCE FILES
#
# TRANSFER_V1 was defined in the Notebook 02 preflight.
# These are the authoritative transfer artifacts.
# ------------------------------------------------------------

OUT272_TAR = (
    TRANSFER_V1
    / "sakai224_full_out272_gray.tar"
)

CENTRAL224_TAR = (
    TRANSFER_V1
    / "sakai224_full_central224.tar"
)

EVENT_MANIFEST = (
    TRANSFER_V1
    / "manifest_events_272.csv"
)

TENSOR_MANIFEST = (
    TRANSFER_V1
    / "manifest_tensors_224_central.csv"
)

TRAIN_SPLIT = (
    TRANSFER_V1
    / "split_train_indices_seed123.csv"
)

VALIDATION_SPLIT = (
    TRANSFER_V1
    / "split_val_indices_seed123.csv"
)


# ------------------------------------------------------------
# DRIVE SOURCE GATE
# ------------------------------------------------------------

drive_sources = {
    "OUT272_TAR":
        OUT272_TAR,

    "CENTRAL224_TAR":
        CENTRAL224_TAR,

    "EVENT_MANIFEST":
        EVENT_MANIFEST,

    "TENSOR_MANIFEST":
        TENSOR_MANIFEST,

    "TRAIN_SPLIT":
        TRAIN_SPLIT,

    "VALIDATION_SPLIT":
        VALIDATION_SPLIT,
}


drive_sources_ok = True

print(
    "=== PERSISTENT DRIVE DATA SOURCES ==="
)

for name, path in drive_sources.items():

    exists = path.is_file()

    print(
        f"{name:22s} =",
        exists,
    )

    drive_sources_ok &= exists


assert drive_sources_ok


# ------------------------------------------------------------
# LOCAL RUNTIME CHECK
# ------------------------------------------------------------

local_root_exists = (
    LOCAL_DATA_ROOT.is_dir()
)

local_out272_exists = (
    LOCAL_OUT272_DIR.is_dir()
)

local_central224_exists = (
    LOCAL_CENTRAL224_DIR.is_dir()
)


local_data_ready = (
    local_root_exists
    and local_out272_exists
    and local_central224_exists
)


print()
print(
    "=== LOCAL COLAB DATA STATE ==="
)

print(
    "LOCAL_DATA_ROOT =",
    LOCAL_DATA_ROOT,
)

print(
    "LOCAL_DATA_ROOT_EXISTS =",
    local_root_exists,
)

print(
    "LOCAL_OUT272_EXISTS =",
    local_out272_exists,
)

print(
    "LOCAL_CENTRAL224_EXISTS =",
    local_central224_exists,
)

print()

print(
    "DRIVE_SOURCES_OK =",
    drive_sources_ok,
)

print(
    "REAL_DATA_LOCAL_READY =",
    local_data_ready,
)

print()

if local_data_ready:

    print(
        "NEXT_ACTION = USE_EXISTING_LOCAL_DATA"
    )

else:

    print(
        "NEXT_ACTION = RESTORE_FROM_VERIFIED_TRANSFER_V1"
    )

print()

print(
    "MODEL_EXECUTED =",
    False,
)

print(
    "FILES_MODIFIED =",
    False,
)

print(
    "NOTEBOOK_02_REAL_DATA_PREFLIGHT_PASS =",
    drive_sources_ok,
)

=== PERSISTENT DRIVE DATA SOURCES ===
OUT272_TAR             = True
CENTRAL224_TAR         = True
EVENT_MANIFEST         = True
TENSOR_MANIFEST        = True
TRAIN_SPLIT            = True
VALIDATION_SPLIT       = True

=== LOCAL COLAB DATA STATE ===
LOCAL_DATA_ROOT = /content/sakai224_full_data
LOCAL_DATA_ROOT_EXISTS = False
LOCAL_OUT272_EXISTS = False
LOCAL_CENTRAL224_EXISTS = False

DRIVE_SOURCES_OK = True
REAL_DATA_LOCAL_READY = False

NEXT_ACTION = RESTORE_FROM_VERIFIED_TRANSFER_V1

MODEL_EXECUTED = False
FILES_MODIFIED = False
NOTEBOOK_02_REAL_DATA_PREFLIGHT_PASS = True


## STEP 02-K1 — Ripristino di `transfer_v1` nello storage temporaneo del runtime

Gli artefatti persistenti di `transfer_v1` vengono copiati nello storage temporaneo del runtime e verificati prima dell'uso.

La procedura crea una **copia temporanea** dei sette artefatti congelati di `transfer_v1`.

La copia non costituisce preprocessing e non modifica i dati scientifici:

- **sorgente persistente:** `TRANSFER_V1` sotto `PROJECT_ROOT`;
- **destinazione temporanea:** `LOCAL_TRANSFER_DIR` sotto `LOCAL_RUNTIME_ROOT`;
- **controllo di integrità:** SHA-256 confrontato con i digest congelati;
- **nessuna estrazione TAR durante questo ripristino**;
- **nessuna modifica al modello**;
- **nessuna modifica allo storage persistente del progetto**.

Se un file temporaneo esiste già, viene riutilizzato soltanto se il suo SHA-256 è corretto; un file esistente ma non conforme provoca uno stop.

In [ ]:
# ============================================================
# NOTEBOOK 02 — RESTORE VERIFIED LOCAL TRANSFER CACHE
# STEP 02-K1
#
# Rebuild the temporary runtime copy of transfer_v1.
#
# This step:
#   - reads the frozen files from persistent project storage;
#   - copies them to temporary runtime storage;
#   - verifies every local SHA-256;
#   - does NOT extract the archives;
#   - does NOT execute the neural model;
#   - does NOT modify persistent project storage.
# ============================================================

from pathlib import Path
import hashlib
import shutil


# ------------------------------------------------------------
# TEMPORARY RUNTIME CACHE
# ------------------------------------------------------------

LOCAL_TRANSFER_DIR = (
    LOCAL_RUNTIME_ROOT
    / "sakai224_full_transfer_v1"
)


# ------------------------------------------------------------
# FROZEN SHA-256 CONTRACT
#
# These are the same values already verified in Notebook 01.
# They identify the exact transfer_v1 artifacts that belong
# to the sakai224_full data contract.
# ------------------------------------------------------------

EXPECTED_TRANSFER_SHA256 = {
    "sakai224_full_out272_gray.tar":
        "70c766c3652f586c8da4c8cadf134a8ce83cfd07c8cd10545367f380017ddab2",

    "sakai224_full_central224.tar":
        "0fad37cdf4ea056e2c3abfa2d1be73b9d9bd55e644d54a58013950697cf4d4b6",

    "manifest_events_272.csv":
        "620960e6b35da31bc4cf1a8a6da8679027d378e1d4b2ac73c5267c1c5311ae82",

    "manifest_tensors_224_central.csv":
        "94b7e2e051bd32de72dd2cb6a5fb7eda9bcaec123cbf3a07d0d901bd19779afa",

    "split_train_indices_seed123.csv":
        "8c7a513b98dbe996261bc80965500ab15ab77f498ec9440858003152e42f926f",

    "split_val_indices_seed123.csv":
        "e63109c88df7b3ee97317f94791f26df3fcaf8b937921cfb27967d01c2546467",

    "transfer_manifest_sha256.txt":
        "59a04c1098ec4465c1a9864e7bc61181c7981e316c8bc78adbf4c18b78da962f",
}


# ------------------------------------------------------------
# SOURCE FILES IN PERSISTENT PROJECT STORAGE
# ------------------------------------------------------------

TRANSFER_SOURCE_FILES = {
    name:
        TRANSFER_V1 / name

    for name
    in EXPECTED_TRANSFER_SHA256
}


# All persistent sources must exist before touching runtime storage.
for name, source in TRANSFER_SOURCE_FILES.items():
    assert source.is_file(), (
        f"Missing persistent source: {name}"
    )


# ------------------------------------------------------------
# SHA-256 HELPER
#
# The file is read in chunks so that the multi-GB TAR archives
# are never loaded entirely into RAM.
# ------------------------------------------------------------

def sha256_file_streaming(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with path.open("rb") as handle:

        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


# ------------------------------------------------------------
# DISK-SPACE PREFLIGHT
#
# We reserve enough space for:
#   1. any transfer files still missing locally;
#   2. the future extracted dataset;
#   3. a 512 MiB safety margin.
#
# Frozen extracted payload:
#   out272_gray                 = 1,144,867,540 bytes
#   tensors_224_central_uint8  = 1,722,642,432 bytes
# ------------------------------------------------------------

EXPECTED_EXTRACTED_BYTES = (
    1_144_867_540
    + 1_722_642_432
)

SAFETY_MARGIN_BYTES = (
    512 * 1024 * 1024
)


missing_copy_bytes = 0

for name, source in (
    TRANSFER_SOURCE_FILES.items()
):
    destination = (
        LOCAL_TRANSFER_DIR
        / name
    )

    if not destination.exists():
        missing_copy_bytes += (
            source.stat().st_size
        )


runtime_disk_probe = existing_disk_probe(
    LOCAL_RUNTIME_ROOT
)

free_bytes_before = (
    shutil.disk_usage(
        runtime_disk_probe
    ).free
)

required_bytes = (
    missing_copy_bytes
    + EXPECTED_EXTRACTED_BYTES
    + SAFETY_MARGIN_BYTES
)


disk_space_ok = (
    free_bytes_before
    >= required_bytes
)


print(
    "=== LOCAL STORAGE PREFLIGHT ==="
)

print(
    "FREE_GIB =",
    round(
        free_bytes_before
        / (1024 ** 3),
        3,
    ),
)

print(
    "REQUIRED_GIB =",
    round(
        required_bytes
        / (1024 ** 3),
        3,
    ),
)

print(
    "DISK_SPACE_OK =",
    disk_space_ok,
)


assert disk_space_ok


# ------------------------------------------------------------
# CREATE LOCAL CACHE DIRECTORY
# ------------------------------------------------------------

LOCAL_TRANSFER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Reject unexpected pre-existing files rather than silently
# mixing another transfer with the frozen transfer_v1.
unexpected_files = sorted(
    path.name
    for path
    in LOCAL_TRANSFER_DIR.iterdir()

    if (
        path.is_file()
        and path.name
        not in EXPECTED_TRANSFER_SHA256
    )
)

assert not unexpected_files, (
    "Unexpected files in local transfer cache: "
    + repr(unexpected_files)
)


# ------------------------------------------------------------
# COPY / REUSE + VERIFY
# ------------------------------------------------------------

local_sha_all_ok = True


print()
print(
    "=== RESTORE VERIFIED TRANSFER_V1 ==="
)


for name, expected_sha in (
    EXPECTED_TRANSFER_SHA256.items()
):

    source = (
        TRANSFER_SOURCE_FILES[
            name
        ]
    )

    destination = (
        LOCAL_TRANSFER_DIR
        / name
    )


    # If a file already exists, never overwrite it silently.
    # It can be reused only when it is already byte-correct.
    if destination.exists():

        existing_sha = (
            sha256_file_streaming(
                destination
            )
        )

        if existing_sha != expected_sha:
            raise RuntimeError(
                "STOP: existing local file "
                f"has wrong SHA-256: {name}"
            )

        action = (
            "REUSED_VERIFIED"
        )

    else:

        shutil.copy2(
            source,
            destination,
        )

        action = "COPIED"


    # Verify the final local byte stream independently.
    actual_sha = (
        sha256_file_streaming(
            destination
        )
    )

    file_ok = (
        actual_sha
        == expected_sha
    )

    local_sha_all_ok &= (
        file_ok
    )

    print(
        f"{name:42s}",
        action,
        "SHA256_OK =",
        file_ok,
    )


# ------------------------------------------------------------
# EXACT LOCAL FILE SET
# ------------------------------------------------------------

local_files = sorted(
    path.name
    for path
    in LOCAL_TRANSFER_DIR.iterdir()
    if path.is_file()
)

expected_files = sorted(
    EXPECTED_TRANSFER_SHA256.keys()
)

exact_file_set = (
    local_files
    == expected_files
)


assert local_sha_all_ok
assert exact_file_set


# ------------------------------------------------------------
# MODEL INVARIANCE
#
# Restoring data files must have no effect on the frozen
# neural-model parameters.
# ------------------------------------------------------------

main_model_still_frozen = (
    project_parameter_sha256(
        model
    )
    == FROZEN_INIT_SHA256
)

assert main_model_still_frozen


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print()
print(
    "=== LOCAL TRANSFER RESTORE AUDIT ==="
)

print(
    "LOCAL_TRANSFER_DIR =",
    LOCAL_TRANSFER_DIR,
)

print(
    "LOCAL_FILE_COUNT =",
    len(local_files),
)

print(
    "EXPECTED_FILE_COUNT =",
    len(expected_files),
)

print(
    "EXACT_FILE_SET =",
    exact_file_set,
)

print(
    "LOCAL_SHA256_ALL_OK =",
    local_sha_all_ok,
)

print()

print(
    "MAIN_FROZEN_MODEL_UNCHANGED =",
    main_model_still_frozen,
)

print(
    "ARCHIVES_EXTRACTED =",
    False,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print()

print(
    "NOTEBOOK_02_LOCAL_TRANSFER_RESTORE_PASS =",
    (
        disk_space_ok
        and exact_file_set
        and local_sha_all_ok
        and main_model_still_frozen
    ),
)

=== LOCAL STORAGE PREFLIGHT ===
FREE_GIB = 65.42
REQUIRED_GIB = 5.873
DISK_SPACE_OK = True

=== RESTORE VERIFIED TRANSFER_V1 ===
sakai224_full_out272_gray.tar              COPIED SHA256_OK = True
sakai224_full_central224.tar               COPIED SHA256_OK = True
manifest_events_272.csv                    COPIED SHA256_OK = True
manifest_tensors_224_central.csv           COPIED SHA256_OK = True
split_train_indices_seed123.csv            COPIED SHA256_OK = True
split_val_indices_seed123.csv              COPIED SHA256_OK = True
transfer_manifest_sha256.txt               COPIED SHA256_OK = True

=== LOCAL TRANSFER RESTORE AUDIT ===
LOCAL_TRANSFER_DIR = /content/sakai224_full_transfer_v1
LOCAL_FILE_COUNT = 7
EXPECTED_FILE_COUNT = 7
EXACT_FILE_SET = True
LOCAL_SHA256_ALL_OK = True

MAIN_FROZEN_MODEL_UNCHANGED = True
ARCHIVES_EXTRACTED = False
DRIVE_MODIFIED = False
MODEL_FORWARD_EXECUTED = False

NOTEBOOK_02_LOCAL_TRANSFER_RESTORE_PASS = True


## STEP 02-K2 — Ripristino dei dati Gravity Spy nello storage temporaneo del runtime

Gli archivi di `transfer_v1` presenti in `LOCAL_TRANSFER_DIR` vengono verificati tramite SHA-256.

In questa fase vengono ricostruite le due sorgenti locali già validate nel Notebook 01:

1. `out272_gray`: 34.332 immagini PNG utilizzate dal training con crop temporale dinamico;
2. `tensors_224_central_uint8`: 8.583 tensori centrali `4×224×224` utilizzati dalla validation deterministica.

L'estrazione non costituisce un nuovo preprocessing: i TAR contengono artefatti già congelati e verificati.  
Lo storage persistente del progetto, il manifest, lo split e il modello neurale non vengono modificati.

In [ ]:
# ============================================================
# NOTEBOOK 02 — RESTORE OUT272_GRAY
# STEP 02-K2A
#
# Reproduce the extraction already validated in Notebook 01.
#
# Expected frozen payload:
#   files = 34,332
#   bytes = 1,144,867,540
#
# Writes only to temporary runtime storage.
# ============================================================

from pathlib import Path
import subprocess


DATA_DIR = LOCAL_DATA_ROOT

OUT272_DIR = (
    DATA_DIR
    / "out272_gray"
)

OUT272_ARCHIVE = (
    LOCAL_TRANSFER_DIR
    / "sakai224_full_out272_gray.tar"
)


# ------------------------------------------------------------
# PREFLIGHT
# ------------------------------------------------------------

assert OUT272_ARCHIVE.is_file()

# We deliberately refuse to merge with an existing directory.
# A pre-existing extraction must be audited separately rather
# than silently overwritten.
if OUT272_DIR.exists():
    raise RuntimeError(
        f"STOP: extraction directory already exists: {OUT272_DIR}"
    )

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# EXTRACT
#
# Same tar extraction strategy used in Notebook 01.
# ------------------------------------------------------------

print(
    "Extracting:",
    OUT272_ARCHIVE.name,
)

subprocess.run(
    [
        "tar",
        "-xf",
        str(OUT272_ARCHIVE),
        "-C",
        str(DATA_DIR),
    ],
    check=True,
)


# ------------------------------------------------------------
# PAYLOAD AUDIT
# ------------------------------------------------------------

out272_files = [
    path
    for path in OUT272_DIR.rglob("*")
    if path.is_file()
]

out272_total_bytes = sum(
    path.stat().st_size
    for path in out272_files
)

out272_file_count_ok = (
    len(out272_files)
    == 34_332
)

out272_bytes_ok = (
    out272_total_bytes
    == 1_144_867_540
)


assert out272_file_count_ok
assert out272_bytes_ok


# ------------------------------------------------------------
# MODEL MUST REMAIN UNCHANGED
# ------------------------------------------------------------

main_model_unchanged = (
    project_parameter_sha256(model)
    == FROZEN_INIT_SHA256
)

assert main_model_unchanged


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print()
print(
    "=== OUT272_GRAY EXTRACTION AUDIT ==="
)

print(
    "DIRECTORY =",
    OUT272_DIR,
)

print(
    "FILE_COUNT =",
    len(out272_files),
)

print(
    "FILE_COUNT_OK =",
    out272_file_count_ok,
)

print(
    "TOTAL_BYTES =",
    out272_total_bytes,
)

print(
    "BYTES_OK =",
    out272_bytes_ok,
)

print()

print(
    "MAIN_FROZEN_MODEL_UNCHANGED =",
    main_model_unchanged,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print(
    "MODEL_EXECUTED =",
    False,
)

print()

print(
    "NOTEBOOK_02_OUT272_RESTORE_PASS =",
    (
        out272_file_count_ok
        and out272_bytes_ok
        and main_model_unchanged
    ),
)

Extracting: sakai224_full_out272_gray.tar

=== OUT272_GRAY EXTRACTION AUDIT ===
DIRECTORY = /content/sakai224_full_data/out272_gray
FILE_COUNT = 34332
FILE_COUNT_OK = True
TOTAL_BYTES = 1144867540
BYTES_OK = True

MAIN_FROZEN_MODEL_UNCHANGED = True
DRIVE_MODIFIED = False
MODEL_EXECUTED = False

NOTEBOOK_02_OUT272_RESTORE_PASS = True


### STEP 02-K2B — Ripristino dei tensori centrali 224×224

Questa fase ricostruisce sotto `LOCAL_RUNTIME_ROOT` i tensori centrali già congelati e validati per la validation.

Contratto atteso:

- **8.583 eventi**
- un file binario per evento;
- shape logica `4×224×224`;
- dtype sorgente `uint8`;
- crop centrale corrispondente a `crop_left = 24`;
- payload totale congelato: `1.722.642.432` byte.

L'operazione estrae esclusivamente un TAR già verificato tramite SHA-256.  
Non viene effettuato alcun nuovo preprocessing e non vengono modificati lo storage persistente del progetto, lo split, il manifest o i parametri del modello.


In [ ]:
# ============================================================
# NOTEBOOK 02 — RESTORE CENTRAL 224 TENSORS
# STEP 02-K2B
#
# Expected frozen payload:
#   files = 8,583
#   bytes = 1,722,642,432
#
# Temporary runtime extraction only.
# ============================================================

from pathlib import Path
import subprocess


CENTRAL224_DIR = (
    DATA_DIR
    / "tensors_224_central_uint8"
)

CENTRAL224_ARCHIVE = (
    LOCAL_TRANSFER_DIR
    / "sakai224_full_central224.tar"
)


# ------------------------------------------------------------
# PREFLIGHT
# ------------------------------------------------------------

assert CENTRAL224_ARCHIVE.is_file()

# Never silently merge an old or partial extraction.
if CENTRAL224_DIR.exists():
    raise RuntimeError(
        f"STOP: extraction directory already exists: {CENTRAL224_DIR}"
    )

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# EXTRACT VERIFIED ARCHIVE
# ------------------------------------------------------------

print(
    "Extracting:",
    CENTRAL224_ARCHIVE.name,
)

subprocess.run(
    [
        "tar",
        "-xf",
        str(CENTRAL224_ARCHIVE),
        "-C",
        str(DATA_DIR),
    ],
    check=True,
)


# ------------------------------------------------------------
# FROZEN PAYLOAD AUDIT
# ------------------------------------------------------------

central224_files = [
    path
    for path in CENTRAL224_DIR.rglob("*")
    if path.is_file()
]

central224_total_bytes = sum(
    path.stat().st_size
    for path in central224_files
)

central224_file_count_ok = (
    len(central224_files)
    == 8_583
)

central224_bytes_ok = (
    central224_total_bytes
    == 1_722_642_432
)


assert central224_file_count_ok
assert central224_bytes_ok


# ------------------------------------------------------------
# MODEL INVARIANCE
#
# Data restoration must not affect the frozen neural model.
# ------------------------------------------------------------

main_model_unchanged = (
    project_parameter_sha256(model)
    == FROZEN_INIT_SHA256
)

assert main_model_unchanged


# ------------------------------------------------------------
# COMBINED LOCAL DATA GATE
#
# Both training and validation sources must now exist.
# ------------------------------------------------------------

full_local_data_ready = (
    OUT272_DIR.is_dir()
    and CENTRAL224_DIR.is_dir()
)

assert full_local_data_ready


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print()
print(
    "=== CENTRAL224 EXTRACTION AUDIT ==="
)

print(
    "DIRECTORY =",
    CENTRAL224_DIR,
)

print(
    "FILE_COUNT =",
    len(central224_files),
)

print(
    "FILE_COUNT_OK =",
    central224_file_count_ok,
)

print(
    "TOTAL_BYTES =",
    central224_total_bytes,
)

print(
    "BYTES_OK =",
    central224_bytes_ok,
)

print()

print(
    "OUT272_PRESENT =",
    OUT272_DIR.is_dir(),
)

print(
    "CENTRAL224_PRESENT =",
    CENTRAL224_DIR.is_dir(),
)

print(
    "FULL_LOCAL_DATA_READY =",
    full_local_data_ready,
)

print()

print(
    "MAIN_FROZEN_MODEL_UNCHANGED =",
    main_model_unchanged,
)

print(
    "DRIVE_MODIFIED =",
    False,
)

print(
    "MODEL_EXECUTED =",
    False,
)

print()

print(
    "NOTEBOOK_02_CENTRAL224_RESTORE_PASS =",
    (
        central224_file_count_ok
        and central224_bytes_ok
        and full_local_data_ready
        and main_model_unchanged
    ),
)

Extracting: sakai224_full_central224.tar

=== CENTRAL224 EXTRACTION AUDIT ===
DIRECTORY = /content/sakai224_full_data/tensors_224_central_uint8
FILE_COUNT = 8583
FILE_COUNT_OK = True
TOTAL_BYTES = 1722642432
BYTES_OK = True

OUT272_PRESENT = True
CENTRAL224_PRESENT = True
FULL_LOCAL_DATA_READY = True

MAIN_FROZEN_MODEL_UNCHANGED = True
DRIVE_MODIFIED = False
MODEL_EXECUTED = False

NOTEBOOK_02_CENTRAL224_RESTORE_PASS = True


## STEP 02-K3 — Collegamento al contratto dati validato nel Notebook 01

Il modello `sakai224_full` viene collegato alle sorgenti Gravity Spy reali.

Questa sezione **non introduce un nuovo loader**: replica nel Notebook 02 il contratto già validato autonomamente nel Notebook 01.

### Training

Per ogni evento del full split train:

`manifest_index + epoch → crop stateless PROJECT → 4 PNG 224×272 → stesso crop sui quattro canali → float32 [0,1]`

Il crop è deterministico rispetto a:

- augmentation seed `789`;
- epoch;
- `manifest_index`;
- modalità PROJECT `symmetric_figure`, con `crop_left ∈ [0,48]`.

### Validation

Per ogni evento del full split validation:

`manifest_index → tensore centrale .bin → uint8 4×224×224 → float32 [0,1]`

La validation non usa augmentation e mantiene `crop_left = 24`.

La verifica ricostruisce e controlla esclusivamente i dataset e i loader CPU.  
Il VAE non viene eseguito e i suoi parametri devono restare invariati.

In [ ]:
# ============================================================
# NOTEBOOK 02 — REBUILD VALIDATED REAL-DATA LOADER CONTRACT
# STEP 02-K3
#
# Source of semantics:
#   Notebook 01 — already validated and closed.
#
# This cell reconstructs:
#   CSV metadata
#   PROJECT stateless train crop
#   train event loader
#   validation event loader
#   GravitySpy224Dataset
#
# No neural-model forward.
# ============================================================

import csv

from PIL import Image
import numpy as np
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
)


# ------------------------------------------------------------
# METADATA
#
# Use the verified local copy of transfer_v1.
# ------------------------------------------------------------

METADATA_DIR = LOCAL_TRANSFER_DIR


def read_csv_dicts(path):
    """
    Read one frozen project CSV as a list of dictionaries.

    Keeping the original row order is important because the
    split files define the canonical split-position ordering.
    """
    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as file:
        return list(
            csv.DictReader(file)
        )


event_rows = read_csv_dicts(
    METADATA_DIR
    / "manifest_events_272.csv"
)

tensor_rows = read_csv_dicts(
    METADATA_DIR
    / "manifest_tensors_224_central.csv"
)

train_rows = read_csv_dicts(
    METADATA_DIR
    / "split_train_indices_seed123.csv"
)

val_rows = read_csv_dicts(
    METADATA_DIR
    / "split_val_indices_seed123.csv"
)


# ------------------------------------------------------------
# DATA CONTRACT HARD GATE
# ------------------------------------------------------------

train_indices = {
    int(row["manifest_index"])
    for row in train_rows
}

val_indices = {
    int(row["manifest_index"])
    for row in val_rows
}


data_contract_checks = {
    "EVENT_ROWS_OK":
        len(event_rows) == 8583,

    "TENSOR_ROWS_OK":
        len(tensor_rows) == 8583,

    "TRAIN_ROWS_OK":
        len(train_rows) == 6866,

    "VAL_ROWS_OK":
        len(val_rows) == 1717,

    "TRAIN_UNIQUE_OK":
        len(train_indices) == 6866,

    "VAL_UNIQUE_OK":
        len(val_indices) == 1717,

    "OVERLAP_ZERO":
        len(
            train_indices
            & val_indices
        ) == 0,

    "FULL_COVERAGE":
        (
            train_indices
            | val_indices
        ) == set(range(8583)),
}


assert all(
    data_contract_checks.values()
)


# ============================================================
# PROJECT STATELESS TRAIN AUGMENTATION
#
# Exact semantics already validated in Notebook 01.
# ============================================================

MASK64 = (
    (1 << 64) - 1
)

GS_AUGMENT_MODE_LEGACY_TEXT = 0
GS_AUGMENT_MODE_SYMMETRIC_FIGURE = 1

GS_AUGMENT_STATELESS_V1_DOMAIN_TAG = (
    0x4753415547563121
)

AUGMENTATION_SEED = 789


def c_u64(value):
    """Apply C uint64_t wraparound semantics."""
    return value & MASK64


def gs_augment_stateless_mix64(value):
    """
    Stateless SplitMix-style mixer used by the PROJECT
    augmentation implementation.
    """
    value = c_u64(
        value
        + 0x9E3779B97F4A7C15
    )

    value = c_u64(
        (
            value
            ^ (value >> 30)
        )
        * 0xBF58476D1CE4E5B9
    )

    value = c_u64(
        (
            value
            ^ (value >> 27)
        )
        * 0x94D049BB133111EB
    )

    return c_u64(
        value
        ^ (value >> 31)
    )


def gs_augment_stateless_value(
    base_seed,
    epoch,
    manifest_index,
    mode,
    stream_index,
):
    """
    Produce one deterministic uint64 value from the complete
    event-level augmentation key.
    """
    h = gs_augment_stateless_mix64(
        GS_AUGMENT_STATELESS_V1_DOMAIN_TAG
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(base_seed)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(epoch)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(manifest_index)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(mode)
    )

    return gs_augment_stateless_mix64(
        h ^ c_u64(stream_index)
    )


def gs_augment_crop_left_for_event_mode(
    base_seed,
    epoch,
    manifest_index,
    mode,
):
    """
    Resolve crop_left with the same rejection-sampling
    semantics validated against the C implementation.

    Main sakai224_full training uses symmetric mode:
        crop_left = 0 ... 48
    """
    if mode == GS_AUGMENT_MODE_LEGACY_TEXT:
        min_crop = 0
        max_crop = 24

    elif mode == GS_AUGMENT_MODE_SYMMETRIC_FIGURE:
        min_crop = 0
        max_crop = 48

    else:
        raise ValueError(
            "Unsupported augmentation mode"
        )

    crop_range = (
        max_crop
        - min_crop
        + 1
    )

    threshold = (
        c_u64(
            0 - crop_range
        )
        % crop_range
    )

    stream_index = 0

    while True:

        value = gs_augment_stateless_value(
            base_seed,
            epoch,
            manifest_index,
            mode,
            stream_index,
        )

        if value >= threshold:
            return (
                min_crop
                + value % crop_range
            )

        stream_index += 1


# Golden value already frozen in Notebook 01.
assert (
    gs_augment_crop_left_for_event_mode(
        789,
        0,
        7663,
        GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
    )
    == 36
)


# ============================================================
# REAL TRAIN EVENT LOADER
# ============================================================

VIEW_COLUMNS = [
    "img_0_5_272",
    "img_1_0_272",
    "img_2_0_272",
    "img_4_0_272",
]


def load_train_event_224(
    manifest_index,
    epoch,
    augmentation_seed=AUGMENTATION_SEED,
):
    """
    Load one real training event.

    PROJECT pipeline:
        manifest_index + epoch
        -> stateless symmetric crop
        -> four grayscale 224x272 PNG views
        -> identical horizontal crop on all channels
        -> float32 tensor (4,224,224) in [0,1]
    """

    event = event_rows[
        manifest_index
    ]

    crop_left = (
        gs_augment_crop_left_for_event_mode(
            augmentation_seed,
            epoch,
            manifest_index,
            GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
        )
    )

    channels = []

    for column in VIEW_COLUMNS:

        path = (
            DATA_DIR
            / event[column]
        )

        if not path.is_file():
            raise RuntimeError(
                f"Missing PNG: {path}"
            )

        with Image.open(path) as image:

            if image.mode != "L":
                raise RuntimeError(
                    f"PNG is not grayscale: {path}"
                )

            array = np.asarray(
                image,
                dtype=np.uint8,
            ).copy()

        if array.shape != (
            224,
            272,
        ):
            raise RuntimeError(
                "Unexpected PNG shape: "
                f"{array.shape}"
            )

        crop = array[
            :,
            crop_left:
            crop_left + 224
        ]

        channels.append(
            crop
        )

    tensor_u8 = np.stack(
        channels,
        axis=0,
    )

    tensor = torch.from_numpy(
        tensor_u8.astype(
            np.float32
        )
        / np.float32(255.0)
    )

    return (
        tensor,
        crop_left,
        event["event_id"],
    )


# ============================================================
# REAL VALIDATION EVENT LOADER
# ============================================================

def load_validation_event_224(
    manifest_index
):
    """
    Load one official central validation tensor.

    PROJECT validation:
        manifest_index
        -> frozen central .bin
        -> uint8 channel-major 4x224x224
        -> float32 tensor in [0,1]

    No augmentation is applied.
    """

    tensor_row = tensor_rows[
        manifest_index
    ]

    path = (
        DATA_DIR
        / tensor_row["tensor_path"]
    )

    if not path.is_file():
        raise RuntimeError(
            f"Missing central tensor: {path}"
        )

    tensor_u8 = np.fromfile(
        path,
        dtype=np.uint8,
    )

    expected_values = (
        4 * 224 * 224
    )

    if tensor_u8.size != expected_values:
        raise RuntimeError(
            "Unexpected tensor value count: "
            f"{tensor_u8.size}"
        )

    tensor_u8 = tensor_u8.reshape(
        4,
        224,
        224,
    )

    tensor = torch.from_numpy(
        tensor_u8.astype(
            np.float32
        )
        / np.float32(255.0)
    )

    return (
        tensor,
        tensor_row["event_id"],
        int(
            tensor_row["crop_left"]
        ),
    )


# ============================================================
# PYTORCH DATASET
# ============================================================

class GravitySpy224Dataset(Dataset):
    """
    Dataset contract for sakai224_full.

    TRAIN:
        canonical train split
        + epoch-dependent stateless augmentation.

    VALIDATION:
        canonical validation split
        + frozen central tensors.
    """

    def __init__(
        self,
        split_rows,
        split_name,
        epoch=0,
        augmentation_seed=AUGMENTATION_SEED,
    ):
        if split_name not in (
            "train",
            "validation",
        ):
            raise ValueError(
                "split_name must be "
                "'train' or 'validation'"
            )

        self.split_rows = split_rows
        self.split_name = split_name
        self.epoch = int(epoch)
        self.augmentation_seed = int(
            augmentation_seed
        )

    def __len__(self):
        return len(
            self.split_rows
        )

    def set_epoch(
        self,
        epoch,
    ):
        self.epoch = int(
            epoch
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.split_rows[
            index
        ]

        manifest_index = int(
            row["manifest_index"]
        )

        if self.split_name == "train":

            (
                tensor,
                crop_left,
                event_id,
            ) = load_train_event_224(
                manifest_index,
                epoch=self.epoch,
                augmentation_seed=(
                    self.augmentation_seed
                ),
            )

        else:

            (
                tensor,
                event_id,
                crop_left,
            ) = load_validation_event_224(
                manifest_index
            )

        return {
            "x":
                tensor,

            "manifest_index":
                manifest_index,

            "event_id":
                event_id,

            "crop_left":
                crop_left,
        }


# ------------------------------------------------------------
# CREATE DATASETS
# ------------------------------------------------------------

train_dataset = GravitySpy224Dataset(
    train_rows,
    split_name="train",
    epoch=0,
)

val_dataset = GravitySpy224Dataset(
    val_rows,
    split_name="validation",
)


assert len(train_dataset) == 6866
assert len(val_dataset) == 1717


# ------------------------------------------------------------
# FIRST REAL ITEMS — SAME FROZEN CHECK AS NOTEBOOK 01
# ------------------------------------------------------------

train_item = train_dataset[0]
val_item = val_dataset[0]


train_item_ok = (
    tuple(
        train_item["x"].shape
    ) == (
        4,
        224,
        224,
    )

    and train_item["x"].dtype
        == torch.float32

    and torch.isfinite(
        train_item["x"]
    ).all().item()

    and float(
        train_item["x"].min()
    ) >= 0.0

    and float(
        train_item["x"].max()
    ) <= 1.0

    and train_item[
        "manifest_index"
    ] == 7663

    and train_item[
        "crop_left"
    ] == 36
)


val_item_ok = (
    tuple(
        val_item["x"].shape
    ) == (
        4,
        224,
        224,
    )

    and val_item["x"].dtype
        == torch.float32

    and torch.isfinite(
        val_item["x"]
    ).all().item()

    and float(
        val_item["x"].min()
    ) >= 0.0

    and float(
        val_item["x"].max()
    ) <= 1.0

    and val_item[
        "manifest_index"
    ] == 6864

    and val_item[
        "crop_left"
    ] == 24
)


assert train_item_ok
assert val_item_ok


# ------------------------------------------------------------
# CONSERVATIVE DATALOADERS
#
# No shuffle yet in this step.
# The exact PROJECT sampler will be reattached separately.
# ------------------------------------------------------------

train_loader_real_test = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

val_loader_real_test = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)


train_batch_real = next(
    iter(
        train_loader_real_test
    )
)

val_batch_real = next(
    iter(
        val_loader_real_test
    )
)


batch_gate = (
    tuple(
        train_batch_real[
            "x"
        ].shape
    ) == (
        4,
        4,
        224,
        224,
    )

    and tuple(
        val_batch_real[
            "x"
        ].shape
    ) == (
        4,
        4,
        224,
        224,
    )

    and train_batch_real[
        "manifest_index"
    ].tolist()
        == [
            7663,
            2606,
            7841,
            4067,
        ]

    and val_batch_real[
        "manifest_index"
    ].tolist()
        == [
            int(
                row["manifest_index"]
            )
            for row
            in val_rows[:4]
        ]

    and val_batch_real[
        "crop_left"
    ].tolist()
        == [
            24,
            24,
            24,
            24,
        ]
)


assert batch_gate


# ------------------------------------------------------------
# MODEL MUST STILL BE FROZEN
# ------------------------------------------------------------

main_model_unchanged = (
    project_parameter_sha256(
        model
    )
    == FROZEN_INIT_SHA256
)

assert main_model_unchanged


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print(
    "=== REAL DATA LOADER CONTRACT AUDIT ==="
)

for name, value in (
    data_contract_checks.items()
):
    print(
        name,
        "=",
        value,
    )

print()

print(
    "TRAIN_LEN =",
    len(train_dataset),
)

print(
    "VAL_LEN =",
    len(val_dataset),
)

print()

print(
    "TRAIN_ITEM_MANIFEST =",
    train_item[
        "manifest_index"
    ],
)

print(
    "TRAIN_ITEM_CROP =",
    train_item[
        "crop_left"
    ],
)

print(
    "TRAIN_ITEM_OK =",
    train_item_ok,
)

print()

print(
    "VAL_ITEM_MANIFEST =",
    val_item[
        "manifest_index"
    ],
)

print(
    "VAL_ITEM_CROP =",
    val_item[
        "crop_left"
    ],
)

print(
    "VAL_ITEM_OK =",
    val_item_ok,
)

print()

print(
    "TRAIN_BATCH_SHAPE =",
    tuple(
        train_batch_real[
            "x"
        ].shape
    ),
)

print(
    "TRAIN_BATCH_MANIFEST =",
    train_batch_real[
        "manifest_index"
    ].tolist(),
)

print(
    "TRAIN_BATCH_CROPS =",
    train_batch_real[
        "crop_left"
    ].tolist(),
)

print()

print(
    "VAL_BATCH_SHAPE =",
    tuple(
        val_batch_real[
            "x"
        ].shape
    ),
)

print(
    "VAL_BATCH_MANIFEST =",
    val_batch_real[
        "manifest_index"
    ].tolist(),
)

print(
    "VAL_BATCH_CROPS =",
    val_batch_real[
        "crop_left"
    ].tolist(),
)

print()

print(
    "REAL_BATCH_GATE =",
    batch_gate,
)

print(
    "MAIN_FROZEN_MODEL_UNCHANGED =",
    main_model_unchanged,
)

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print()

print(
    "NOTEBOOK_02_REAL_LOADER_CONTRACT_PASS =",
    (
        all(
            data_contract_checks.values()
        )
        and train_item_ok
        and val_item_ok
        and batch_gate
        and main_model_unchanged
    ),
)

=== REAL DATA LOADER CONTRACT AUDIT ===
EVENT_ROWS_OK = True
TENSOR_ROWS_OK = True
TRAIN_ROWS_OK = True
VAL_ROWS_OK = True
TRAIN_UNIQUE_OK = True
VAL_UNIQUE_OK = True
OVERLAP_ZERO = True
FULL_COVERAGE = True

TRAIN_LEN = 6866
VAL_LEN = 1717

TRAIN_ITEM_MANIFEST = 7663
TRAIN_ITEM_CROP = 36
TRAIN_ITEM_OK = True

VAL_ITEM_MANIFEST = 6864
VAL_ITEM_CROP = 24
VAL_ITEM_OK = True

TRAIN_BATCH_SHAPE = (4, 4, 224, 224)
TRAIN_BATCH_MANIFEST = [7663, 2606, 7841, 4067]
TRAIN_BATCH_CROPS = [36, 20, 33, 5]

VAL_BATCH_SHAPE = (4, 4, 224, 224)
VAL_BATCH_MANIFEST = [6864, 1488, 8384, 6585]
VAL_BATCH_CROPS = [24, 24, 24, 24]

REAL_BATCH_GATE = True
MAIN_FROZEN_MODEL_UNCHANGED = True
MODEL_FORWARD_EXECUTED = False

NOTEBOOK_02_REAL_LOADER_CONTRACT_PASS = True


=== REAL DATA LOADER CONTRACT AUDIT ===
EVENT_ROWS_OK = True
TENSOR_ROWS_OK = True
TRAIN_ROWS_OK = True
VAL_ROWS_OK = True
TRAIN_UNIQUE_OK = True
VAL_UNIQUE_OK = True
OVERLAP_ZERO = True
FULL_COVERAGE = True

TRAIN_LEN = 6866
VAL_LEN = 1717

TRAIN_ITEM_MANIFEST = 7663
TRAIN_ITEM_CROP = 36
TRAIN_ITEM_OK = True

VAL_ITEM_MANIFEST = 6864
VAL_ITEM_CROP = 24
VAL_ITEM_OK = True

TRAIN_BATCH_SHAPE = (4, 4, 224, 224)
TRAIN_BATCH_MANIFEST = [7663, 2606, 7841, 4067]
TRAIN_BATCH_CROPS = [36, 20, 33, 5]

VAL_BATCH_SHAPE = (4, 4, 224, 224)
VAL_BATCH_MANIFEST = [6864, 1488, 8384, 6585]
VAL_BATCH_CROPS = [24, 24, 24, 24]

REAL_BATCH_GATE = True
MAIN_FROZEN_MODEL_UNCHANGED = True
MODEL_FORWARD_EXECUTED = False

NOTEBOOK_02_REAL_LOADER_CONTRACT_PASS = True


## STEP 02-K4 — Forward smoke test su dati Gravity Spy reali

Il VAE `sakai224_full` riceve un batch di dati Gravity Spy reali.

La verifica usa un piccolo batch della **validation centrale**, perché rappresenta il percorso più controllato:

- nessuna augmentation;
- `crop_left = 24`;
- input congelato `4×224×224`;
- modalità modello `EVAL`;
- latent deterministico `z = μ`;
- nessun backward;
- nessun optimizer;
- nessuna modifica dei parametri o delle statistiche BatchNorm.

La verifica è un **forward smoke test**, non training.

In [ ]:
# ============================================================
# NOTEBOOK 02 — FIRST REAL GRAVITY SPY MODEL FORWARD
# STEP 02-K4
#
# REAL validation data
# MEAN + EVAL
#
# NO training.
# NO backward.
# NO optimizer.
# ============================================================

import math
import torch


# ------------------------------------------------------------
# USE ONLY FIRST TWO REAL VALIDATION EVENTS
# ------------------------------------------------------------

REAL_VAL_SMOKE_COUNT = 2

x_real_val = (
    val_batch_real["x"][
        :REAL_VAL_SMOKE_COUNT
    ]
    .to(
        device=device,
        dtype=torch.float32,
    )
)

real_val_manifest_indices = (
    val_batch_real[
        "manifest_index"
    ][
        :REAL_VAL_SMOKE_COUNT
    ].tolist()
)

real_val_event_ids = (
    val_batch_real[
        "event_id"
    ][
        :REAL_VAL_SMOKE_COUNT
    ]
)

real_val_crops = (
    val_batch_real[
        "crop_left"
    ][
        :REAL_VAL_SMOKE_COUNT
    ].tolist()
)


# ------------------------------------------------------------
# INPUT CONTRACT
# ------------------------------------------------------------

input_contract_pass = (
    tuple(x_real_val.shape)
        == (2, 4, 224, 224)

    and x_real_val.dtype
        == torch.float32

    and torch.isfinite(
        x_real_val
    ).all().item()

    and float(
        x_real_val.min()
    ) >= 0.0

    and float(
        x_real_val.max()
    ) <= 1.0

    and real_val_manifest_indices
        == [6864, 1488]

    and real_val_crops
        == [24, 24]
)

assert input_contract_pass


# ------------------------------------------------------------
# FROZEN MODEL STATE
# ------------------------------------------------------------

model.eval()

hash_before_real_forward = (
    project_parameter_sha256(
        model
    )
)

assert (
    hash_before_real_forward
    == FROZEN_INIT_SHA256
)

bn_before_real_forward = (
    snapshot_bn_state(
        bn_layers
    )
)


# ------------------------------------------------------------
# REAL FORWARD — MEAN + EVAL
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


with torch.no_grad():

    (
        real_logits,
        real_mu,
        real_logvar,
        real_z,
    ) = model(
        x_real_val,
        epsilon=None,
        mean_mode=True,
    )


torch.cuda.synchronize()


# ------------------------------------------------------------
# OUTPUT CONTRACT
# ------------------------------------------------------------

shape_pass = (
    tuple(real_logits.shape)
        == (2, 4, 224, 224)

    and tuple(real_mu.shape)
        == (2, 512)

    and tuple(real_logvar.shape)
        == (2, 512)

    and tuple(real_z.shape)
        == (2, 512)
)

assert shape_pass


mean_mode_pass = torch.equal(
    real_z,
    real_mu,
)

assert mean_mode_pass


outputs_finite = (
    torch.isfinite(
        real_logits
    ).all().item()

    and torch.isfinite(
        real_mu
    ).all().item()

    and torch.isfinite(
        real_logvar
    ).all().item()

    and torch.isfinite(
        real_z
    ).all().item()
)

assert outputs_finite


# ------------------------------------------------------------
# REAL VALIDATION LOSS
#
# Diagnostic only.
# No backward.
# ------------------------------------------------------------

with torch.no_grad():

    real_val_loss_stats = (
        sakai_vae_loss(
            real_logits,
            x_real_val,
            real_mu,
            real_logvar,
            beta_kl=SPEC.beta_kl,
        )
    )


real_loss_finite = math.isfinite(
    float(
        real_val_loss_stats[
            "loss"
        ]
    )
)

assert real_loss_finite


# ------------------------------------------------------------
# BATCHNORM MUST REMAIN UNCHANGED IN EVAL
# ------------------------------------------------------------

bn_after_real_forward = (
    snapshot_bn_state(
        bn_layers
    )
)

bn_state_unchanged = True

for before, after in zip(
    bn_before_real_forward,
    bn_after_real_forward,
):

    for before_tensor, after_tensor in zip(
        before,
        after,
    ):

        if not torch.equal(
            before_tensor,
            after_tensor,
        ):
            bn_state_unchanged = False


assert bn_state_unchanged


# ------------------------------------------------------------
# PARAMETERS MUST REMAIN FROZEN
# ------------------------------------------------------------

hash_after_real_forward = (
    project_parameter_sha256(
        model
    )
)

parameters_unchanged = (
    hash_after_real_forward
    == FROZEN_INIT_SHA256
)

assert parameters_unchanged


# ------------------------------------------------------------
# MEMORY DIAGNOSTIC
# ------------------------------------------------------------

peak_real_val_mib = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 2)
)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print(
    "=== FIRST REAL GRAVITY SPY FORWARD AUDIT ==="
)

print(
    "DATA_SPLIT = VALIDATION"
)

print(
    "MANIFEST_INDICES =",
    real_val_manifest_indices,
)

print(
    "EVENT_IDS =",
    list(real_val_event_ids),
)

print(
    "CROP_LEFT =",
    real_val_crops,
)

print(
    "INPUT_SHAPE =",
    tuple(x_real_val.shape),
)

print()

print(
    "INPUT_CONTRACT_PASS =",
    input_contract_pass,
)

print(
    "OUTPUT_SHAPES_PASS =",
    shape_pass,
)

print(
    "MEAN_MODE_Z_EQUALS_MU =",
    mean_mode_pass,
)

print(
    "ALL_OUTPUTS_FINITE =",
    outputs_finite,
)

print()

print(
    "REAL_RECON_MEAN_PER_SAMPLE =",
    float(
        real_val_loss_stats[
            "reconstruction_mean_per_sample"
        ]
    ),
)

print(
    "REAL_KL_MEAN_PER_SAMPLE =",
    float(
        real_val_loss_stats[
            "kl_mean_per_sample"
        ]
    ),
)

print(
    "REAL_TOTAL_LOSS =",
    float(
        real_val_loss_stats[
            "loss"
        ]
    ),
)

print(
    "REAL_LOSS_FINITE =",
    real_loss_finite,
)

print()

print(
    "BN_EVAL_STATE_UNCHANGED =",
    bn_state_unchanged,
)

print(
    "PARAMETERS_UNCHANGED =",
    parameters_unchanged,
)

print(
    "PARAMETER_SHA256 =",
    hash_after_real_forward,
)

print()

print(
    "PEAK_CUDA_ALLOCATED_MIB =",
    round(
        peak_real_val_mib,
        3,
    ),
)

print()

print(
    "FORWARD_MODE = MEAN+EVAL"
)

print(
    "REAL_DATA_USED =",
    True,
)

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print(
    "TRAINING_STARTED =",
    False,
)

print()

print(
    "SAKAI224_FULL_REAL_VAL_FORWARD_PASS =",
    (
        input_contract_pass
        and shape_pass
        and mean_mode_pass
        and outputs_finite
        and real_loss_finite
        and bn_state_unchanged
        and parameters_unchanged
    ),
)

=== FIRST REAL GRAVITY SPY FORWARD AUDIT ===
DATA_SPLIT = VALIDATION
MANIFEST_INDICES = [6864, 1488]
EVENT_IDS = ['H1_4D7IcowE8a', 'H1_NGodo27RM7']
CROP_LEFT = [24, 24]
INPUT_SHAPE = (2, 4, 224, 224)

INPUT_CONTRACT_PASS = True
OUTPUT_SHAPES_PASS = True
MEAN_MODE_Z_EQUALS_MU = True
ALL_OUTPUTS_FINITE = True

REAL_RECON_MEAN_PER_SAMPLE = 138416.78125
REAL_KL_MEAN_PER_SAMPLE = 5.042012691497803
REAL_TOTAL_LOSS = 138421.828125
REAL_LOSS_FINITE = True

BN_EVAL_STATE_UNCHANGED = True
PARAMETERS_UNCHANGED = True
PARAMETER_SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b

PEAK_CUDA_ALLOCATED_MIB = 584.723

FORWARD_MODE = MEAN+EVAL
REAL_DATA_USED = True
BACKWARD_EXECUTED = False
OPTIMIZER_EXECUTED = False
TRAINING_STARTED = False

SAKAI224_FULL_REAL_VAL_FORWARD_PASS = True


## STEP 02-K5A — Ordine deterministico del training

Il full split train contiene 6.866 eventi, ma durante il training non viene percorso nell'ordine fisso del CSV.

Il progetto utilizza uno shuffle deterministico PROJECT che replica la semantica del runner C:

1. viene permutata la **posizione nello split**, non il `manifest_index`;
2. la permutazione dipende da `shuffle_seed = 101112` e dall'epoch;
3. il Dataset traduce successivamente ogni posizione permutata nel relativo `manifest_index`;
4. lo stesso numero di epoch controlla anche l'augmentation stateless dell'evento.

Questa separazione preserva il full split ufficiale senza modificarlo.

La verifica riguarda esclusivamente sampler, ordine degli eventi e crop.  
**Il VAE non viene eseguito e il training non rientra in questa verifica.**

In [ ]:
# ============================================================
# NOTEBOOK 02 — PROJECT TRAIN SAMPLER
# STEP 02-K5A
#
# Reconstruct the exact PROJECT/C shuffle semantics already
# validated independently in Notebook 01.
#
# IMPORTANT:
#   - shuffle SPLIT POSITIONS, not manifest_index values;
#   - epoch controls both shuffle and train augmentation;
#   - no model forward;
#   - no backward;
#   - no optimizer.
# ============================================================

from torch.utils.data import (
    Sampler,
    DataLoader,
)


PROJECT_SHUFFLE_SEED = 101112
PROJECT_RNG_DOMAIN_SHUFFLE = 2


# ------------------------------------------------------------
# UNBIASED BOUNDED RANDOM INTEGER
#
# Same rejection-sampling rule used by the C implementation.
# This avoids modulo bias.
# ------------------------------------------------------------

def project_shuffle_bounded_u64(
    state,
    upper_bound_exclusive,
):
    if upper_bound_exclusive <= 0:
        raise ValueError(
            "upper_bound_exclusive must be > 0"
        )

    threshold = (
        ((-upper_bound_exclusive) & MASK64)
        % upper_bound_exclusive
    )

    while True:

        state, random_value = (
            c_next_u64(state)
        )

        if random_value >= threshold:
            return (
                state,
                random_value
                % upper_bound_exclusive,
            )


# ------------------------------------------------------------
# FULL FISHER-YATES PERMUTATION
#
# The RNG seed includes:
#   shuffle seed
#   RNG domain = SHUFFLE
#   epoch
#
# manifest_index and stream_index are zero at permutation
# derivation, exactly as in the PROJECT C shuffle.
# ------------------------------------------------------------

def project_shuffle_permutation(
    count,
    base_seed,
    epoch,
):
    count = int(count)

    if count < 0:
        raise ValueError(
            "count must be >= 0"
        )

    if count == 0:
        return []

    state = c_derive_seed(
        base_seed,
        PROJECT_RNG_DOMAIN_SHUFFLE,
        epoch,
        0,
        0,
    )

    permutation = list(
        range(count)
    )

    # Fisher-Yates:
    # count-1, count-2, ..., 1
    for index in range(
        count - 1,
        0,
        -1,
    ):

        state, selected = (
            project_shuffle_bounded_u64(
                state,
                index + 1,
            )
        )

        (
            permutation[index],
            permutation[selected],
        ) = (
            permutation[selected],
            permutation[index],
        )

    return permutation


# ------------------------------------------------------------
# C GOLDEN VECTORS
#
# These small vectors were already validated against the real
# C shuffle implementation in Notebook 01. Rechecking them
# here prevents accidental semantic drift.
# ------------------------------------------------------------

shuffle_golden_cases = [
    (
        8,
        101112,
        0,
        [0, 5, 4, 7, 1, 3, 6, 2],
    ),
    (
        8,
        101112,
        1,
        [2, 3, 1, 4, 0, 5, 6, 7],
    ),
    (
        8,
        202224,
        0,
        [3, 6, 4, 2, 1, 7, 5, 0],
    ),
    (
        12,
        101112,
        7,
        [8, 11, 10, 6, 2, 0, 9, 4, 5, 1, 7, 3],
    ),
]


shuffle_golden_pass = True

for (
    count,
    seed,
    epoch,
    expected,
) in shuffle_golden_cases:

    actual = project_shuffle_permutation(
        count,
        seed,
        epoch,
    )

    case_pass = (
        actual == expected
    )

    shuffle_golden_pass &= (
        case_pass
    )

    print(
        "GOLDEN",
        "count =",
        count,
        "seed =",
        seed,
        "epoch =",
        epoch,
        "PASS =",
        case_pass,
    )


assert shuffle_golden_pass


# ============================================================
# PYTORCH SAMPLER
# ============================================================

class ProjectEpochSampler(Sampler):
    """
    Deterministic sampler for the canonical train split.

    The sampler emits positions 0..6865 in PROJECT shuffled
    order. GravitySpy224Dataset then resolves each position to
    the corresponding frozen manifest_index.
    """

    def __init__(
        self,
        count,
        base_seed=PROJECT_SHUFFLE_SEED,
        epoch=0,
    ):
        self.count = int(count)
        self.base_seed = int(base_seed)
        self.epoch = int(epoch)

        if self.count < 0:
            raise ValueError(
                "Invalid sample count"
            )

    def set_epoch(
        self,
        epoch,
    ):
        self.epoch = int(epoch)

    def __iter__(self):

        permutation = (
            project_shuffle_permutation(
                self.count,
                self.base_seed,
                self.epoch,
            )
        )

        return iter(permutation)

    def __len__(self):
        return self.count


train_sampler_project = (
    ProjectEpochSampler(
        count=len(train_dataset),
        base_seed=PROJECT_SHUFFLE_SEED,
        epoch=0,
    )
)


def set_project_train_epoch(
    epoch,
):
    """
    Synchronize the two epoch-dependent PROJECT mechanisms:

      1. split-position shuffle;
      2. stateless image augmentation.

    This synchronization is required for reproducibility.
    """

    epoch = int(epoch)

    train_sampler_project.set_epoch(
        epoch
    )

    train_dataset.set_epoch(
        epoch
    )


train_loader_project = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=train_sampler_project,
    num_workers=0,
    drop_last=False,
)


# ============================================================
# FULL TRAIN PERMUTATION AUDIT — EPOCH 0
# ============================================================

set_project_train_epoch(0)

permutation_epoch0 = (
    project_shuffle_permutation(
        len(train_dataset),
        PROJECT_SHUFFLE_SEED,
        0,
    )
)


# Frozen prefix already established in Notebook 01.
EXPECTED_POSITIONS_EPOCH0 = [
    752,
    2659,
    4819,
    4649,
    4369,
    3945,
    4482,
    2825,
    2403,
    3407,
    1964,
    286,
]


prefix_positions_pass = (
    permutation_epoch0[:12]
    == EXPECTED_POSITIONS_EPOCH0
)


full_permutation_valid = (
    len(permutation_epoch0)
        == 6866

    and sorted(
        permutation_epoch0
    ) == list(range(6866))
)


assert prefix_positions_pass
assert full_permutation_valid


# ------------------------------------------------------------
# FIRST REAL PROJECT-SHUFFLED BATCH
# ------------------------------------------------------------

train_batch_project_epoch0 = next(
    iter(
        train_loader_project
    )
)


actual_manifest_epoch0 = (
    train_batch_project_epoch0[
        "manifest_index"
    ].tolist()
)

actual_crops_epoch0 = (
    train_batch_project_epoch0[
        "crop_left"
    ].tolist()
)


expected_manifest_epoch0 = [
    int(
        train_rows[position][
            "manifest_index"
        ]
    )
    for position
    in EXPECTED_POSITIONS_EPOCH0[:4]
]


expected_crops_epoch0 = [
    gs_augment_crop_left_for_event_mode(
        AUGMENTATION_SEED,
        0,
        manifest_index,
        GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
    )
    for manifest_index
    in expected_manifest_epoch0
]


manifest_order_pass = (
    actual_manifest_epoch0
    == expected_manifest_epoch0
)

crop_order_pass = (
    actual_crops_epoch0
    == expected_crops_epoch0
)

batch_shape_pass = (
    tuple(
        train_batch_project_epoch0[
            "x"
        ].shape
    )
    == (
        4,
        4,
        224,
        224,
    )
)


assert manifest_order_pass
assert crop_order_pass
assert batch_shape_pass


# ============================================================
# EPOCH MUST CHANGE THE ORDER
# ============================================================

set_project_train_epoch(1)

train_batch_project_epoch1 = next(
    iter(
        train_loader_project
    )
)

actual_manifest_epoch1 = (
    train_batch_project_epoch1[
        "manifest_index"
    ].tolist()
)

epoch_changes_order = (
    actual_manifest_epoch1
    != actual_manifest_epoch0
)

assert epoch_changes_order


# Restore canonical starting state for the next test.
set_project_train_epoch(0)


# ------------------------------------------------------------
# MAIN MODEL MUST STILL BE UNTOUCHED
# ------------------------------------------------------------

main_model_unchanged = (
    project_parameter_sha256(
        model
    )
    == FROZEN_INIT_SHA256
)

assert main_model_unchanged


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== PROJECT TRAIN SAMPLER AUDIT ==="
)

print(
    "SHUFFLE_SEED =",
    PROJECT_SHUFFLE_SEED,
)

print(
    "TRAIN_COUNT =",
    len(train_dataset),
)

print()

print(
    "SHUFFLE_GOLDEN_PASS =",
    shuffle_golden_pass,
)

print(
    "FULL_PERMUTATION_VALID =",
    full_permutation_valid,
)

print(
    "PREFIX_POSITIONS_PASS =",
    prefix_positions_pass,
)

print(
    "FIRST_12_POSITIONS_EPOCH0 =",
    permutation_epoch0[:12],
)

print()

print(
    "EXPECTED_MANIFEST_EPOCH0 =",
    expected_manifest_epoch0,
)

print(
    "ACTUAL_MANIFEST_EPOCH0 =",
    actual_manifest_epoch0,
)

print(
    "MANIFEST_ORDER_PASS =",
    manifest_order_pass,
)

print()

print(
    "EXPECTED_CROPS_EPOCH0 =",
    expected_crops_epoch0,
)

print(
    "ACTUAL_CROPS_EPOCH0 =",
    actual_crops_epoch0,
)

print(
    "CROP_ORDER_PASS =",
    crop_order_pass,
)

print()

print(
    "FIRST_MANIFEST_EPOCH1 =",
    actual_manifest_epoch1,
)

print(
    "EPOCH_CHANGES_ORDER =",
    epoch_changes_order,
)

print()

print(
    "TRAIN_BATCH_SHAPE =",
    tuple(
        train_batch_project_epoch0[
            "x"
        ].shape
    ),
)

print(
    "MAIN_FROZEN_MODEL_UNCHANGED =",
    main_model_unchanged,
)

print()

print(
    "MODEL_FORWARD_EXECUTED =",
    False,
)

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print(
    "TRAINING_STARTED =",
    False,
)

print()

print(
    "NOTEBOOK_02_PROJECT_SAMPLER_PASS =",
    (
        shuffle_golden_pass
        and full_permutation_valid
        and prefix_positions_pass
        and manifest_order_pass
        and crop_order_pass
        and batch_shape_pass
        and epoch_changes_order
        and main_model_unchanged
    ),
)

GOLDEN count = 8 seed = 101112 epoch = 0 PASS = True
GOLDEN count = 8 seed = 101112 epoch = 1 PASS = True
GOLDEN count = 8 seed = 202224 epoch = 0 PASS = True
GOLDEN count = 12 seed = 101112 epoch = 7 PASS = True

=== PROJECT TRAIN SAMPLER AUDIT ===
SHUFFLE_SEED = 101112
TRAIN_COUNT = 6866

SHUFFLE_GOLDEN_PASS = True
FULL_PERMUTATION_VALID = True
PREFIX_POSITIONS_PASS = True
FIRST_12_POSITIONS_EPOCH0 = [752, 2659, 4819, 4649, 4369, 3945, 4482, 2825, 2403, 3407, 1964, 286]

EXPECTED_MANIFEST_EPOCH0 = [2676, 6250, 2396, 5602]
ACTUAL_MANIFEST_EPOCH0 = [2676, 6250, 2396, 5602]
MANIFEST_ORDER_PASS = True

EXPECTED_CROPS_EPOCH0 = [20, 15, 19, 12]
ACTUAL_CROPS_EPOCH0 = [20, 15, 19, 12]
CROP_ORDER_PASS = True

FIRST_MANIFEST_EPOCH1 = [6028, 4009, 12, 5170]
EPOCH_CHANGES_ORDER = True

TRAIN_BATCH_SHAPE = (4, 4, 224, 224)
MAIN_FROZEN_MODEL_UNCHANGED = True

MODEL_FORWARD_EXECUTED = False
BACKWARD_EXECUTED = False
OPTIMIZER_EXECUTED = False
TRAINING_STARTED = False

NOTEBOOK_02_PROJECT_SAMPLER_PA

## STEP 02-K5B — Forward smoke test sul train Gravity Spy reale

Il loader completo del training viene validato con:

- full split train di 6.866 eventi;
- shuffle PROJECT deterministico;
- augmentation stateless dipendente da `epoch` e `manifest_index`;
- crop identico sulle quattro viste dello stesso evento.

La verifica passa un batch reale a un **modello di audit separato** in modalità `TRAIN`.

Viene utilizzata la reparameterization stocastica:

`z = μ + exp(0.5 · logvar) · ε`

dove `ε` è generato esternamente tramite il RNG stateless PROJECT del training.

### Importante

Questo è esclusivamente un **forward smoke test**:

- `torch.no_grad()` attivo;
- nessun backward;
- nessun optimizer;
- nessun aggiornamento dei parametri;
- il modello principale congelato non viene eseguito;
- la verifica non avvia il training ufficiale.

Le statistiche running della BatchNorm del solo modello di audit devono invece aggiornarsi, perché il forward viene deliberatamente eseguito in modalità `TRAIN`.

In [ ]:
# ============================================================
# NOTEBOOK 02 — FIRST REAL TRAIN FORWARD
# STEP 02-K5B
#
# REAL Gravity Spy train batch
# STOCHASTIC + TRAIN mode
# FORWARD ONLY
#
# No gradients.
# No backward.
# No optimizer.
# No official training.
# ============================================================

import gc
import math
import numpy as np
import torch


# ------------------------------------------------------------
# CLEAN OLD AUDIT-ONLY OBJECTS
#
# Previous backward/optimizer smoke models are no longer
# needed. Removing them only frees temporary GPU/RAM memory.
# The frozen main model is deliberately preserved.
# ------------------------------------------------------------

for name in [
    "backward_model",
    "training_model",
    "training_optimizer",
    "parameters_before",
    "x_backward",
    "x_train_step",
]:
    globals().pop(
        name,
        None,
    )

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# MAIN MODEL PRECONDITION
#
# The official/frozen model must still have exactly the
# verified initial PROJECT fingerprint.
# ------------------------------------------------------------

main_hash_before = (
    project_parameter_sha256(
        model
    )
)

assert (
    main_hash_before
    == FROZEN_INIT_SHA256
)

main_bn_before = (
    snapshot_bn_state(
        bn_layers
    )
)


# ------------------------------------------------------------
# REBUILD FIRST PROJECT TRAIN BATCH AT EPOCH 0
#
# We explicitly reset both:
#   - sampler epoch;
#   - augmentation epoch.
# ------------------------------------------------------------

REAL_TRAIN_SMOKE_EPOCH = 0

set_project_train_epoch(
    REAL_TRAIN_SMOKE_EPOCH
)

real_train_batch = next(
    iter(
        train_loader_project
    )
)


real_train_manifests = (
    real_train_batch[
        "manifest_index"
    ].tolist()
)

real_train_crops = (
    real_train_batch[
        "crop_left"
    ].tolist()
)


EXPECTED_REAL_TRAIN_MANIFESTS = [
    2676,
    6250,
    2396,
    5602,
]

EXPECTED_REAL_TRAIN_CROPS = [
    20,
    15,
    19,
    12,
]


batch_identity_pass = (
    real_train_manifests
        == EXPECTED_REAL_TRAIN_MANIFESTS

    and real_train_crops
        == EXPECTED_REAL_TRAIN_CROPS
)

assert batch_identity_pass


# ------------------------------------------------------------
# REAL INPUT TO GPU
# ------------------------------------------------------------

x_real_train = (
    real_train_batch["x"]
    .to(
        device=device,
        dtype=torch.float32,
    )
)


input_contract_pass = (
    tuple(
        x_real_train.shape
    )
        == (4, 4, 224, 224)

    and x_real_train.dtype
        == torch.float32

    and torch.isfinite(
        x_real_train
    ).all().item()

    and float(
        x_real_train.min()
    ) >= 0.0

    and float(
        x_real_train.max()
    ) <= 1.0
)

assert input_contract_pass


# ------------------------------------------------------------
# EXTERNAL STATELESS TRAIN EPSILON
#
# Every event gets its own deterministic epsilon using:
#
#   latent_seed = 456
#   RNG domain  = LATENT_TRAIN
#   epoch       = 0
#   manifest_index
#
# No RNG is executed inside the VAE.
# ------------------------------------------------------------

real_train_epsilon_np = np.stack(
    [
        python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_TRAIN,
            REAL_TRAIN_SMOKE_EPOCH,
            manifest_index,
            SPEC.latent_dim,
        )

        for manifest_index
        in real_train_manifests
    ],
    axis=0,
)


real_train_epsilon = (
    torch.from_numpy(
        real_train_epsilon_np
    )
    .to(
        device=device,
        dtype=torch.float32,
    )
)


epsilon_contract_pass = (
    tuple(
        real_train_epsilon.shape
    ) == (4, 512)

    and torch.isfinite(
        real_train_epsilon
    ).all().item()
)

assert epsilon_contract_pass


# ------------------------------------------------------------
# FRESH AUDIT MODEL
#
# It starts from exactly the same frozen initialization as
# the main model, but is independent from it.
# ------------------------------------------------------------

real_train_audit_model = (
    Sakai224FullVAE(
        SPEC
    )
)

initialize_project_c_exact_v1(
    real_train_audit_model,
    PROJECT_INITIALIZATION_SEED,
)


audit_hash_initial = (
    project_parameter_sha256(
        real_train_audit_model
    )
)

assert (
    audit_hash_initial
    == FROZEN_INIT_SHA256
)


real_train_audit_model = (
    real_train_audit_model.to(
        device=device,
        dtype=torch.float32,
    )
)

# TRAIN mode is intentional here:
# BatchNorm uses batch statistics and updates running state.
real_train_audit_model.train()


audit_bn_layers = (
    get_project_batchnorm_layers(
        real_train_audit_model
    )
)

audit_bn_before = (
    snapshot_bn_state(
        audit_bn_layers
    )
)


# ------------------------------------------------------------
# REAL STOCHASTIC TRAIN-MODE FORWARD
#
# torch.no_grad() ensures:
#   - no autograd graph;
#   - no gradients;
#   - no backward possible from this forward.
#
# BatchNorm running statistics still update because the
# model itself is in TRAIN mode.
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


with torch.no_grad():

    (
        real_train_logits,
        real_train_mu,
        real_train_logvar,
        real_train_z,
    ) = real_train_audit_model(
        x_real_train,
        epsilon=real_train_epsilon,
        mean_mode=False,
    )


torch.cuda.synchronize()


# ------------------------------------------------------------
# OUTPUT SHAPE / FINITENESS CONTRACT
# ------------------------------------------------------------

output_shape_pass = (
    tuple(
        real_train_logits.shape
    ) == (4, 4, 224, 224)

    and tuple(
        real_train_mu.shape
    ) == (4, 512)

    and tuple(
        real_train_logvar.shape
    ) == (4, 512)

    and tuple(
        real_train_z.shape
    ) == (4, 512)
)

assert output_shape_pass


outputs_finite = (
    torch.isfinite(
        real_train_logits
    ).all().item()

    and torch.isfinite(
        real_train_mu
    ).all().item()

    and torch.isfinite(
        real_train_logvar
    ).all().item()

    and torch.isfinite(
        real_train_z
    ).all().item()
)

assert outputs_finite


# ------------------------------------------------------------
# REPARAMETERIZATION FORMULA
# ------------------------------------------------------------

with torch.no_grad():

    real_train_z_expected = (
        real_train_mu
        + torch.exp(
            0.5
            * real_train_logvar
        )
        * real_train_epsilon
    )


z_formula_pass = torch.allclose(
    real_train_z,
    real_train_z_expected,
    rtol=1e-6,
    atol=1e-7,
)

assert z_formula_pass


stochastic_z_differs_from_mu = (
    not torch.equal(
        real_train_z,
        real_train_mu,
    )
)

assert stochastic_z_differs_from_mu


# ------------------------------------------------------------
# LOSS — DIAGNOSTIC ONLY
#
# The loss is evaluated, but no gradient is requested.
# ------------------------------------------------------------

with torch.no_grad():

    real_train_loss_stats = (
        sakai_vae_loss(
            real_train_logits,
            x_real_train,
            real_train_mu,
            real_train_logvar,
            beta_kl=SPEC.beta_kl,
        )
    )


real_train_loss_finite = (
    math.isfinite(
        float(
            real_train_loss_stats[
                "loss"
            ]
        )
    )
)

assert real_train_loss_finite


# ------------------------------------------------------------
# BATCHNORM TRAIN-MODE AUDIT
#
# One forward must increment every BN batch counter by one.
# At least one running statistic must change.
# ------------------------------------------------------------

audit_bn_after = (
    snapshot_bn_state(
        audit_bn_layers
    )
)


bn_counters_incremented = True
bn_running_state_changed = False


for before, after in zip(
    audit_bn_before,
    audit_bn_after,
):

    (
        before_mean,
        before_var,
        before_count,
    ) = before

    (
        after_mean,
        after_var,
        after_count,
    ) = after


    if int(
        after_count.item()
    ) != (
        int(
            before_count.item()
        ) + 1
    ):
        bn_counters_incremented = False


    if (
        not torch.equal(
            before_mean,
            after_mean,
        )
        or
        not torch.equal(
            before_var,
            after_var,
        )
    ):
        bn_running_state_changed = True


assert bn_counters_incremented
assert bn_running_state_changed


# ------------------------------------------------------------
# FORWARD MUST NOT CHANGE TRAINABLE PARAMETERS
#
# BatchNorm running buffers change, but parameter values do not.
# ------------------------------------------------------------

audit_hash_after_forward = (
    project_parameter_sha256(
        real_train_audit_model
    )
)

audit_parameters_unchanged = (
    audit_hash_after_forward
    == FROZEN_INIT_SHA256
)

assert audit_parameters_unchanged


# ------------------------------------------------------------
# MAIN FROZEN MODEL MUST BE COMPLETELY UNTOUCHED
# ------------------------------------------------------------

main_hash_after = (
    project_parameter_sha256(
        model
    )
)


main_parameters_unchanged = (
    main_hash_after
    == FROZEN_INIT_SHA256
)


main_bn_after = (
    snapshot_bn_state(
        bn_layers
    )
)


main_bn_unchanged = True

for before, after in zip(
    main_bn_before,
    main_bn_after,
):

    for before_tensor, after_tensor in zip(
        before,
        after,
    ):

        if not torch.equal(
            before_tensor,
            after_tensor,
        ):
            main_bn_unchanged = False


assert main_parameters_unchanged
assert main_bn_unchanged


# ------------------------------------------------------------
# MEMORY DIAGNOSTIC
# ------------------------------------------------------------

peak_cuda_mib = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 2)
)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print(
    "=== FIRST REAL TRAIN FORWARD AUDIT ==="
)

print(
    "FORWARD_MODE =",
    "STOCHASTIC+TRAIN_FORWARD_ONLY",
)

print(
    "TRAIN_EPOCH =",
    REAL_TRAIN_SMOKE_EPOCH,
)

print(
    "MANIFEST_INDICES =",
    real_train_manifests,
)

print(
    "CROP_LEFT =",
    real_train_crops,
)

print(
    "INPUT_SHAPE =",
    tuple(
        x_real_train.shape
    ),
)

print()

print(
    "BATCH_IDENTITY_PASS =",
    batch_identity_pass,
)

print(
    "INPUT_CONTRACT_PASS =",
    input_contract_pass,
)

print(
    "EPSILON_CONTRACT_PASS =",
    epsilon_contract_pass,
)

print(
    "OUTPUT_SHAPES_PASS =",
    output_shape_pass,
)

print(
    "ALL_OUTPUTS_FINITE =",
    outputs_finite,
)

print(
    "Z_FORMULA_PASS =",
    z_formula_pass,
)

print(
    "STOCHASTIC_Z_DIFFERS_FROM_MU =",
    stochastic_z_differs_from_mu,
)

print()

print(
    "REAL_RECON_MEAN_PER_SAMPLE =",
    float(
        real_train_loss_stats[
            "reconstruction_mean_per_sample"
        ]
    ),
)

print(
    "REAL_KL_MEAN_PER_SAMPLE =",
    float(
        real_train_loss_stats[
            "kl_mean_per_sample"
        ]
    ),
)

print(
    "REAL_TOTAL_LOSS =",
    float(
        real_train_loss_stats[
            "loss"
        ]
    ),
)

print(
    "REAL_LOSS_FINITE =",
    real_train_loss_finite,
)

print()

print(
    "AUDIT_BN_COUNTERS_INCREMENTED =",
    bn_counters_incremented,
)

print(
    "AUDIT_BN_RUNNING_STATE_CHANGED =",
    bn_running_state_changed,
)

print(
    "AUDIT_PARAMETERS_UNCHANGED =",
    audit_parameters_unchanged,
)

print()

print(
    "MAIN_PARAMETERS_UNCHANGED =",
    main_parameters_unchanged,
)

print(
    "MAIN_BN_STATE_UNCHANGED =",
    main_bn_unchanged,
)

print(
    "MAIN_PARAMETER_SHA256 =",
    main_hash_after,
)

print()

print(
    "PEAK_CUDA_ALLOCATED_MIB =",
    round(
        peak_cuda_mib,
        3,
    ),
)

print()

print(
    "REAL_DATA_USED =",
    True,
)

print(
    "AUTOGRAD_ENABLED_FOR_FORWARD =",
    False,
)

print(
    "BACKWARD_EXECUTED =",
    False,
)

print(
    "OPTIMIZER_EXECUTED =",
    False,
)

print(
    "PARAMETER_UPDATE_EXECUTED =",
    False,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "SAKAI224_FULL_REAL_TRAIN_FORWARD_PASS =",
    (
        batch_identity_pass
        and input_contract_pass
        and epsilon_contract_pass
        and output_shape_pass
        and outputs_finite
        and z_formula_pass
        and stochastic_z_differs_from_mu
        and real_train_loss_finite
        and bn_counters_incremented
        and bn_running_state_changed
        and audit_parameters_unchanged
        and main_parameters_unchanged
        and main_bn_unchanged
    ),
)

=== FIRST REAL TRAIN FORWARD AUDIT ===
FORWARD_MODE = STOCHASTIC+TRAIN_FORWARD_ONLY
TRAIN_EPOCH = 0
MANIFEST_INDICES = [2676, 6250, 2396, 5602]
CROP_LEFT = [20, 15, 19, 12]
INPUT_SHAPE = (4, 4, 224, 224)

BATCH_IDENTITY_PASS = True
INPUT_CONTRACT_PASS = True
EPSILON_CONTRACT_PASS = True
OUTPUT_SHAPES_PASS = True
ALL_OUTPUTS_FINITE = True
Z_FORMULA_PASS = True
STOCHASTIC_Z_DIFFERS_FROM_MU = True

REAL_RECON_MEAN_PER_SAMPLE = 175900.0
REAL_KL_MEAN_PER_SAMPLE = 51.277156829833984
REAL_TOTAL_LOSS = 175951.28125
REAL_LOSS_FINITE = True

AUDIT_BN_COUNTERS_INCREMENTED = True
AUDIT_BN_RUNNING_STATE_CHANGED = True
AUDIT_PARAMETERS_UNCHANGED = True

MAIN_PARAMETERS_UNCHANGED = True
MAIN_BN_STATE_UNCHANGED = True
MAIN_PARAMETER_SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b

PEAK_CUDA_ALLOCATED_MIB = 491.48

REAL_DATA_USED = True
AUTOGRAD_ENABLED_FOR_FORWARD = False
BACKWARD_EXECUTED = False
OPTIMIZER_EXECUTED = False
PARAMETER_UPDATE_EXECUTED = False
OFFICIAL_TRAINING_

## STEP 02-K5C — Singolo training-step su dati Gravity Spy reali

Questa verifica controlla l'intera catena di training usando un batch reale del full split train:

`shuffle PROJECT → augmentation stateless → epsilon stateless → forward → VAE loss → backward → Adam`

Il test utilizza esclusivamente il primo batch deterministico dell'epoch 0:

- `manifest_index = [2676, 6250, 2396, 5602]`
- `crop_left = [20, 15, 19, 12]`

Viene creato un **modello di audit separato**, inizializzato dallo stesso stato PROJECT congelato.

Il modello principale non viene addestrato e deve mantenere:

`SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b`

### Obiettivo del test

Verificare che, su dati Gravity Spy reali:

- la loss sia finita;
- tutti i parametri ricevano gradienti validi;
- Adam costruisca correttamente il proprio stato;
- almeno i percorsi principali encoder/decoder vengano aggiornati;
- BatchNorm evolva correttamente in modalità training;
- il modello principale congelato resti completamente invariato.

La verifica è un **engineering smoke test di un solo step**.  
La verifica non avvia la run scientifica ufficiale.

In [ ]:
# ============================================================
# NOTEBOOK 02 — ONE REAL-DATA TRAINING STEP
# STEP 02-K5C
#
# REAL Gravity Spy data
# STOCHASTIC + TRAIN
#
# forward -> loss -> backward -> Adam
#
# IMPORTANT:
#   separate audit model
#   exactly ONE optimizer step
#   official training is NOT started
# ============================================================

import gc
import hashlib
import math
import numpy as np
import torch


# ------------------------------------------------------------
# RELEASE PREVIOUS AUDIT-ONLY MODEL / ACTIVATIONS
#
# They are no longer needed. This only frees temporary memory.
# The frozen main model is deliberately preserved.
# ------------------------------------------------------------

for name in [
    "real_train_audit_model",
    "real_train_logits",
    "real_train_mu",
    "real_train_logvar",
    "real_train_z",
    "real_train_z_expected",
]:
    globals().pop(
        name,
        None,
    )

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# MAIN FROZEN MODEL — PRECONDITION
# ============================================================

main_hash_before_real_step = (
    project_parameter_sha256(
        model
    )
)

assert (
    main_hash_before_real_step
    == FROZEN_INIT_SHA256
)

main_bn_before_real_step = (
    snapshot_bn_state(
        bn_layers
    )
)


# ============================================================
# REBUILD THE CANONICAL FIRST TRAIN BATCH — EPOCH 0
#
# Rebuilding it here makes this audit self-contained and
# ensures that both shuffle and augmentation use epoch 0.
# ============================================================

REAL_STEP_EPOCH = 0

set_project_train_epoch(
    REAL_STEP_EPOCH
)

real_step_batch = next(
    iter(
        train_loader_project
    )
)


real_step_manifests = (
    real_step_batch[
        "manifest_index"
    ].tolist()
)

real_step_crops = (
    real_step_batch[
        "crop_left"
    ].tolist()
)


EXPECTED_REAL_STEP_MANIFESTS = [
    2676,
    6250,
    2396,
    5602,
]

EXPECTED_REAL_STEP_CROPS = [
    20,
    15,
    19,
    12,
]


batch_identity_pass = (
    real_step_manifests
        == EXPECTED_REAL_STEP_MANIFESTS

    and real_step_crops
        == EXPECTED_REAL_STEP_CROPS
)

assert batch_identity_pass


# ------------------------------------------------------------
# COPY REAL INPUT TO GPU
# ------------------------------------------------------------

x_real_step = (
    real_step_batch["x"]
    .to(
        device=device,
        dtype=torch.float32,
    )
)


input_contract_pass = (
    tuple(
        x_real_step.shape
    ) == (
        4,
        4,
        224,
        224,
    )

    and x_real_step.dtype
        == torch.float32

    and torch.isfinite(
        x_real_step
    ).all().item()

    and float(
        x_real_step.min()
    ) >= 0.0

    and float(
        x_real_step.max()
    ) <= 1.0
)

assert input_contract_pass


# ============================================================
# EXTERNAL PROJECT TRAIN EPSILON
#
# epsilon is event-specific and epoch-specific.
# The VAE itself contains no random-number generator.
# ============================================================

real_step_epsilon_np = np.stack(
    [
        python_latent_epsilon(
            PROJECT_LATENT_SEED,
            RNG_DOMAIN_LATENT_TRAIN,
            REAL_STEP_EPOCH,
            manifest_index,
            SPEC.latent_dim,
        )

        for manifest_index
        in real_step_manifests
    ],
    axis=0,
)


real_step_epsilon = (
    torch.from_numpy(
        real_step_epsilon_np
    )
    .to(
        device=device,
        dtype=torch.float32,
    )
)


epsilon_contract_pass = (
    tuple(
        real_step_epsilon.shape
    ) == (
        4,
        512,
    )

    and torch.isfinite(
        real_step_epsilon
    ).all().item()
)

assert epsilon_contract_pass


# ============================================================
# FRESH INDEPENDENT TRAINING MODEL
#
# Initialize on CPU with the verified PROJECT C policy,
# confirm the fingerprint, then move to CUDA.
# ============================================================

real_step_model = Sakai224FullVAE(
    SPEC
)

initialize_project_c_exact_v1(
    real_step_model,
    PROJECT_INITIALIZATION_SEED,
)


real_step_initial_hash = (
    project_parameter_sha256(
        real_step_model
    )
)

assert (
    real_step_initial_hash
    == FROZEN_INIT_SHA256
)


real_step_model = real_step_model.to(
    device=device,
    dtype=torch.float32,
)

real_step_model.train()


# ============================================================
# PAPER ADAM CONFIGURATION
# ============================================================

real_step_optimizer = torch.optim.Adam(
    real_step_model.parameters(),

    lr=PAPER_LEARNING_RATE,

    betas=(
        PAPER_ADAM_BETA1,
        PAPER_ADAM_BETA2,
    ),

    eps=PAPER_ADAM_EPSILON,

    weight_decay=0.0,

    amsgrad=False,

    foreach=False,

    fused=False,
)


# ============================================================
# EXACT PER-PARAMETER HASHES
#
# We avoid keeping a second full GPU copy of the parameters.
# Instead each parameter is hashed before and after Adam.
# ============================================================

def parameter_hash_map(module):
    result = {}

    for name, parameter in (
        module.named_parameters()
    ):

        array = (
            parameter
            .detach()
            .cpu()
            .contiguous()
            .numpy()
        )

        result[name] = (
            hashlib.sha256(
                array.tobytes(
                    order="C"
                )
            ).hexdigest()
        )

    return result


parameter_hashes_before = (
    parameter_hash_map(
        real_step_model
    )
)


# ------------------------------------------------------------
# BATCHNORM SNAPSHOT BEFORE TRAIN FORWARD
# ------------------------------------------------------------

real_step_bn_layers = (
    get_project_batchnorm_layers(
        real_step_model
    )
)

real_step_bn_before = (
    snapshot_bn_state(
        real_step_bn_layers
    )
)


# ============================================================
# CLEAR GRADIENTS
# ============================================================

real_step_optimizer.zero_grad(
    set_to_none=True
)

assert all(
    parameter.grad is None
    for parameter
    in real_step_model.parameters()
)


# ============================================================
# REAL STOCHASTIC TRAIN FORWARD
#
# Autograd IS enabled in this step because we deliberately
# want to verify the real backward path.
# ============================================================

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


(
    real_step_logits,
    real_step_mu,
    real_step_logvar,
    real_step_z,
) = real_step_model(
    x_real_step,
    epsilon=real_step_epsilon,
    mean_mode=False,
)


# ------------------------------------------------------------
# VAE LOSS
# ------------------------------------------------------------

real_step_loss_stats = (
    sakai_vae_loss(
        real_step_logits,
        x_real_step,
        real_step_mu,
        real_step_logvar,
        beta_kl=SPEC.beta_kl,
    )
)

real_step_loss = (
    real_step_loss_stats[
        "loss"
    ]
)


loss_finite = torch.isfinite(
    real_step_loss
).item()

assert loss_finite


# ============================================================
# BACKWARD
# ============================================================

real_step_loss.backward()


gradient_tensor_count = 0
nonzero_gradient_tensor_count = 0

all_gradients_present = True
all_gradients_finite = True


for name, parameter in (
    real_step_model.named_parameters()
):

    gradient = parameter.grad

    if gradient is None:

        all_gradients_present = False
        continue

    gradient_tensor_count += 1

    if not torch.isfinite(
        gradient
    ).all().item():

        all_gradients_finite = False

    if float(
        gradient
        .detach()
        .abs()
        .sum()
    ) > 0.0:

        nonzero_gradient_tensor_count += 1


assert all_gradients_present
assert all_gradients_finite
assert gradient_tensor_count == 28


# ------------------------------------------------------------
# IMPORTANT END-TO-END PATHS
# ------------------------------------------------------------

key_gradient_parameters = {
    "encoder.conv1.weight":
        real_step_model.encoder.conv1.weight,

    "encoder.fc.weight":
        real_step_model.encoder.fc.weight,

    "decoder.fc.weight":
        real_step_model.decoder.fc.weight,

    "decoder.out_conv.weight":
        real_step_model.decoder.out_conv.weight,
}


key_gradients_nonzero = True


for name, parameter in (
    key_gradient_parameters.items()
):

    gradient = parameter.grad

    case_pass = (
        gradient is not None

        and torch.isfinite(
            gradient
        ).all().item()

        and float(
            gradient
            .detach()
            .abs()
            .sum()
        ) > 0.0
    )

    key_gradients_nonzero &= (
        case_pass
    )

    print(
        "KEY_GRAD",
        name,
        "PASS =",
        case_pass,
    )


assert key_gradients_nonzero


# ============================================================
# ADAM — EXACTLY ONE STEP
# ============================================================

real_step_optimizer.step()

torch.cuda.synchronize()


# ------------------------------------------------------------
# PARAMETER CHANGE AUDIT
# ------------------------------------------------------------

parameter_hashes_after = (
    parameter_hash_map(
        real_step_model
    )
)


changed_parameter_names = [
    name

    for name
    in parameter_hashes_before

    if (
        parameter_hashes_before[
            name
        ]
        != parameter_hashes_after[
            name
        ]
    )
]


changed_parameter_tensor_count = (
    len(
        changed_parameter_names
    )
)


key_parameters_changed = all(
    name in changed_parameter_names

    for name in [
        "encoder.conv1.weight",
        "encoder.fc.weight",
        "decoder.fc.weight",
        "decoder.out_conv.weight",
    ]
)


assert (
    changed_parameter_tensor_count
    > 0
)

assert key_parameters_changed


# ------------------------------------------------------------
# GLOBAL POST-ADAM FINGERPRINT
# ------------------------------------------------------------

post_adam_hash = (
    project_parameter_sha256(
        real_step_model
    )
)


optimizer_changed_model = (
    post_adam_hash
    != FROZEN_INIT_SHA256
)

assert optimizer_changed_model


# ============================================================
# ADAM INTERNAL STATE
# ============================================================

adam_state_count = len(
    real_step_optimizer.state
)

adam_states_finite = True
adam_steps_are_one = True


for parameter in (
    real_step_model.parameters()
):

    state = (
        real_step_optimizer.state[
            parameter
        ]
    )

    if not torch.isfinite(
        state["exp_avg"]
    ).all().item():

        adam_states_finite = False


    if not torch.isfinite(
        state["exp_avg_sq"]
    ).all().item():

        adam_states_finite = False


    if int(
        state["step"].item()
    ) != 1:

        adam_steps_are_one = False


assert adam_state_count == 28
assert adam_states_finite
assert adam_steps_are_one


# ============================================================
# BATCHNORM TRAIN STATE
# ============================================================

real_step_bn_after = (
    snapshot_bn_state(
        real_step_bn_layers
    )
)


bn_counters_incremented = True
bn_running_state_changed = False


for before, after in zip(
    real_step_bn_before,
    real_step_bn_after,
):

    (
        before_mean,
        before_var,
        before_count,
    ) = before

    (
        after_mean,
        after_var,
        after_count,
    ) = after


    if int(
        after_count.item()
    ) != (
        int(
            before_count.item()
        ) + 1
    ):
        bn_counters_incremented = False


    if (
        not torch.equal(
            before_mean,
            after_mean,
        )
        or
        not torch.equal(
            before_var,
            after_var,
        )
    ):
        bn_running_state_changed = True


assert bn_counters_incremented
assert bn_running_state_changed


# ------------------------------------------------------------
# UPDATED MODEL MUST REMAIN NUMERICALLY VALID
# ------------------------------------------------------------

updated_parameters_finite = all(
    torch.isfinite(
        parameter
    ).all().item()

    for parameter
    in real_step_model.parameters()
)

assert updated_parameters_finite


# ============================================================
# MAIN FROZEN MODEL MUST REMAIN COMPLETELY UNTOUCHED
# ============================================================

main_hash_after_real_step = (
    project_parameter_sha256(
        model
    )
)

main_parameters_unchanged = (
    main_hash_after_real_step
    == FROZEN_INIT_SHA256
)


main_bn_after_real_step = (
    snapshot_bn_state(
        bn_layers
    )
)


main_bn_unchanged = True


for before, after in zip(
    main_bn_before_real_step,
    main_bn_after_real_step,
):

    for before_tensor, after_tensor in zip(
        before,
        after,
    ):

        if not torch.equal(
            before_tensor,
            after_tensor,
        ):
            main_bn_unchanged = False


assert main_parameters_unchanged
assert main_bn_unchanged


# ============================================================
# MEMORY DIAGNOSTIC
# ============================================================

peak_real_step_cuda_mib = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 2)
)


# ============================================================
# REPORT
# ============================================================

print()
print(
    "=== ONE REAL-DATA TRAINING STEP AUDIT ==="
)

print(
    "TRAIN_EPOCH =",
    REAL_STEP_EPOCH,
)

print(
    "MANIFEST_INDICES =",
    real_step_manifests,
)

print(
    "CROP_LEFT =",
    real_step_crops,
)

print(
    "INPUT_SHAPE =",
    tuple(
        x_real_step.shape
    ),
)

print()

print(
    "BATCH_IDENTITY_PASS =",
    batch_identity_pass,
)

print(
    "INPUT_CONTRACT_PASS =",
    input_contract_pass,
)

print(
    "EPSILON_CONTRACT_PASS =",
    epsilon_contract_pass,
)

print()

print(
    "LOSS_BEFORE_ADAM =",
    float(
        real_step_loss.detach()
    ),
)

print(
    "RECON_MEAN_PER_SAMPLE =",
    float(
        real_step_loss_stats[
            "reconstruction_mean_per_sample"
        ].detach()
    ),
)

print(
    "KL_MEAN_PER_SAMPLE =",
    float(
        real_step_loss_stats[
            "kl_mean_per_sample"
        ].detach()
    ),
)

print(
    "LOSS_FINITE =",
    loss_finite,
)

print()

print(
    "GRADIENT_TENSORS =",
    gradient_tensor_count,
)

print(
    "NONZERO_GRADIENT_TENSORS =",
    nonzero_gradient_tensor_count,
)

print(
    "ALL_GRADIENTS_PRESENT =",
    all_gradients_present,
)

print(
    "ALL_GRADIENTS_FINITE =",
    all_gradients_finite,
)

print(
    "KEY_GRADIENTS_NONZERO =",
    key_gradients_nonzero,
)

print()

print(
    "CHANGED_PARAMETER_TENSORS =",
    changed_parameter_tensor_count,
)

print(
    "KEY_PARAMETERS_CHANGED =",
    key_parameters_changed,
)

print(
    "INITIAL_PARAMETER_SHA256 =",
    FROZEN_INIT_SHA256,
)

print(
    "POST_ADAM_PARAMETER_SHA256 =",
    post_adam_hash,
)

print(
    "OPTIMIZER_CHANGED_MODEL =",
    optimizer_changed_model,
)

print()

print(
    "ADAM_STATE_TENSORS =",
    adam_state_count,
)

print(
    "ADAM_STATES_FINITE =",
    adam_states_finite,
)

print(
    "ADAM_STEPS_ARE_ONE =",
    adam_steps_are_one,
)

print()

print(
    "BN_COUNTERS_INCREMENTED =",
    bn_counters_incremented,
)

print(
    "BN_RUNNING_STATE_CHANGED =",
    bn_running_state_changed,
)

print(
    "UPDATED_PARAMETERS_FINITE =",
    updated_parameters_finite,
)

print()

print(
    "MAIN_PARAMETERS_UNCHANGED =",
    main_parameters_unchanged,
)

print(
    "MAIN_BN_STATE_UNCHANGED =",
    main_bn_unchanged,
)

print(
    "MAIN_PARAMETER_SHA256 =",
    main_hash_after_real_step,
)

print()

print(
    "PEAK_CUDA_ALLOCATED_MIB =",
    round(
        peak_real_step_cuda_mib,
        3,
    ),
)

print()

print(
    "REAL_DATA_USED =",
    True,
)

print(
    "BACKWARD_EXECUTED =",
    True,
)

print(
    "OPTIMIZER_EXECUTED =",
    True,
)

print(
    "OPTIMIZER_STEP =",
    1,
)

print(
    "OFFICIAL_TRAINING_STARTED =",
    False,
)

print()

print(
    "SAKAI224_FULL_REAL_ONE_STEP_PASS =",
    (
        batch_identity_pass
        and input_contract_pass
        and epsilon_contract_pass
        and loss_finite
        and all_gradients_present
        and all_gradients_finite
        and gradient_tensor_count == 28
        and key_gradients_nonzero
        and changed_parameter_tensor_count > 0
        and key_parameters_changed
        and optimizer_changed_model
        and adam_state_count == 28
        and adam_states_finite
        and adam_steps_are_one
        and bn_counters_incremented
        and bn_running_state_changed
        and updated_parameters_finite
        and main_parameters_unchanged
        and main_bn_unchanged
    ),
)

KEY_GRAD encoder.conv1.weight PASS = True
KEY_GRAD encoder.fc.weight PASS = True
KEY_GRAD decoder.fc.weight PASS = True
KEY_GRAD decoder.out_conv.weight PASS = True

=== ONE REAL-DATA TRAINING STEP AUDIT ===
TRAIN_EPOCH = 0
MANIFEST_INDICES = [2676, 6250, 2396, 5602]
CROP_LEFT = [20, 15, 19, 12]
INPUT_SHAPE = (4, 4, 224, 224)

BATCH_IDENTITY_PASS = True
INPUT_CONTRACT_PASS = True
EPSILON_CONTRACT_PASS = True

LOSS_BEFORE_ADAM = 175951.28125
RECON_MEAN_PER_SAMPLE = 175900.0
KL_MEAN_PER_SAMPLE = 51.277156829833984
LOSS_FINITE = True

GRADIENT_TENSORS = 28
NONZERO_GRADIENT_TENSORS = 28
ALL_GRADIENTS_PRESENT = True
ALL_GRADIENTS_FINITE = True
KEY_GRADIENTS_NONZERO = True

CHANGED_PARAMETER_TENSORS = 28
KEY_PARAMETERS_CHANGED = True
INITIAL_PARAMETER_SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b
POST_ADAM_PARAMETER_SHA256 = 987870475c2059ded2defecabc19368a725da105bb4cba0e211c9bcc2b73b675
OPTIMIZER_CHANGED_MODEL = True

ADAM_STATE_TENSORS = 28
ADAM_STATES_FINITE =

## Chiusura del Notebook 02

Il modello `sakai224_full` e la sua integrazione con i dati Gravity Spy reali hanno superato tutti gli smoke test previsti.

Sono stati verificati:

- architettura Sakai-like 4×224×224 con latent dimension 512;
- inizializzazione PROJECT deterministica;
- forward `MEAN+EVAL`;
- funzione obiettivo VAE;
- RNG latente stateless;
- forward `STOCHASTIC+EVAL`;
- backward end-to-end;
- Adam con i parametri pubblicati da Sakai;
- singolo training-step sintetico;
- full split Gravity Spy 6866/1717;
- loader train e validation;
- shuffle PROJECT;
- augmentation stateless;
- forward su validation reale;
- forward `STOCHASTIC+TRAIN` su train reale;
- singolo training-step completo su dati Gravity Spy reali.

Il modello principale è rimasto nello stato iniziale congelato:

`SHA256 = 1df6f7d32b4d0705c273b5bfae7c3218f72ff7cd1d160560b497afb3f235859b`

La run scientifica ufficiale non rientra nel Notebook 02.

Il Notebook 03 documenta benchmark hardware, scelta del batch size, checkpoint/recovery e preparazione della run da 100 epoche.